# TableGuard 08D — Bounded fork-lift planning recovery

**Save beside the existing notebooks. Select Python (TableGuard), then Run All once.**
There are two code cells: load the implementation, then run once or view. No installs, new assets, or modifications to old checkpoints.

## What the latest 08C output establishes
The cup stage and fork grasp passed. The lift path failed in scratch planning at waypoint 38 with a **0.581215 mm position residual** and **0.204580 degree axis residual**. The planner requires <=0.500 mm and <=2.000 degrees. The fork was not lifted through physics. A local solver stopping with `no_local_improvement` does not prove global unreachability.

## Targeted change: planning only
1. Keep the original solver first. At a failed carrying waypoint, try a bounded damped solver that normalizes position and angular residuals by their **existing** acceptance tolerances. It can use the available angular slack without increasing either tolerance. Collision checks still validate the resulting candidate path.
2. For the lift only, first try the requested **40 mm** complete path. If IK still fails, try **35 mm**, then **30 mm**. This is a **declared change to commanded lift height**, not a relaxed pass condition. No rejected candidate is executed; the chosen height is saved. A collision, lost-contact precondition or timeout is not ignored to try another height.
3. The actual lift and carry must still show **at least 25 mm measured elevation**, sustained two-jaw contact and no table support. The **25 mm horizontal transfer, 8 mm placement tolerance, 15-degree final orientation limit, original speed/force/contact/tilt and cup/plate-preservation checks remain unchanged**.

A 30-second checked planning budget bounds each carrying phase. All numerical recovery remains in separate scratch data. The physical controller, jaw geometry, collision masks, friction, gravity, masses and timestep are unchanged. No welding, external lifting force, direct execution-position assignment or reset between the arms is added.

## Run and evidence
First use creates one new sequence under `artifacts/two_object_08d/`. Later Run All calls display that same outcome rather than launching another trial. Recording is enabled. Existing 06B, 07C and 08C evidence remains intact.

The report shows the chosen lift height, any recovered waypoints, measured lift/transfer/release results, and stop reasons. **Planning success is not proof of a physical lift or placement.**

This notebook remains an **expert demonstration generator**, not the competition's learned VLA controller. Do not label its actions as learned-policy output.


In [ ]:
# Load code only. The next cell starts or displays one attempt.
SOURCE_TEXTS = {'base06b.py': '"""TableGuard 06B: carried-object scratch planning correction for the cup trial.\n\nThis is an engineering demonstration generator using simulator state, NOT a\ntrained policy, camera-based decision system, bimanual task, or challenge score.\nOnly SCRATCH planning state has positions assigned, including the predicted cup pose. EXECUTION advances via ctrl\nand mj_step. No equality weld, object teleportation, external lift force, or\ncollision disable is used. Source scene and prior evidence remain unchanged.\n"""\nfrom pathlib import Path\nimport csv\nimport hashlib\nimport json\nimport math\nimport os\nimport sys\nimport time\nimport traceback\nimport xml.etree.ElementTree as ET\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\n\nJOINT_SUFFIXES = (\'shoulder_pan\',\'shoulder_lift\',\'elbow_flex\',\'wrist_flex\',\'wrist_roll\',\'gripper\')\nCOMMAND_SPEED = 0.35\nPOSITION_TOL = 0.0005\nAXIS_TOL_DEG = 2.0\nORIENTATION_WEIGHT = 0.08\nCONTACT_MIN_N = 0.02\nGRIPPER_TORQUE_CAP = 0.25\nPENETRATION_LIMIT = 0.003\nMAX_WALL_SECONDS = 240.0\n\n\ndef write_json(path, value):\n    path = Path(path)\n    tmp = path.with_suffix(path.suffix + \'.tmp\')\n    tmp.write_text(json.dumps(value, indent=2, allow_nan=False), encoding=\'utf-8\')\n    tmp.replace(path)\n\n\ndef digest(path):\n    return hashlib.sha256(Path(path).read_bytes()).hexdigest()\n\n\ndef normalize(vector):\n    vector = np.asarray(vector, dtype=float)\n    length = float(np.linalg.norm(vector))\n    if vector.shape != (3,) or not np.isfinite(vector).all() or length < 1e-10:\n        raise ValueError(\'A finite nonzero 3D vector is required\')\n    return vector / length\n\n\ndef skew(v):\n    x,y,z = v\n    return np.array([[0,-z,y],[z,0,-x],[-y,x,0]],dtype=float)\n\n\ndef smooth(s):\n    if not math.isfinite(float(s)):\n        raise ValueError(\'Non-finite interpolation parameter\')\n    x = float(np.clip(s,0,1))\n    return x*x*(3-2*x)\n\n\ndef solve_pose(evaluate, target, axis, seed, lo, hi, budget):\n    """Damped IK for a body-attached point and one tool axis; yaw is free."""\n    target, axis = np.asarray(target,dtype=float), normalize(axis)\n    q, lo, hi = [np.array(a,dtype=float,copy=True) for a in (seed,lo,hi)]\n    if (target.shape != (3,) or q.shape != lo.shape or q.shape != hi.shape or q.ndim != 1\n        or not all(np.isfinite(a).all() for a in (target,q,lo,hi)) or np.any(lo >= hi)\n        or np.any(q < lo) or np.any(q > hi)):\n        raise ValueError(\'Invalid pose IK inputs\')\n    reason = \'iteration_limit\'\n    for it in range(220):\n        budget()\n        p,a,jp,ja = evaluate(q)\n        if not all(np.isfinite(v).all() for v in (p,a,jp,ja)):\n            raise ValueError(\'Non-finite pose/Jacobian\')\n        pe = float(np.linalg.norm(target-p))\n        ae = math.degrees(math.acos(float(np.clip(np.dot(a,axis),-1,1))))\n        if pe <= POSITION_TOL and ae <= AXIS_TOL_DEG:\n            return q, dict(converged=True,position_error_m=pe,axis_error_deg=ae,iterations=it+1)\n        err = np.r_[target-p,ORIENTATION_WEIGHT*(axis-a)]\n        jac = np.vstack([jp,ORIENTATION_WEIGHT*ja])\n        dq = np.linalg.solve(jac.T@jac + 1e-6*np.eye(len(q)), jac.T@err)\n        dq *= min(1.,.055/max(float(np.max(np.abs(dq))),1e-12))\n        improved = False\n        for scale in (1.,.5,.25,.125,.0625):\n            candidate = np.clip(q+scale*dq,lo,hi)\n            pc,ac,_,_ = evaluate(candidate)\n            ec = np.r_[target-pc,ORIENTATION_WEIGHT*(axis-ac)]\n            if float(np.linalg.norm(ec)) < float(np.linalg.norm(err))-1e-11:\n                q,improved = candidate,True\n                break\n        if not improved:\n            reason = \'no_local_improvement\'\n            break\n    p,a,_,_ = evaluate(q)\n    return q,dict(converged=False,reason=reason,iterations=it+1,\n                  position_error_m=float(np.linalg.norm(target-p)),\n                  axis_error_deg=math.degrees(math.acos(float(np.clip(np.dot(a,axis),-1,1)))))\n\n\ndef parameterize(path, dt):\n    arr = np.asarray(path,dtype=float)\n    if arr.ndim != 2 or len(arr)<2 or not np.isfinite(arr).all() or not 0<dt<=.01:\n        raise ValueError(\'Invalid command path\')\n    keep = np.r_[True,np.max(np.abs(np.diff(arr,axis=0)),axis=1)>1e-10]\n    arr = arr[keep]\n    if len(arr)<2:\n        raise ValueError(\'Zero command path\')\n    lengths=np.max(np.abs(np.diff(arr,axis=0)),axis=1)\n    total=float(lengths.sum())\n    duration=math.ceil(max(1.,1.6*total/COMMAND_SPEED)/dt)*dt\n    if duration>18.:\n        raise RuntimeError(\'Local path exceeds 18 simulated seconds per leg\')\n    return dict(path=arr.tolist(),fractions=(np.r_[0,np.cumsum(lengths)]/total).tolist(),duration_s=duration)\n\n\ndef path_value(plan, s):\n    if not math.isfinite(float(s)) or not 0<=s<=1:\n        raise ValueError(\'Invalid path fraction\')\n    p=np.asarray(plan[\'path\']); f=np.asarray(plan[\'fractions\'])\n    return np.array([np.interp(s,f,p[:,j]) for j in range(p.shape[1])])\n\n\ndef prepare_scene(source, dest):\n    """Rebase a COPY and lower right gripper effort; do not edit object geometry."""\n    raw=Path(source).read_bytes()\n    if b\'<!DOCTYPE\' in raw.upper() or b\'<!ENTITY\' in raw.upper():\n        raise ValueError(\'XML entities are not accepted\')\n    tree=ET.fromstring(raw)\n    if tree.tag!=\'mujoco\' or tree.find(\'.//include\') is not None or tree.find(\'.//plugin\') is not None:\n        raise ValueError(\'Expected the self-contained, plugin-free development scene\')\n    if tree.find(\'equality\') is not None:\n        raise ValueError(\'Review scene constraints before running this contact-only test\')\n    compiler=tree.find(\'compiler\')\n    if compiler is None or compiler.get(\'angle\')!=\'radian\':\n        raise ValueError(\'Expected the reviewed radian model\')\n    for attr in (\'assetdir\',\'meshdir\',\'texturedir\'):\n        if compiler.get(attr):\n            absolute=(Path(source).parent/compiler.get(attr)).resolve()\n            if not absolute.is_dir():\n                raise FileNotFoundError(absolute)\n            compiler.set(attr,os.path.relpath(absolute,Path(dest).parent).replace(\'\\\\\',\'/\'))\n    servo=tree.find("actuator/position[@name=\'right_gripper\']")\n    if servo is None or servo.get(\'joint\')!=\'right_gripper\':\n        raise ValueError(\'Expected right_gripper position actuator\')\n    servo.set(\'forcerange\',f\'{-GRIPPER_TORQUE_CAP} {GRIPPER_TORQUE_CAP}\')\n    servo.set(\'forcelimited\',\'true\')\n    tree.set(\'model\',\'TableGuard_contact_cup_engineering_not_VLA\')\n    if Path(dest).exists():\n        raise FileExistsError(\'New output folder required; previous scene preserved\')\n    ET.indent(tree,space=\'  \')\n    ET.ElementTree(tree).write(dest,encoding=\'utf-8\',xml_declaration=True)\n    return {\'change\':\'right_gripper actuator effort reduced before execution\',\n            \'force_range_nm\':[-GRIPPER_TORQUE_CAP,GRIPPER_TORQUE_CAP],\n            \'geometry_mounts_friction_mass_gravity_timestep\':\'unchanged\'}\n\ndef validate_mapping(model, data, mj):\n    """Reject incompatible actuators rather than assume index/unit semantics."""\n    expected = {f"{side}_{joint}" for side in ("left", "right")\n                for joint in JOINT_SUFFIXES}\n    nactuator = len(model.actuator_trntype)\n    if nactuator != 12 or int(model.nu) != 12 or len(data.ctrl) != 12:\n        raise ValueError("This test requires the inspected 12 scalar actuators.")\n    items = []\n    used_controls, used_joints, names = set(), set(), set()\n    for aid in range(nactuator):\n        name = mj.mj_id2name(model, mj.mjtObj.mjOBJ_ACTUATOR, aid)\n        if name not in expected or name in names:\n            raise ValueError(f"Unexpected/duplicate actuator name: {name}")\n        names.add(name)\n        if int(model.actuator_trntype[aid]) != int(mj.mjtTrn.mjTRN_JOINT):\n            raise ValueError(f"Not a joint transmission: {name}")\n        jid = int(model.actuator_trnid[aid, 0])\n        if not 0 <= jid < int(model.njnt):\n            raise ValueError(f"Invalid joint ID: {name}")\n        jname = mj.mj_id2name(model, mj.mjtObj.mjOBJ_JOINT, jid)\n        if jname != name or jid in used_joints:\n            raise ValueError(f"Unexpected actuator-to-joint mapping: {name} -> {jname}")\n        used_joints.add(jid)\n        if int(model.jnt_type[jid]) != int(mj.mjtJoint.mjJNT_HINGE):\n            raise ValueError(f"Expected a hinge joint: {name}")\n        # Newer MuJoCo versions expose separate control addresses/counts.\n        cid = int(model.actuator_ctrladr[aid]) if hasattr(model, "actuator_ctrladr") else aid\n        num = int(model.actuator_ctrlnum[aid]) if hasattr(model, "actuator_ctrlnum") else 1\n        if num != 1 or not 0 <= cid < len(data.ctrl) or cid in used_controls:\n            raise ValueError(f"Non-scalar/duplicate control address: {name}")\n        used_controls.add(cid)\n        if not np.allclose(model.actuator_gear[aid], [1, 0, 0, 0, 0, 0], atol=1e-9, rtol=0):\n            raise ValueError(f"Non-unit gear requires a different control mapping: {name}")\n        if int(model.actuator_dyntype[aid]) != int(mj.mjtDyn.mjDYN_NONE):\n            raise ValueError(f"Filtered/dynamic actuators are outside this test: {name}")\n        if (int(model.actuator_gaintype[aid]) != int(mj.mjtGain.mjGAIN_FIXED)\n                or int(model.actuator_biastype[aid]) != int(mj.mjtBias.mjBIAS_AFFINE)):\n            raise ValueError(f"Not an unfiltered fixed-gain position servo: {name}")\n        kp = float(model.actuator_gainprm[aid, 0])\n        bp = np.asarray(model.actuator_biasprm[aid])\n        if (not math.isfinite(kp) or kp <= 0 or not np.isfinite(bp).all()\n                or not np.isclose(bp[0], 0, atol=1e-9)\n                or not np.isclose(bp[1], -kp, atol=1e-6)\n                or bp[2] > 1e-9 or not np.allclose(bp[3:], 0)):\n            raise ValueError(f"Unexpected position-servo coefficients: {name}")\n        if not np.allclose(model.actuator_gainprm[aid, 1:], 0):\n            raise ValueError(f"Unexpected gain parameters: {name}")\n        if not model.actuator_ctrllimited[aid] or not model.jnt_limited[jid]:\n            raise ValueError(f"Explicit joint and control ranges required: {name}")\n        cr = np.asarray(model.actuator_ctrlrange[cid], dtype=float)\n        jr = np.asarray(model.jnt_range[jid], dtype=float)\n        if not np.isfinite(cr).all() or not np.isfinite(jr).all():\n            raise ValueError(f"Non-finite limits: {name}")\n        lo, hi = max(float(cr[0]), float(jr[0])), min(float(cr[1]), float(jr[1]))\n        qadr = int(model.jnt_qposadr[jid])\n        vadr = int(model.jnt_dofadr[jid])\n        q0 = float(data.qpos[qadr])\n        if not math.isfinite(q0) or not lo + 0.01 < q0 < hi - 0.01:\n            raise ValueError(f"Initial position is too close to/outside limits: {name}")\n        items.append({"name": name, "aid": aid, "cid": cid, "jid": jid,\n                      "qadr": qadr, "vadr": vadr, "q0": q0,\n                      "low": lo, "high": hi, "kp": kp,\n                      "kv": float(-bp[2]), "control_range": cr.tolist(),\n                      "joint_range": jr.tolist()})\n    if names != expected or used_controls != set(range(12)):\n        raise ValueError("Incomplete actuator/control mapping.")\n    return items\n\n\ndef checked_pose(position, rotation):\n    """Validate an observed rigid transform; never alter the supplied arrays."""\n    p = np.array(position, dtype=float, copy=True)\n    R = np.array(rotation, dtype=float, copy=True)\n    if (p.shape != (3,) or R.shape != (3, 3)\n        or not np.isfinite(p).all() or not np.isfinite(R).all()\n        or not np.allclose(R.T @ R, np.eye(3), atol=1e-6, rtol=0)\n        or not np.isclose(np.linalg.det(R), 1.0, atol=1e-6, rtol=0)):\n        raise ValueError(\'Invalid position or proper rotation matrix\')\n    return p, R\n\n\ndef relative_pose(parent_position, parent_rotation, child_position, child_rotation):\n    """Observed child pose in parent coordinates, not a simulator attachment."""\n    p, R = checked_pose(parent_position, parent_rotation)\n    c, C = checked_pose(child_position, child_rotation)\n    return R.T @ (c - p), R.T @ C\n\n\ndef predicted_pose(parent_position, parent_rotation, local_position, local_rotation):\n    """Rigid-pose approximation for SCRATCH collision prediction only."""\n    p, R = checked_pose(parent_position, parent_rotation)\n    a, A = checked_pose(local_position, local_rotation)\n    return p + R @ a, R @ A\n\n\ndef primitive_plane_radius(kind, size, rotation, plane_normal):\n    """Exact support radius for the development cup\'s boxes/cylinders."""\n    _, R = checked_pose(np.zeros(3), rotation)\n    s = np.asarray(size, dtype=float)\n    if s.shape != (3,) or not np.isfinite(s).all() or np.any(s < 0):\n        raise ValueError(\'Invalid primitive size\')\n    n = normalize(plane_normal)\n    local_n = R.T @ n\n    if kind == \'box\':\n        if np.any(s <= 0): raise ValueError(\'Box half-sizes must be positive\')\n        return float(np.abs(local_n) @ s)\n    if kind == \'cylinder\':\n        if s[0] <= 0 or s[1] <= 0: raise ValueError(\'Cylinder size must be positive\')\n        return float(s[0] * np.linalg.norm(local_n[:2]) + s[1] * abs(local_n[2]))\n    raise ValueError(\'Only the reviewed box/cylinder cup geometry is supported\')\n\n\ndef measured_lowering_target(point, normal, clearance_m, compression_m=0.001):\n    """End 1 mm beyond measured first-touch height, with a bounded drop."""\n    p = np.asarray(point, dtype=float)\n    if p.shape != (3,) or not np.isfinite(p).all(): raise ValueError(\'Invalid tool point\')\n    n = normalize(normal)\n    if not math.isfinite(clearance_m) or not math.isfinite(compression_m):\n        raise ValueError(\'Non-finite set-down clearance\')\n    if not 0 <= clearance_m <= 0.080 or not 0 <= compression_m <= 0.001:\n        raise ValueError(\'Set-down drop/compression outside the diagnostic bounds\')\n    return p - (clearance_m + compression_m) * n\n\n\nclass CupTrial:\n    def __init__(self,config):\n        self.config=config; self.source=Path(config[\'scene\']).resolve(); self.out=Path(config[\'output\']).resolve()\n        self.out.mkdir(parents=True,exist_ok=True)\n        self.started=time.monotonic(); self.steps=0; self.rows=[]; self.events=[]; self.snapshots={}\n        self.phase=\'initialization\'; self.frames=[]; self.video_rows=[]; self.renderer=None\n        self.model=None; self.data=None; self.gripper_close_target=None; self.contact_history=[]\n        self.report=dict(kind=\'ENGINEERING_SINGLE_CUP_GRASP_NOT_LEARNED_POLICY\',\n            record_data_requested=bool(config.get(\'record_data\',False)),\n            completed=False,grasp_test_passed=False,grasp_verified=False,lift_verified=False,\n            release_verified=False,physics_steps=0,policy_calls=0,task_success=None,\n            observation_source=\'simulator state and known geometry: privileged engineering controller\',\n            evaluation_ground_truth_used=True,trained_policy=False,\n            source_scene_modified=False,execution_positions_assigned=False,\n            external_lifting_force=False,weld_or_attachment=False,collision_geometry_disabled=False,\n            stops=[],graphics_error=None,recording_mode=\'paired_camera_frames_for_development\',\n            training_dataset_ready=False,\n            planner_revision=\'06b-scratch-carried-cup-v1\',\n            planning_only_cup_pose_prediction=False,carry_plans={},\n            notes=[\n              \'Single nominal right-arm trial; left-arm targets held, not bimanual task success.\',\n              \'Data includes expert observations/actions and separate evaluator state. Not a LeRobot dataset yet.\',\n              \'Supplied tilt, contact/force/success bounds are engineering choices, not safety certification.\',\n              \'A failed local solve does not prove that the grasp is globally impossible.\'])\n\n    def event(self,name,**details):\n        e=dict(stage=name,elapsed_s=round(time.monotonic()-self.started,3),**details)\n        self.events.append(e);print(\'PROGRESS \'+json.dumps(e,allow_nan=False),flush=True)\n        write_json(self.out/\'grasp_progress.json\',e)\n\n    def budget(self):\n        if time.monotonic()-self.started>MAX_WALL_SECONDS:\n            raise RuntimeError(\'Engineering grasp worker time budget exceeded\')\n\n    def name(self,kind,i):\n        return self.mj.mj_id2name(self.model,kind,int(i)) or f\'unnamed_{i}\'\n\n    def needed(self,kind,name):\n        i=self.mj.mj_name2id(self.model,kind,name)\n        if i<0:raise ValueError(f\'Required model element missing: {name}\')\n        return i\n\n    def gripper_point(self,data):\n        return data.xpos[self.gripper_body]+data.xmat[self.gripper_body].reshape(3,3)@self.local_point\n\n    def load(self):\n        import mujoco as mj\n        self.mj=mj\n        self.event(\'load_scene\')\n        if digest(self.source)!=self.config[\'scene_sha256\']:\n            raise ValueError(\'Passed hover scene changed; keep source unchanged\')\n        # Verify the effort change only reduces an already finite scalar actuator limit.\n        original=mj.MjModel.from_xml_path(str(self.source)); check_data=mj.MjData(original)\n        mj.mj_forward(original,check_data); validate_mapping(original,check_data,mj)\n        aid=mj.mj_name2id(original,mj.mjtObj.mjOBJ_ACTUATOR,\'right_gripper\')\n        if not original.actuator_forcelimited[aid] or not (\n            original.actuator_forcerange[aid,0]<=-GRIPPER_TORQUE_CAP and\n            original.actuator_forcerange[aid,1]>=GRIPPER_TORQUE_CAP):\n            raise ValueError(\'Cannot verify that the proposed gripper effort cap only reduces effort\')\n        self.report[\'original_gripper_force_range_nm\']=original.actuator_forcerange[aid].tolist()\n        copied=self.out/\'grasp_development.xml\'\n        self.report[\'scene_adjustment\']=prepare_scene(self.source,copied)\n        self.model=mj.MjModel.from_xml_path(str(copied));self.data=mj.MjData(self.model)\n        m,d=self.model,self.data; mj.mj_forward(m,d)\n        self.mapping=validate_mapping(m,d,mj);self.byname={r[\'name\']:r for r in self.mapping}\n        self.arm=[self.byname[\'right_\'+j] for j in JOINT_SUFFIXES[:-1]]\n        self.g=self.byname[\'right_gripper\'];self.dt=float(m.opt.timestep)\n        if not 0<self.dt<=.01 or not math.isclose(round(.1/self.dt)*self.dt,.1,abs_tol=1e-9):\n            raise ValueError(\'Unexpected timestep\')\n        if np.any(np.asarray(m.body_gravcomp)>0):\n            raise ValueError(\'Unexpected gravity compensation in this engineering model\')\n        self.gripper_body=self.needed(mj.mjtObj.mjOBJ_BODY,\'right_gripper\')\n        self.moving_body=self.needed(mj.mjtObj.mjOBJ_BODY,\'right_moving_jaw_so101_v1\')\n        self.cup=self.needed(mj.mjtObj.mjOBJ_BODY,\'development_cup\')\n        self.table=self.needed(mj.mjtObj.mjOBJ_GEOM,\'development_table\')\n        self.fixed_tip=self.needed(mj.mjtObj.mjOBJ_GEOM,\'right_fixed_jaw_sph_tip1\')\n        self.moving_tip=self.needed(mj.mjtObj.mjOBJ_GEOM,\'right_moving_jaw_sph_tip1\')\n        if int(m.geom_bodyid[self.fixed_tip])!=self.gripper_body or int(m.geom_bodyid[self.moving_tip])!=self.moving_body:\n            raise ValueError(\'Unexpected fingertip ownership\')\n        if any(int(m.geom_type[t])!=int(mj.mjtGeom.mjGEOM_SPHERE) for t in (self.fixed_tip,self.moving_tip)):\n            raise ValueError(\'Expected named spherical tip markers/collision geoms\')\n        if int(m.body_jntnum[self.cup])!=1 or int(m.jnt_type[int(m.body_jntadr[self.cup])])!=int(mj.mjtJoint.mjJNT_FREE):\n            raise ValueError(\'Cup must be a freely moving rigid body\')\n        # Intentionally narrow to the known, upright faceted hollow development cup.\n        self.cup_geoms={int(g) for g in range(m.ngeom) if int(m.geom_bodyid[g])==self.cup}\n        walls=[g for g in self.cup_geoms if self.name(mj.mjtObj.mjOBJ_GEOM,g).startswith(\'cup_wall_\')]\n        if len(walls)!=16:\n            raise ValueError(\'This first grasp is configured for the existing 16-wall development cup\')\n        radius=0.\n        for g in walls:\n            if int(m.geom_type[g])!=int(mj.mjtGeom.mjGEOM_BOX):\n                raise ValueError(\'Expected primitive box walls\')\n            R=d.geom_xmat[g].reshape(3,3)\n            for sx in (-1,1):\n                for sy in (-1,1):\n                    v=d.geom_xpos[g]+R@np.array([sx*m.geom_size[g,0],sy*m.geom_size[g,1],0])\n                    radius=max(radius,float(np.linalg.norm((v-d.xpos[self.cup])[:2])))\n        if not .020<=radius<=.030:\n            raise ValueError(\'Cup radius outside the reviewed small-object range\')\n        self.radius=radius\n        # A body-attached control point near the nominal cup center at fingertip height.\n        fixed_local=m.geom_pos[self.fixed_tip].copy()\n        self.local_point=fixed_local+np.array([radius+float(m.geom_size[self.fixed_tip,0])+.0018,0,0])\n        self.open_q=.70; self.close_floor=.15\n        if not self.g[\'low\']+.01<self.close_floor<self.open_q<self.g[\'high\']-.01:\n            raise ValueError(\'Proposed jaw commands outside validated bounds\')\n        for item in self.mapping:d.ctrl[item[\'cid\']]=item[\'q0\']\n        self.warnings0=np.array(d.warning.number,copy=True)\n        self.scratch=mj.MjData(m);self.scratch.qpos[:]=d.qpos\n        distances={}\n        for q in (self.g[\'q0\'],self.open_q):\n            self.scratch.qpos[self.g[\'qadr\']]=q;mj.mj_forward(m,self.scratch)\n            distance=np.linalg.norm(self.scratch.geom_xpos[self.fixed_tip]-self.scratch.geom_xpos[self.moving_tip])\n            distance-=float(m.geom_size[self.fixed_tip,0]+m.geom_size[self.moving_tip,0])\n            distances[str(q)]=float(distance)\n        if distances[str(self.open_q)]<=2*radius+.006:\n            raise RuntimeError(\'Proposed open-tip gap is not wide enough for the nominal cup\')\n        self.geom_group={};self.robot_geoms=set()\n        for gid in range(m.ngeom):\n            bid=int(m.geom_bodyid[gid]);cur=bid;side=None\n            while cur:\n                n=self.name(mj.mjtObj.mjOBJ_BODY,cur)\n                if n.startswith((\'left_\',\'right_\')):side=n.split(\'_\')[0];break\n                cur=int(m.body_parentid[cur])\n            if side:self.robot_geoms.add(gid)\n            n=self.name(mj.mjtObj.mjOBJ_GEOM,gid)\n            mesh=\'\'\n            if int(m.geom_type[gid])==int(mj.mjtGeom.mjGEOM_MESH):\n                mesh=self.name(mj.mjtObj.mjOBJ_MESH,int(m.geom_dataid[gid]))\n            if bid==self.gripper_body and (n.startswith(\'right_fixed_jaw_sph_\') or n in\n                {\'right_fixed_jaw_box3\',\'right_fixed_jaw_box4\',\'right_fixed_jaw_box5\',\'right_fixed_jaw_box6\',\'right_fixed_jaw_box7\'}\n                or mesh==\'wrist_roll_follower_so101_gripper_part0_v1\'):\n                self.geom_group[gid]=\'fixed\'\n            elif bid==self.moving_body:\n                self.geom_group[gid]=\'moving\'\n        self.arm_indices=[r[\'qadr\'] for r in self.arm];self.arm_dofs=[r[\'vadr\'] for r in self.arm]\n        self.low=np.array([r[\'low\']+.02 for r in self.arm]);self.high=np.array([r[\'high\']-.02 for r in self.arm])\n        self.prop_reference={n:d.xpos[self.needed(mj.mjtObj.mjOBJ_BODY,\'development_\'+n)].copy() for n in (\'plate\',\'fork\')}\n        self.report.update(scene=str(copied),scene_sha256=digest(copied),source_scene=str(self.source),\n            mujoco_version=mj.__version__,python=sys.executable,timestep_s=self.dt,\n            native_actuator_mapping=self.mapping,grasp_reference_local_m=self.local_point.tolist(),\n            cup_outer_radius_m=radius,tip_gap_diagnostic_m=distances,\n            requested_jaw_open_rad=self.open_q,maximum_closure_floor_rad=self.close_floor,\n            thresholds=dict(lift_required_m=.025,lift_command_m=.040,release_position_m=.015,\n              allowed_cup_tilt_deg=30,contact_fraction=.80,gripper_force_limit_nm=GRIPPER_TORQUE_CAP,\n              penetration_limit_m=PENETRATION_LIMIT,contact_min_normal_n=CONTACT_MIN_N))\n        self.event(\'scene_ready\',cup_diameter_mm=round(2000*radius,3),jaw_open_rad=self.open_q)\n\n    def contacts(self,data,allow_cup,planning=False):\n        m,mj=self.model,self.mj\n        f={\'fixed\':0.,\'moving\':0.};table=0.;issues=[]\n        for i in range(data.ncon):\n            c=data.contact[i]; a,b=int(c.geom1),int(c.geom2);dist=float(c.dist)\n            if a<0 or b<0 or not math.isfinite(dist):raise RuntimeError(\'Unsupported contact\')\n            ca,cb=a in self.cup_geoms,b in self.cup_geoms\n            other=b if ca else a\n            jaw=self.geom_group.get(other) if ca or cb else None\n            robot=a in self.robot_geoms or b in self.robot_geoms\n            intended=bool(allow_cup and jaw and (ca or cb))\n            if planning and intended:\n                # Intended finger contacts are force-checked in execution, not scored from scratch forces.\n                # Carrying plans also check their predicted penetration in extra_carry_contact_issues().\n                continue\n            force=np.zeros(6)\n            if c.efc_address>=0:mj.mj_contactForce(m,data,i,force)\n            normal=max(0.,float(force[0]))\n            if jaw and (ca or cb) and dist<=.0005:f[jaw]+=normal\n            if (ca or cb) and other==self.table:table+=normal\n            if dist < -PENETRATION_LIMIT:issues.append(\'excess penetration\')\n            if robot and dist<=0 and not intended:\n                issues.append(f\'unexpected robot contact: {self.name(mj.mjtObj.mjOBJ_GEOM,a)} / {self.name(mj.mjtObj.mjOBJ_GEOM,b)}\')\n            if intended and normal>25:issues.append(\'jaw contact normal force exceeds engineering bound\')\n        return f,table,issues\n\n    def verify_scratch_separation(self):\n        if self.scratch is self.data or np.shares_memory(self.scratch.qpos, self.data.qpos):\n            raise RuntimeError(\'Planning state must be separate from execution state\')\n\n    def make_carry_reference(self, label, allow_cup):\n        """Capture current cup-to-gripper relation only after measured grasp gates."""\n        self.verify_scratch_separation()\n        if label not in (\'lift\', \'set_down\') or not allow_cup:\n            raise ValueError(\'Carry prediction is only permitted for verified lift/set_down plans\')\n        if not self.report.get(\'grasp_verified\') or self.gripper_close_target is None:\n            raise RuntimeError(\'Cannot predict a carried cup before a verified grasp\')\n        if label == \'set_down\' and not self.report.get(\'lift_verified\'):\n            raise RuntimeError(\'Set-down carrying plan requires a verified lift\')\n        f, _, issues = self.contacts(self.data, True)\n        if issues or min(f[\'fixed\'], f[\'moving\']) < CONTACT_MIN_N:\n            raise RuntimeError(\'Two-jaw contact must still be valid at carry-plan entry: \' + repr(issues))\n        m, d, mj = self.model, self.data, self.mj\n        jid = int(m.body_jntadr[self.cup])\n        if (int(m.body_parentid[self.cup]) != 0 or int(m.body_jntnum[self.cup]) != 1\n            or int(m.jnt_type[jid]) != int(mj.mjtJoint.mjJNT_FREE)):\n            raise ValueError(\'Carry prediction requires the reviewed free cup body\')\n        qadr = int(m.jnt_qposadr[jid])\n        if qadr < 0 or qadr + 7 > len(self.scratch.qpos):\n            raise ValueError(\'Invalid free-cup position/quaternion address\')\n        local_p, local_R = relative_pose(\n            d.xpos[self.gripper_body], d.xmat[self.gripper_body].reshape(3, 3),\n            d.xpos[self.cup], d.xmat[self.cup].reshape(3, 3))\n        record = dict(\n            mode=\'measured rigid relative pose in SCRATCH ONLY; no execution attachment\',\n            label=label, cup_qpos_address=qadr, entry_sim_time_s=float(d.time),\n            relative_position_m=local_p.tolist(), relative_rotation=local_R.tolist(),\n            entry_jaw_normal_n=f, candidate_evaluations=0,\n            minimum_sampled_contact_distance_m=None)\n        self.report[\'carry_plans\'][label] = record\n        self.event(\'carry_plan_reference\', phase=label, mode=record[\'mode\'])\n        return dict(qadr=qadr, position=local_p, rotation=local_R, record=record)\n\n    def update_scratch_cup(self, reference):\n        """Move only the predictive copy with the candidate gripper pose."""\n        self.verify_scratch_separation()\n        m, s, mj = self.model, self.scratch, self.mj\n        # Refresh gripper transforms after candidate arm qpos was assigned.\n        # This does not step either simulation and does not update execution.\n        mj.mj_kinematics(m, s)\n        p, R = predicted_pose(s.xpos[self.gripper_body],\n                              s.xmat[self.gripper_body].reshape(3, 3),\n                              reference[\'position\'], reference[\'rotation\'])\n        quat = np.empty(4, dtype=float)\n        mj.mju_mat2Quat(quat, np.ascontiguousarray(R).reshape(9))\n        length = float(np.linalg.norm(quat))\n        if not np.isfinite(quat).all() or length < 1e-10:\n            raise RuntimeError(\'Invalid predicted cup quaternion\')\n        quat /= length\n        address = reference[\'qadr\']\n        s.qpos[address:address + 3] = p\n        s.qpos[address + 3:address + 7] = quat\n        self.report[\'planning_only_cup_pose_prediction\'] = True\n        reference[\'record\'][\'candidate_evaluations\'] += 1\n\n    def extra_carry_contact_issues(self, reference):\n        """Keep a geometric penetration bound even for intended scratch contacts."""\n        issues = []\n        for i in range(self.scratch.ncon):\n            c = self.scratch.contact[i]\n            distance = float(c.dist)\n            if not math.isfinite(distance): raise RuntimeError(\'Non-finite predicted contact\')\n            old = reference[\'record\'][\'minimum_sampled_contact_distance_m\']\n            reference[\'record\'][\'minimum_sampled_contact_distance_m\'] = distance if old is None else min(old, distance)\n            if distance < -PENETRATION_LIMIT:\n                issues.append(\'carried-pose sampled penetration exceeds existing bound\')\n        return issues\n\n    def set_down_target(self, original_nominal_target):\n        """Use current cup/table geometry instead of assuming zero slip after lift."""\n        if not self.report.get(\'lift_verified\'):\n            raise RuntimeError(\'A measured lift is required before choosing set-down height\')\n        m, d, mj = self.model, self.data, self.mj\n        if int(m.geom_type[self.table]) != int(mj.mjtGeom.mjGEOM_BOX):\n            raise ValueError(\'Expected the unchanged box tabletop\')\n        table_R = d.geom_xmat[self.table].reshape(3, 3)\n        normal = table_R[:, 2]\n        if not np.allclose(normal, [0, 0, 1], atol=1e-6, rtol=0):\n            raise ValueError(\'This bounded lowering diagnostic requires the horizontal tabletop\')\n        top = d.geom_xpos[self.table] + normal * float(m.geom_size[self.table, 2])\n        gaps = []\n        for gid in self.cup_geoms:\n            typ = int(m.geom_type[gid])\n            if typ == int(mj.mjtGeom.mjGEOM_BOX): kind = \'box\'\n            elif typ == int(mj.mjtGeom.mjGEOM_CYLINDER): kind = \'cylinder\'\n            else: raise ValueError(\'Changed cup geometry needs a new support-plane calculation\')\n            support = primitive_plane_radius(kind, m.geom_size[gid],\n                                             d.geom_xmat[gid].reshape(3, 3), normal)\n            gaps.append(float(np.dot(d.geom_xpos[gid] - top, normal) - support))\n        if not gaps: raise ValueError(\'No cup geometry for set-down measurement\')\n        gap = min(gaps)\n        target = measured_lowering_target(self.gripper_point(d), normal, gap)\n        self.report[\'set_down_height\'] = dict(\n            source=\'current simulator cup geometry; privileged engineering measurement, not camera perception\',\n            measured_bottom_clearance_m=gap, downward_contact_allowance_m=0.001,\n            original_nominal_target_m=np.asarray(original_nominal_target).tolist(),\n            adjusted_target_m=target.tolist(),\n            meaning=\'same 1 mm contact allowance, referenced to measured bottom clearance after lifting\')\n        self.event(\'set_down_height_measured\', clearance_mm=round(1000 * gap, 3))\n        return target\n\n    def plan(self,target,axis,label,allow_cup=False,carry_cup=False):\n        m,mj,d=self.model,self.mj,self.data;self.event(\'plan_start\',phase=label)\n        self.verify_scratch_separation()\n        carry = self.make_carry_reference(label, allow_cup) if carry_cup else None\n        scratch=self.scratch; scratch.qpos[:]=d.qpos  # Planning copy only.\n        seed=np.array([d.ctrl[r[\'cid\']] for r in self.arm])\n        jacp,jacr=np.zeros((3,m.nv)),np.zeros((3,m.nv))\n        def evaluate(q):\n            self.budget();scratch.qpos[self.arm_indices]=q\n            if carry is not None:\n                self.update_scratch_cup(carry)\n            mj.mj_forward(m,scratch)\n            R=scratch.xmat[self.gripper_body].reshape(3,3);a=R[:,2].copy()\n            p=self.gripper_point(scratch)\n            mj.mj_jac(m,scratch,jacp,jacr,p,self.gripper_body)\n            return p.copy(),a,jacp[:,self.arm_dofs].copy(),(-skew(a)@jacr[:,self.arm_dofs]).copy()\n        p0,a0,_,_=evaluate(seed);target=np.asarray(target,dtype=float);axis=normalize(axis)\n        path=[seed.copy()];diags=[]\n        for k in range(1,41):\n            t=k/40;p=p0+t*(target-p0);a=normalize((1-t)*a0+t*axis)\n            q,diag=solve_pose(evaluate,p,a,seed,self.low,self.high,self.budget)\n            diags.append(diag)\n            if not diag[\'converged\']:raise RuntimeError(f\'{label}: IK waypoint {k} failed: {diag}\')\n            count=max(1,math.ceil(float(np.max(np.abs(q-seed)))/.02))\n            for alpha in np.linspace(0,1,count+1)[1:]:\n                evaluate(seed+alpha*(q-seed));_,_,issues=self.contacts(scratch,allow_cup,planning=True)\n                if carry is not None:\n                    issues += self.extra_carry_contact_issues(carry)\n                if issues:raise RuntimeError(f\'{label}: sampled path rejected: {issues[:2]}\')\n            path.append(q.copy());seed=q\n            if k%10==0:self.event(\'plan_progress\',phase=label,waypoint=k,error_mm=round(1000*diag[\'position_error_m\'],3))\n        plan=parameterize(path,self.dt)\n        plan.update(target_m=target.tolist(),target_axis=axis.tolist(),diagnostics=diags,\n                    carried_object_prediction=carry[\'record\'] if carry is not None else None)\n        write_json(self.out/f\'plan_{label}.json\',plan)\n        self.event(\'plan_complete\',phase=label,duration_s=plan[\'duration_s\'])\n        return plan\n\n    def setup_render(self):\n        mj=self.mj\n        self.event(\'renderer_start\')\n        try:\n            self.renderer=mj.Renderer(self.model,height=480,width=640)\n            self.view=mj.MjvCamera();self.view.type=mj.mjtCamera.mjCAMERA_FREE\n            self.view.lookat[:]=[.12,.04,.11];self.view.distance=1.3;self.view.azimuth=115;self.view.elevation=-35\n            self.options=mj.MjvOption();self.options.geomgroup[3:5]=0;self.options.sitegroup[:]=0\n        except Exception as exc:self.report[\'graphics_error\']=f\'{type(exc).__name__}: {exc}\'\n\n    def snapshot(self,label):\n        if self.renderer is None:return\n        try:\n            self.renderer.update_scene(self.data,camera=self.view,scene_option=self.options)\n            im=Image.fromarray(self.renderer.render().copy())\n            draw=ImageDraw.Draw(im);draw.rectangle((0,0,640,45),fill=\'black\')\n            draw.text((8,4),\'CONTACT-GRASP ENGINEERING TEST | not trained VLA / not table-setting\',fill=\'white\')\n            draw.text((8,23),f\'{label} | simulated t={self.data.time:.2f}s\',fill=\'white\')\n            p=self.out/f\'{label}.png\';im.save(p);self.snapshots[label]=str(p)\n        except Exception as exc:self.report[\'graphics_error\']=f\'{type(exc).__name__}: {exc}\'\n\n    def tick(self,command,label,allow_cup=False):\n        m,d,mj=self.model,self.data,self.mj;self.phase=label;self.budget()\n        command=np.asarray(command,dtype=float)\n        if command.shape!=(12,) or not np.isfinite(command).all():raise ValueError(\'Invalid 12-control command\')\n        if np.any(np.abs(command-d.ctrl)>COMMAND_SPEED*self.dt*1.06+1e-8):\n            raise RuntimeError(\'Command slew guard\')\n        for r in self.mapping:\n            if not r[\'low\']<=command[r[\'cid\']]<=r[\'high\']:raise ValueError(\'Command outside joint/control limits\')\n        # BEFORE-action observation and the exact command applied in the following step.\n        if (self.config.get(\'record_data\',False) and self.renderer is not None\n            and self.steps%round(.1/self.dt)==0):\n            try:\n                obs_dir=self.out/\'observations\';obs_dir.mkdir(exist_ok=True)\n                frame_id=len(self.video_rows);image_paths={}\n                for cam in (\'top\',\'right_wrist_cam\'):\n                    if mj.mj_name2id(m,mj.mjtObj.mjOBJ_CAMERA,cam)<0:raise ValueError(f\'Camera missing: {cam}\')\n                    self.renderer.update_scene(d,camera=cam,scene_option=self.options)\n                    pix=self.renderer.render().copy();file=obs_dir/f\'{frame_id:05d}_{cam}.jpg\'\n                    Image.fromarray(pix).save(file,quality=85)\n                    image_paths[cam]=str(file.relative_to(self.out))\n                self.video_rows.append(dict(frame_id=frame_id,step=self.steps,time_s=float(d.time),images=image_paths,\n                    instruction=\'Lift the cup, hold it, and set it back on the table.\',\n                    state=[float(d.qpos[r[\'qadr\']]) for r in self.mapping],\n                    action=[float(command[r[\'cid\']]) for r in self.mapping],action_order=[r[\'name\'] for r in self.mapping],\n                    action_semantics=\'native absolute joint position targets, radians; applied immediately after observation\',\n                    expert_source=\'privileged simulator-state engineering controller; not learned-policy output\'))\n            except Exception as exc:\n                self.report[\'graphics_error\']=f\'recording: {type(exc).__name__}: {exc}\'\n                self.config[\'record_data\']=False\n        d.ctrl[:]=command\n        previous=float(d.time);mj.mj_step(m,d);self.steps+=1;mj.mj_forward(m,d)\n        if not math.isclose(float(d.time),previous+self.dt,abs_tol=1e-8):raise RuntimeError(\'Unexpected simulation time/reset\')\n        if not np.isfinite(d.qpos).all() or not np.isfinite(d.qvel).all():raise RuntimeError(\'Non-finite simulation\')\n        if np.any(np.asarray(d.warning.number)>self.warnings0):raise RuntimeError(\'MuJoCo numerical/capacity warning\')\n        f,support,issues=self.contacts(d,allow_cup)\n        xyz=d.xpos[self.cup].copy();tilt=math.degrees(math.acos(float(np.clip(d.xmat[self.cup].reshape(3,3)[2,2],-1,1))))\n        row=dict(step=self.steps,time_s=float(d.time),phase=label,cup_x=xyz[0],cup_y=xyz[1],cup_z=xyz[2],\n            cup_tilt_deg=tilt,fixed_normal_n=f[\'fixed\'],moving_normal_n=f[\'moving\'],table_normal_n=support,\n            two_jaw_contact=f[\'fixed\']>=CONTACT_MIN_N and f[\'moving\']>=CONTACT_MIN_N)\n        for r in self.mapping:\n            q,v=float(d.qpos[r[\'qadr\']]),float(d.qvel[r[\'vadr\']])\n            row[r[\'name\']+\'_position_rad\']=q;row[r[\'name\']+\'_command_rad\']=float(command[r[\'cid\']])\n            if not r[\'low\']-.03<=q<=r[\'high\']+.03:issues.append(\'joint range: \'+r[\'name\'])\n            if abs(v)>3.:issues.append(\'joint speed: \'+r[\'name\'])\n            # Contact blocks the jaw, so ordinary free-motion tracking does not apply to that one joint.\n            limit=.65 if allow_cup and r[\'name\']==\'right_gripper\' else .12\n            if abs(q-command[r[\'cid\']])>limit:issues.append(\'tracking: \'+r[\'name\'])\n            if r[\'name\'].startswith(\'left_\') and abs(q-r[\'q0\'])>.15:issues.append(\'inactive arm drift\')\n        for name,p0 in self.prop_reference.items():\n            b=self.needed(mj.mjtObj.mjOBJ_BODY,\'development_\'+name)\n            dist=float(np.linalg.norm(d.xpos[b]-p0));row[name+\'_displacement_m\']=dist\n            if dist>.010:issues.append(\'other object moved: \'+name)\n        if tilt>30.:issues.append(\'cup tilted beyond engineering bound\')\n        if hasattr(self,\'cup_initial\') and np.linalg.norm(xyz[:2]-self.cup_initial[:2])>.035:\n            issues.append(\'cup lateral displacement too large for set-back trial\')\n        self.rows.append(row)\n        if issues:raise RuntimeError(\'; \'.join(issues[:3]))\n        return row\n\n    def move_jaw(self,target,label,allow_cup=False):\n        start=self.data.ctrl.copy();end=start.copy();end[self.g[\'cid\']]=target\n        seconds=math.ceil(max(.6,1.6*abs(target-start[self.g[\'cid\']])/COMMAND_SPEED)/self.dt)*self.dt\n        n=round(seconds/self.dt);self.event(\'segment_start\',phase=label,simulated_seconds=seconds)\n        for k in range(1,n+1):self.tick(start+smooth(k/n)*(end-start),label,allow_cup)\n        self.event(\'segment_complete\',phase=label)\n\n    def hold(self,seconds,label,allow_cup=False):\n        self.event(\'segment_start\',phase=label,simulated_seconds=seconds)\n        for _ in range(round(seconds/self.dt)):self.tick(self.data.ctrl.copy(),label,allow_cup)\n        self.event(\'segment_complete\',phase=label)\n\n    def execute_plan(self,plan,label,allow_cup=False):\n        n=round(plan[\'duration_s\']/self.dt);base=self.data.ctrl.copy()\n        self.event(\'segment_start\',phase=label,simulated_seconds=plan[\'duration_s\'])\n        for k in range(1,n+1):\n            cmd=base.copy();q=path_value(plan,smooth(k/n))\n            for r,value in zip(self.arm,q):cmd[r[\'cid\']]=value\n            self.tick(cmd,label,allow_cup)\n        self.event(\'segment_complete\',phase=label)\n\n    def stable_rows(self,label,seconds=.25):\n        rows=[r for r in self.rows if r[\'phase\']==label]\n        if not rows:raise RuntimeError(\'No measured samples for \'+label)\n        end=rows[-1][\'time_s\'];return [r for r in rows if r[\'time_s\']>=end-seconds-1e-9]\n\n    def close_on_cup(self):\n        self.event(\'segment_start\',phase=\'close_gripper\')\n        start=float(self.data.ctrl[self.g[\'cid\']]);n=round(4./self.dt);consecutive=0\n        for k in range(1,n+1):\n            cmd=self.data.ctrl.copy();cmd[self.g[\'cid\']]=start+smooth(k/n)*(self.close_floor-start)\n            row=self.tick(cmd,\'close_gripper\',True)\n            consecutive=consecutive+1 if row[\'two_jaw_contact\'] else 0\n            if consecutive>=max(1,round(.10/self.dt)):\n                self.gripper_close_target=float(cmd[self.g[\'cid\']]);break\n        if self.gripper_close_target is None:\n            raise RuntimeError(\'No sustained two-jaw contact; the cup will not be lifted\')\n        self.hold(.4,\'grasp_hold\',True);rows=self.stable_rows(\'grasp_hold\')\n        fraction=sum(bool(r[\'two_jaw_contact\']) for r in rows)/len(rows)\n        self.report[\'grasp_contact_fraction\']=fraction\n        if fraction<.80:raise RuntimeError(\'Two-jaw contact did not remain stable before lift\')\n        self.report[\'grasp_verified\']=True;self.snapshot(\'grasped\')\n        self.event(\'grasp_contact_verified\',fraction=fraction,command_rad=self.gripper_close_target)\n\n    def execute(self):\n        self.load();self.setup_render();self.snapshot(\'start\')\n        self.hold(.75,\'initial_settle\')\n        # This is an explicitly privileged expert/controller measurement, not a camera estimate.\n        self.cup_initial=self.data.xpos[self.cup].copy()\n        self.prop_reference={n:self.data.xpos[self.needed(self.mj.mjtObj.mjOBJ_BODY,\'development_\'+n)].copy() for n in (\'plate\',\'fork\')}\n        self.report[\'initial_cup_position_m\']=self.cup_initial.tolist()\n        self.move_jaw(self.open_q,\'open_gripper\');self.snapshot(\'gripper_open\')\n        delta=self.data.xpos[self.needed(self.mj.mjtObj.mjOBJ_BODY,\'right_mount\')]-self.cup_initial\n        delta[2]=0;direction=normalize(delta)\n        axis=normalize(direction*math.sin(math.radians(25))+np.array([0,0,math.cos(math.radians(25))]))\n        pre=self.cup_initial.copy();pre[2]=.15\n        grip=self.cup_initial.copy();grip[2]=float(self.cup_initial[2]+.032)\n        raised=grip+np.array([0,0,.040])\n        self.report.update(tool_axis=axis.tolist(),tool_tilt_command_deg=25,\n            pregrasp_target_m=pre.tolist(),grasp_target_m=grip.tolist(),lift_target_m=raised.tolist())\n        p=self.plan(pre,axis,\'pregrasp\');self.execute_plan(p,\'pregrasp\');self.hold(.25,\'pregrasp_hold\');self.snapshot(\'pregrasp\')\n        p=self.plan(grip,axis,\'lower_to_grasp\',True);self.execute_plan(p,\'lower_to_grasp\',True)\n        self.hold(.25,\'grasp_approach_hold\',True)\n        if np.linalg.norm((self.data.xpos[self.cup]-self.cup_initial)[:2])>.010:\n            raise RuntimeError(\'Cup shifted too far before closure\')\n        self.close_on_cup()\n        p=self.plan(raised,axis,\'lift\',True,carry_cup=True);self.execute_plan(p,\'lift\',True);self.hold(.75,\'lift_hold\',True)\n        rows=self.stable_rows(\'lift_hold\',.5)\n        rise=min(float(r[\'cup_z\']-self.cup_initial[2]) for r in rows)\n        contact_fraction=sum(bool(r[\'two_jaw_contact\']) for r in rows)/len(rows)\n        max_support=max(r[\'table_normal_n\'] for r in rows)\n        self.report[\'lift_measurements\']=dict(minimum_cup_rise_m=rise,two_jaw_contact_fraction=contact_fraction,max_table_normal_n=max_support)\n        if rise<.025 or contact_fraction<.80 or max_support>=CONTACT_MIN_N:\n            raise RuntimeError(\'Lift not verified: require elevation, two jaws, and no table support\')\n        self.report[\'lift_verified\']=True;self.snapshot(\'lifted\')\n        # Set it back, rather than claiming a new table-setting arrangement.\n        nominal_put=grip-np.array([0,0,.001])\n        put=self.set_down_target(nominal_put)\n        p=self.plan(put,axis,\'set_down\',True,carry_cup=True);self.execute_plan(p,\'set_down\',True);self.hold(.5,\'support_hold\',True)\n        rows=self.stable_rows(\'support_hold\')\n        support_fraction=sum(r[\'table_normal_n\']>=CONTACT_MIN_N for r in rows)/len(rows)\n        self.report[\'set_down_measurements\'] = dict(table_support_fraction=support_fraction)\n        if support_fraction<.8:raise RuntimeError(\'Table support not established before opening\')\n        self.snapshot(\'supported\')\n        self.move_jaw(self.open_q,\'release_gripper\',True);self.snapshot(\'released\')\n        p=self.plan(pre,axis,\'retreat\',True);self.execute_plan(p,\'retreat\',True);self.hold(.75,\'final_settle\')\n        rows=self.stable_rows(\'final_settle\',.5)\n        err=max(float(np.linalg.norm(np.array([r[\'cup_x\'],r[\'cup_y\'],r[\'cup_z\']])-self.cup_initial)) for r in rows)\n        fraction=sum(r[\'table_normal_n\']>=CONTACT_MIN_N and r[\'fixed_normal_n\']<CONTACT_MIN_N and r[\'moving_normal_n\']<CONTACT_MIN_N for r in rows)/len(rows)\n        self.report[\'release_measurements\']=dict(maximum_position_error_m=err,supported_and_released_fraction=fraction)\n        if err>.015 or fraction<.8:raise RuntimeError(\'Release not verified on table near initial position\')\n        self.report[\'release_verified\']=True;self.report[\'completed\']=True;self.snapshot(\'final\')\n        self.report[\'grasp_test_passed\']=True\n\n    def finish(self):\n        if self.renderer is not None:\n            try:self.renderer.close()\n            except Exception as exc:self.report[\'graphics_error\']=self.report[\'graphics_error\'] or str(exc)\n        self.report.update(physics_steps=self.steps,simulation_time_s=float(self.data.time) if self.data is not None else 0,\n            snapshots=self.snapshots,rendered_snapshots=len(self.snapshots),timing_events=self.events,\n            source_scene_modified=self.source.is_file() and digest(self.source)!=self.config[\'scene_sha256\'])\n        if self.report[\'source_scene_modified\']:\n            self.report[\'grasp_test_passed\']=False;self.report[\'stops\'].append(\'Source scene changed\')\n        if self.rows:\n            fields=list(self.rows[0]);file=self.out/\'grasp_trace.csv\'\n            with file.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n                writer=csv.DictWriter(f,fieldnames=fields);writer.writeheader();writer.writerows(self.rows)\n            self.report[\'trace_file\']=str(file)\n        if self.video_rows:\n            stride=round(.1/self.dt)\n            for obs in self.video_rows:\n                selected=self.rows[obs[\'step\']:obs[\'step\']+stride]\n                obs[\'action_chunk\']=[\n                    [float(row[r[\'name\']+\'_command_rad\']) for r in self.mapping]\n                    for row in selected]\n                obs[\'action_chunk_dt_s\']=self.dt\n                obs[\'action_chunk_valid_length\']=len(selected)\n                obs[\'episode_outcome\']=(\'passed_engineering_trial\' if self.report[\'grasp_test_passed\'] else \'failed_or_partial_engineering_trial\')\n            file=self.out/\'expert_observations.jsonl\'\n            with file.open(\'w\',encoding=\'utf-8\') as f:\n                for row in self.video_rows:f.write(json.dumps(row,allow_nan=False)+\'\\n\')\n            self.report[\'observation_manifest\']=str(file)\n        self.report[\'recorded_observation_action_pairs\']=len(self.video_rows)\n        self.report[\'wall_time_s_before_report\']=time.monotonic()-self.started\n        self.report[\'report_file\']=str(self.out/\'grasp_report.json\')\n        write_json(self.out/\'grasp_report.json\',self.report)\n\n\ndef main(config):\n    trial=CupTrial(config)\n    try:trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.event(\'stopped\',reason=trial.report[\'stops\'][-1])\n        (trial.out/\'failure_traceback.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n        if trial.renderer is not None:trial.snapshot(\'stopped\')\n    finally:trial.finish()\n    print(\'GRASP RESULT \'+json.dumps({k:trial.report.get(k) for k in\n        (\'completed\',\'grasp_test_passed\',\'grasp_verified\',\'lift_verified\',\'release_verified\',\n         \'physics_steps\',\'simulation_time_s\',\'graphics_error\',\'stops\',\'report_file\')},allow_nan=False),flush=True)\n    return 0 if trial.report[\'grasp_test_passed\'] else 2\n\n\nif __name__==\'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'transfer_worker.py': '"""Small single-cup transfer built on the unchanged, passed 06B worker.\n\nEngineering controller using privileged simulator state, not learned VLA behavior.\nThe new task is a 25 mm +world-X transfer. Runtime guards in the base are retained.\nOnly the inherited scratch planner predicts carried-object poses; physics execution\nuses actual actuator commands and contact, without state teleportation or welding.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport sys\nimport traceback\n\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport base06b as base\n\nOFFSET_M = np.array([0.025, 0.0, 0.0])\nPLACEMENT_TOL_M = 0.008\nTABLE_MARGIN_M = 0.005\nINSTRUCTION = (\n    "Lift the cup, move it 25 mm in the table\'s positive X direction, "\n    "place it on the table, and release it."\n)\n\n\ndef target_from_initial(initial, offset):\n    """Narrow this first transfer to the reviewed offset; do not grow the workspace."""\n    p, v = np.asarray(initial, dtype=float), np.asarray(offset, dtype=float)\n    if p.shape != (3,) or v.shape != (3,) or not np.isfinite(p).all() or not np.isfinite(v).all():\n        raise ValueError("Finite 3D initial position and offset required")\n    if not np.allclose(v, OFFSET_M, atol=1e-12, rtol=0):\n        raise ValueError("This first transfer is configured only for +25 mm world X")\n    return p + v\n\n\ndef placement_metrics(rows, initial, target):\n    """Independent measurements from executed samples; the old position cannot pass."""\n    if not rows:\n        raise ValueError("Final hold has no measured samples")\n    initial, target = np.asarray(initial, dtype=float), np.asarray(target, dtype=float)\n    if initial.shape != (3,) or target.shape != (3,) or not np.isfinite([initial, target]).all():\n        raise ValueError("Invalid placement references")\n    if not np.allclose(target - initial, OFFSET_M, atol=1e-10, rtol=0):\n        raise ValueError("Placement reference must be the new destination")\n    positions = np.array([[r[\'cup_x\'], r[\'cup_y\'], r[\'cup_z\']] for r in rows], dtype=float)\n    forces = np.array([[r[\'table_normal_n\'], r[\'fixed_normal_n\'], r[\'moving_normal_n\']]\n                       for r in rows], dtype=float)\n    if not np.isfinite(positions).all() or not np.isfinite(forces).all():\n        raise ValueError("Non-finite final measurements")\n    max_error = float(np.linalg.norm(positions - target, axis=1).max())\n    progress = float((positions[:, 0] - initial[0]).min())\n    supported = (forces[:, 0] >= base.CONTACT_MIN_N) & (forces[:, 1] < base.CONTACT_MIN_N) & (forces[:, 2] < base.CONTACT_MIN_N)\n    fraction = float(np.mean(supported))\n    return dict(\n        destination_position_m=target.tolist(),\n        maximum_destination_error_m=max_error,\n        minimum_x_displacement_m=progress,\n        supported_and_released_fraction=fraction,\n        placement_verified=bool(max_error <= PLACEMENT_TOL_M\n                                and progress >= OFFSET_M[0] - PLACEMENT_TOL_M\n                                and fraction >= 0.80),\n    )\n\n\ndef transfer_hold_metrics(rows, initial, target):\n    if not rows:\n        raise ValueError("No transfer-hold samples")\n    pos = np.array([[r[\'cup_x\'], r[\'cup_y\'], r[\'cup_z\']] for r in rows], dtype=float)\n    support = np.array([r[\'table_normal_n\'] for r in rows], dtype=float)\n    if not np.isfinite(pos).all() or not np.isfinite(support).all():\n        raise ValueError("Non-finite transfer measurements")\n    rise = float((pos[:, 2] - initial[2]).min())\n    error = float(np.linalg.norm(pos[:, :2] - target[:2], axis=1).max())\n    fraction = sum(bool(r[\'two_jaw_contact\']) for r in rows) / len(rows)\n    max_support = float(support.max())\n    return dict(minimum_cup_rise_m=rise, maximum_horizontal_target_error_m=error,\n                two_jaw_contact_fraction=fraction, max_table_normal_n=max_support,\n                transfer_verified=bool(rise >= .025 and error <= PLACEMENT_TOL_M\n                    and fraction >= .80 and max_support < base.CONTACT_MIN_N))\n\n\nclass TransferTrial(base.CupTrial):\n    def __init__(self, config):\n        super().__init__(config)\n        self.target = None\n        self.loss_steps = 0\n        self.report.update(\n            kind="ENGINEERING_RIGHT_CUP_TRANSFER_NOT_BIMANUAL_VLA",\n            task_revision="07-small-cup-transfer-v1",\n            transfer_test_passed=False, transfer_verified=False, placement_verified=False,\n            destination_offset_m=OFFSET_M.tolist(), placement_tolerance_m=PLACEMENT_TOL_M,\n            instruction_label=INSTRUCTION, instruction_interpreted=False,\n            lateral_runtime_limit_m=.035,\n            original_contact_force_joint_speed_and_drift_guards="retained via base06b",\n            recording_mode="paired_camera_frames" if config.get("record_data") else "snapshots_only",\n        )\n        self.report[\'notes\'] = [\n            "A new destination 25 mm along world X, not set-back to the original position.",\n            "One right-arm nominal engineering trial. Left-arm targets remain at home.",\n            "Simulator geometry and contact state are privileged controller inputs.",\n            "Instruction is a demonstration label, not evidence of language understanding.",\n            "No trained policy, camera-guided decisions, OpenVINO benchmark, or official score.",\n            "Success is measured against the destination; original 06B records are preserved.",\n        ]\n\n    def make_carry_reference(self, label, allow_cup):\n        """Extend the carry phases to transfer, retaining grasp/contact preconditions."""\n        self.verify_scratch_separation()\n        if label not in (\'lift\', \'transfer\', \'set_down\') or not allow_cup:\n            raise ValueError(\'Carry prediction allowed only for lift/transfer/set_down\')\n        if not self.report.get(\'grasp_verified\') or self.gripper_close_target is None:\n            raise RuntimeError(\'A verified grasp is required for carried-cup planning\')\n        if label in (\'transfer\', \'set_down\') and not self.report.get(\'lift_verified\'):\n            raise RuntimeError(\'Transfer/set-down requires a verified lift\')\n        f, _, issues = self.contacts(self.data, True)\n        if issues or min(f[\'fixed\'], f[\'moving\']) < base.CONTACT_MIN_N:\n            raise RuntimeError(\'Valid two-jaw contact required at carry-plan entry: \' + repr(issues))\n        m, d, mj = self.model, self.data, self.mj\n        jid = int(m.body_jntadr[self.cup])\n        if (int(m.body_parentid[self.cup]) != 0 or int(m.body_jntnum[self.cup]) != 1\n                or int(m.jnt_type[jid]) != int(mj.mjtJoint.mjJNT_FREE)):\n            raise ValueError(\'Expected the reviewed free cup body\')\n        qadr = int(m.jnt_qposadr[jid])\n        if qadr < 0 or qadr + 7 > len(self.scratch.qpos):\n            raise ValueError(\'Invalid free-cup address\')\n        local_p, local_R = base.relative_pose(\n            d.xpos[self.gripper_body], d.xmat[self.gripper_body].reshape(3, 3),\n            d.xpos[self.cup], d.xmat[self.cup].reshape(3, 3))\n        record = dict(mode=\'measured relative pose in SCRATCH ONLY; no physical attachment\',\n            label=label, cup_qpos_address=qadr, entry_sim_time_s=float(d.time),\n            relative_position_m=local_p.tolist(), relative_rotation=local_R.tolist(),\n            entry_jaw_normal_n=f, candidate_evaluations=0, minimum_sampled_contact_distance_m=None)\n        self.report[\'carry_plans\'][label] = record\n        self.event(\'carry_plan_reference\', phase=label, mode=record[\'mode\'])\n        return dict(qadr=qadr, position=local_p, rotation=local_R, record=record)\n\n    def check_destination(self):\n        """Static workspace precheck; actual path/contact checks still run separately."""\n        m, d, mj = self.model, self.data, self.mj\n        R = d.geom_xmat[self.table].reshape(3, 3)\n        if int(m.geom_type[self.table]) != int(mj.mjtGeom.mjGEOM_BOX) or not np.allclose(R[:, 2], [0, 0, 1], atol=1e-6):\n            raise ValueError(\'Expected unchanged horizontal box tabletop\')\n        local = R.T @ (self.target - d.geom_xpos[self.table])\n        remaining = m.geom_size[self.table, :2] - np.abs(local[:2]) - self.radius\n        if np.any(remaining < TABLE_MARGIN_M):\n            raise RuntimeError(\'Destination is too close to the table edge\')\n        prop_gaps = {}\n        for name in (\'plate\', \'fork\'):\n            bid = self.needed(mj.mjtObj.mjOBJ_BODY, \'development_\' + name)\n            ids = [g for g in range(m.ngeom) if int(m.geom_bodyid[g]) == bid]\n            if not ids:\n                raise ValueError(\'No geometry found for \' + name)\n            gaps = [float(np.linalg.norm(self.target[:2] - d.geom_xpos[g, :2])\n                          - self.radius - m.geom_rbound[g]) for g in ids]\n            prop_gaps[name] = min(gaps)\n            if min(gaps) < .005:\n                raise RuntimeError(\'Destination is too close to \' + name)\n        self.report[\'destination_precheck\'] = dict(\n            minimum_edge_clearance_m=float(remaining.min()),\n            conservative_prop_clearance_m=prop_gaps,\n            scope=\'initial workspace check; not continuous collision proof\')\n\n    def snapshot(self, label):\n        """Actual top-camera results improve visibility of the cup and destination."""\n        if self.renderer is None:\n            return\n        try:\n            self.renderer.update_scene(self.data, camera=\'top\', scene_option=self.options)\n            im = Image.fromarray(self.renderer.render().copy())\n            draw = ImageDraw.Draw(im)\n            draw.rectangle((0, 0, im.width, 42), fill=\'black\')\n            draw.text((8, 4), \'CUP TRANSFER | scripted engineering trial, not learned-policy output\', fill=\'white\')\n            draw.text((8, 23), f\'{label} | simulated t={self.data.time:.2f}s | target: +25 mm world X\', fill=\'white\')\n            p = self.out / f\'{label}.png\'\n            im.save(p)\n            self.snapshots[label] = str(p)\n        except Exception as exc:\n            self.report[\'graphics_error\'] = f\'{type(exc).__name__}: {exc}\'\n\n    def tick(self, command, label, allow_cup=False):\n        # The base call performs every original actuator/contact/force/drift check.\n        nobs = len(self.video_rows)\n        try:\n            row = super().tick(command, label, allow_cup)\n        finally:\n            # Label even a recorded step that the base subsequently rejects.\n            for obs in self.video_rows[nobs:]:\n                obs[\'instruction\'] = INSTRUCTION\n                obs[\'demonstration_task\'] = \'cup_transfer_positive_world_x_25mm\'\n        if label in (\'transfer\', \'transfer_hold\'):\n            if row[\'cup_z\'] - self.cup_initial[2] < .025:\n                raise RuntimeError(\'Cup lost required elevation during transfer\')\n            if row[\'table_normal_n\'] >= base.CONTACT_MIN_N:\n                raise RuntimeError(\'Cup was table-supported during transfer\')\n            self.loss_steps = 0 if row[\'two_jaw_contact\'] else self.loss_steps + 1\n            if self.loss_steps >= max(1, round(.10 / self.dt)):\n                raise RuntimeError(\'Two-jaw contact lost during transfer\')\n        return row\n\n    def execute(self):\n        self.load()\n        self.setup_render()\n        self.hold(.75, \'initial_settle\')\n        self.cup_initial = self.data.xpos[self.cup].copy()\n        self.target = target_from_initial(self.cup_initial, self.config[\'offset_m\'])\n        self.prop_reference = {n: self.data.xpos[self.needed(self.mj.mjtObj.mjOBJ_BODY, \'development_\' + n)].copy()\n                               for n in (\'plate\', \'fork\')}\n        self.report[\'initial_cup_position_m\'] = self.cup_initial.tolist()\n        self.report[\'destination_position_m\'] = self.target.tolist()\n        self.check_destination()\n        self.snapshot(\'start\')\n        self.move_jaw(self.open_q, \'open_gripper\')\n        delta = self.data.xpos[self.needed(self.mj.mjtObj.mjOBJ_BODY, \'right_mount\')] - self.cup_initial\n        delta[2] = 0\n        direction = base.normalize(delta)\n        axis = base.normalize(direction * math.sin(math.radians(25)) + [0, 0, math.cos(math.radians(25))])\n        pre = self.cup_initial.copy(); pre[2] = .15\n        grip = self.cup_initial.copy(); grip[2] = self.cup_initial[2] + .032\n        raised = grip + [0, 0, .040]\n        self.report.update(tool_axis=axis.tolist(), tool_tilt_command_deg=25,\n                           pregrasp_target_m=pre.tolist(), grasp_target_m=grip.tolist(), lift_target_m=raised.tolist())\n        p = self.plan(pre, axis, \'pregrasp\')\n        self.execute_plan(p, \'pregrasp\'); self.hold(.25, \'pregrasp_hold\')\n        p = self.plan(grip, axis, \'lower_to_grasp\', True)\n        self.execute_plan(p, \'lower_to_grasp\', True); self.hold(.25, \'grasp_approach_hold\', True)\n        if np.linalg.norm((self.data.xpos[self.cup] - self.cup_initial)[:2]) > .010:\n            raise RuntimeError(\'Cup shifted too far before closure\')\n        self.close_on_cup()\n        p = self.plan(raised, axis, \'lift\', True, carry_cup=True)\n        self.execute_plan(p, \'lift\', True); self.hold(.75, \'lift_hold\', True)\n        rows = self.stable_rows(\'lift_hold\', .5)\n        rise = min(float(r[\'cup_z\'] - self.cup_initial[2]) for r in rows)\n        fraction = sum(bool(r[\'two_jaw_contact\']) for r in rows) / len(rows)\n        support = max(r[\'table_normal_n\'] for r in rows)\n        self.report[\'lift_measurements\'] = dict(minimum_cup_rise_m=rise,\n            two_jaw_contact_fraction=fraction, max_table_normal_n=support)\n        if rise < .025 or fraction < .80 or support >= base.CONTACT_MIN_N:\n            raise RuntimeError(\'Lift not verified\')\n        self.report[\'lift_verified\'] = True; self.snapshot(\'lifted\')\n\n        # This is the new movement: carry to a destination, not back to the start.\n        transfer_target = self.gripper_point(self.data).copy()\n        transfer_target[:2] += self.target[:2] - self.data.xpos[self.cup, :2]\n        self.report[\'transfer_tool_target_m\'] = transfer_target.tolist()\n        p = self.plan(transfer_target, axis, \'transfer\', True, carry_cup=True)\n        self.execute_plan(p, \'transfer\', True); self.hold(.5, \'transfer_hold\', True)\n        transfer = transfer_hold_metrics(self.stable_rows(\'transfer_hold\'), self.cup_initial, self.target)\n        self.report[\'transfer_measurements\'] = transfer\n        if not transfer[\'transfer_verified\']:\n            raise RuntimeError(\'Destination hover or maintained lift/contact failed\')\n        self.report[\'transfer_verified\'] = True; self.snapshot(\'transferred\')\n\n        nominal_put = grip + OFFSET_M - [0, 0, .001]\n        put = self.set_down_target(nominal_put)\n        p = self.plan(put, axis, \'set_down\', True, carry_cup=True)\n        self.execute_plan(p, \'set_down\', True); self.hold(.5, \'support_hold\', True)\n        rows = self.stable_rows(\'support_hold\')\n        support_fraction = sum(r[\'table_normal_n\'] >= base.CONTACT_MIN_N for r in rows) / len(rows)\n        self.report[\'set_down_measurements\'] = dict(table_support_fraction=support_fraction)\n        if support_fraction < .8:\n            raise RuntimeError(\'Table support not established before opening\')\n        self.snapshot(\'supported\')\n        self.move_jaw(self.open_q, \'release_gripper\', True); self.snapshot(\'released\')\n        retreat = self.gripper_point(self.data).copy(); retreat[2] = .15\n        p = self.plan(retreat, axis, \'retreat\', True)\n        self.execute_plan(p, \'retreat\', True); self.hold(.75, \'final_settle\')\n        placement = placement_metrics(self.stable_rows(\'final_settle\', .5), self.cup_initial, self.target)\n        self.report[\'placement_measurements\'] = placement\n        self.report[\'release_measurements\'] = placement\n        if not placement[\'placement_verified\']:\n            raise RuntimeError(\'Supported release at NEW destination did not meet tolerance\')\n        self.report.update(placement_verified=True, release_verified=True, completed=True,\n                           grasp_test_passed=True, transfer_test_passed=True)\n        self.snapshot(\'final\')\n\n    def finish(self):\n        self.report[\'preservation_measurements\'] = {\n            name + \'_maximum_displacement_m\': max((float(r.get(name + \'_displacement_m\', 0.0)) for r in self.rows), default=None)\n            for name in (\'plate\', \'fork\')\n        }\n        super().finish()\n        # Parent writes complete traces/manifests using the same proven recorder.\n        self.report[\'transfer_test_passed\'] = bool(\n            self.report.get(\'completed\') and self.report.get(\'grasp_test_passed\')\n            and self.report.get(\'transfer_verified\') and self.report.get(\'placement_verified\')\n            and self.report.get(\'release_verified\') and not self.report.get(\'stops\')\n            and not self.report.get(\'source_scene_modified\'))\n        self.report[\'report_file\'] = str(self.out / \'transfer_report.json\')\n        base.write_json(self.out / \'transfer_report.json\', self.report)\n        # This compatibility report belongs to the NEW run, never the old checkpoint.\n        base.write_json(self.out / \'grasp_report.json\', self.report)\n\n\ndef main(config):\n    target_from_initial([0, 0, 0], config[\'offset_m\'])\n    if not isinstance(config.get(\'record_data\'), bool):\n        raise ValueError(\'record_data must be boolean\')\n    trial = TransferTrial(config)\n    try:\n        trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.event(\'stopped\', reason=trial.report[\'stops\'][-1])\n        (trial.out / \'failure_traceback.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n        if trial.renderer is not None:\n            trial.snapshot(\'stopped\')\n    finally:\n        trial.finish()\n    print(\'TRANSFER RESULT \' + json.dumps({k: trial.report.get(k) for k in (\n        \'completed\', \'transfer_test_passed\', \'grasp_verified\', \'lift_verified\',\n        \'transfer_verified\', \'release_verified\', \'physics_steps\', \'simulation_time_s\',\n        \'recorded_observation_action_pairs\', \'graphics_error\', \'stops\', \'report_file\')}, allow_nan=False), flush=True)\n    return 0 if trial.report[\'transfer_test_passed\'] else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_base.py': '"""Offline regression checks: synthetic kinematics/contact data, NOT SO-101 physics."""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport ast, hashlib, importlib.util, json, math, sys, tempfile, unittest\nimport numpy as np\n\nWORKER = Path(sys.argv[1] if len(sys.argv) > 1 else Path(__file__).with_name(\'grasp_worker.py\')).resolve()\n\nEXPECTED_WORKER_TEXT_SHA256 = \'027f2a91e66be817776dc36d4f2c434e1f2f1e16fdfee6ed01ad58f86134463e\'\n\n\ndef checked_worker_text(path):\n    """Accept platform line endings, but reject any other worker source change."""\n    raw = Path(path).read_bytes()\n    # Windows text-file writes convert LF to CRLF; normalize only that conversion.\n    normalized = raw.replace(b"\\r\\n", b"\\n")\n    if hashlib.sha256(normalized).hexdigest() != EXPECTED_WORKER_TEXT_SHA256:\n        raise ValueError("Worker source differs from the reviewed 06B implementation; no trial allowed")\n    return normalized.decode("utf-8")\n\n\nWORKER_TEXT = checked_worker_text(WORKER)\nspec=importlib.util.spec_from_file_location(\'grasp06b\',WORKER)\nmod=importlib.util.module_from_spec(spec);spec.loader.exec_module(mod)\n\n# Reference extracted from original notebook 06, worker SHA-256\n# ac7aae0914728c8605114e8be7726cd39cf2ea814355360dc681598d031b7035.\n# This reference is fixed, independent of the worker under test, and never executed.\n# Compare both parsed sources using the SAME interpreter, not persisted ast.dump hashes.\nBASELINE_METHODS_SOURCE = "class CupTrial:\\n    def load(self):\\n        import mujoco as mj\\n        self.mj=mj\\n        self.event(\'load_scene\')\\n        if digest(self.source)!=self.config[\'scene_sha256\']:\\n            raise ValueError(\'Passed hover scene changed; keep source unchanged\')\\n        # Verify the effort change only reduces an already finite scalar actuator limit.\\n        original=mj.MjModel.from_xml_path(str(self.source)); check_data=mj.MjData(original)\\n        mj.mj_forward(original,check_data); validate_mapping(original,check_data,mj)\\n        aid=mj.mj_name2id(original,mj.mjtObj.mjOBJ_ACTUATOR,\'right_gripper\')\\n        if not original.actuator_forcelimited[aid] or not (\\n            original.actuator_forcerange[aid,0]<=-GRIPPER_TORQUE_CAP and\\n            original.actuator_forcerange[aid,1]>=GRIPPER_TORQUE_CAP):\\n            raise ValueError(\'Cannot verify that the proposed gripper effort cap only reduces effort\')\\n        self.report[\'original_gripper_force_range_nm\']=original.actuator_forcerange[aid].tolist()\\n        copied=self.out/\'grasp_development.xml\'\\n        self.report[\'scene_adjustment\']=prepare_scene(self.source,copied)\\n        self.model=mj.MjModel.from_xml_path(str(copied));self.data=mj.MjData(self.model)\\n        m,d=self.model,self.data; mj.mj_forward(m,d)\\n        self.mapping=validate_mapping(m,d,mj);self.byname={r[\'name\']:r for r in self.mapping}\\n        self.arm=[self.byname[\'right_\'+j] for j in JOINT_SUFFIXES[:-1]]\\n        self.g=self.byname[\'right_gripper\'];self.dt=float(m.opt.timestep)\\n        if not 0<self.dt<=.01 or not math.isclose(round(.1/self.dt)*self.dt,.1,abs_tol=1e-9):\\n            raise ValueError(\'Unexpected timestep\')\\n        if np.any(np.asarray(m.body_gravcomp)>0):\\n            raise ValueError(\'Unexpected gravity compensation in this engineering model\')\\n        self.gripper_body=self.needed(mj.mjtObj.mjOBJ_BODY,\'right_gripper\')\\n        self.moving_body=self.needed(mj.mjtObj.mjOBJ_BODY,\'right_moving_jaw_so101_v1\')\\n        self.cup=self.needed(mj.mjtObj.mjOBJ_BODY,\'development_cup\')\\n        self.table=self.needed(mj.mjtObj.mjOBJ_GEOM,\'development_table\')\\n        self.fixed_tip=self.needed(mj.mjtObj.mjOBJ_GEOM,\'right_fixed_jaw_sph_tip1\')\\n        self.moving_tip=self.needed(mj.mjtObj.mjOBJ_GEOM,\'right_moving_jaw_sph_tip1\')\\n        if int(m.geom_bodyid[self.fixed_tip])!=self.gripper_body or int(m.geom_bodyid[self.moving_tip])!=self.moving_body:\\n            raise ValueError(\'Unexpected fingertip ownership\')\\n        if any(int(m.geom_type[t])!=int(mj.mjtGeom.mjGEOM_SPHERE) for t in (self.fixed_tip,self.moving_tip)):\\n            raise ValueError(\'Expected named spherical tip markers/collision geoms\')\\n        if int(m.body_jntnum[self.cup])!=1 or int(m.jnt_type[int(m.body_jntadr[self.cup])])!=int(mj.mjtJoint.mjJNT_FREE):\\n            raise ValueError(\'Cup must be a freely moving rigid body\')\\n        # Intentionally narrow to the known, upright faceted hollow development cup.\\n        self.cup_geoms={int(g) for g in range(m.ngeom) if int(m.geom_bodyid[g])==self.cup}\\n        walls=[g for g in self.cup_geoms if self.name(mj.mjtObj.mjOBJ_GEOM,g).startswith(\'cup_wall_\')]\\n        if len(walls)!=16:\\n            raise ValueError(\'This first grasp is configured for the existing 16-wall development cup\')\\n        radius=0.\\n        for g in walls:\\n            if int(m.geom_type[g])!=int(mj.mjtGeom.mjGEOM_BOX):\\n                raise ValueError(\'Expected primitive box walls\')\\n            R=d.geom_xmat[g].reshape(3,3)\\n            for sx in (-1,1):\\n                for sy in (-1,1):\\n                    v=d.geom_xpos[g]+R@np.array([sx*m.geom_size[g,0],sy*m.geom_size[g,1],0])\\n                    radius=max(radius,float(np.linalg.norm((v-d.xpos[self.cup])[:2])))\\n        if not .020<=radius<=.030:\\n            raise ValueError(\'Cup radius outside the reviewed small-object range\')\\n        self.radius=radius\\n        # A body-attached control point near the nominal cup center at fingertip height.\\n        fixed_local=m.geom_pos[self.fixed_tip].copy()\\n        self.local_point=fixed_local+np.array([radius+float(m.geom_size[self.fixed_tip,0])+.0018,0,0])\\n        self.open_q=.70; self.close_floor=.15\\n        if not self.g[\'low\']+.01<self.close_floor<self.open_q<self.g[\'high\']-.01:\\n            raise ValueError(\'Proposed jaw commands outside validated bounds\')\\n        for item in self.mapping:d.ctrl[item[\'cid\']]=item[\'q0\']\\n        self.warnings0=np.array(d.warning.number,copy=True)\\n        self.scratch=mj.MjData(m);self.scratch.qpos[:]=d.qpos\\n        distances={}\\n        for q in (self.g[\'q0\'],self.open_q):\\n            self.scratch.qpos[self.g[\'qadr\']]=q;mj.mj_forward(m,self.scratch)\\n            distance=np.linalg.norm(self.scratch.geom_xpos[self.fixed_tip]-self.scratch.geom_xpos[self.moving_tip])\\n            distance-=float(m.geom_size[self.fixed_tip,0]+m.geom_size[self.moving_tip,0])\\n            distances[str(q)]=float(distance)\\n        if distances[str(self.open_q)]<=2*radius+.006:\\n            raise RuntimeError(\'Proposed open-tip gap is not wide enough for the nominal cup\')\\n        self.geom_group={};self.robot_geoms=set()\\n        for gid in range(m.ngeom):\\n            bid=int(m.geom_bodyid[gid]);cur=bid;side=None\\n            while cur:\\n                n=self.name(mj.mjtObj.mjOBJ_BODY,cur)\\n                if n.startswith((\'left_\',\'right_\')):side=n.split(\'_\')[0];break\\n                cur=int(m.body_parentid[cur])\\n            if side:self.robot_geoms.add(gid)\\n            n=self.name(mj.mjtObj.mjOBJ_GEOM,gid)\\n            mesh=\'\'\\n            if int(m.geom_type[gid])==int(mj.mjtGeom.mjGEOM_MESH):\\n                mesh=self.name(mj.mjtObj.mjOBJ_MESH,int(m.geom_dataid[gid]))\\n            if bid==self.gripper_body and (n.startswith(\'right_fixed_jaw_sph_\') or n in\\n                {\'right_fixed_jaw_box3\',\'right_fixed_jaw_box4\',\'right_fixed_jaw_box5\',\'right_fixed_jaw_box6\',\'right_fixed_jaw_box7\'}\\n                or mesh==\'wrist_roll_follower_so101_gripper_part0_v1\'):\\n                self.geom_group[gid]=\'fixed\'\\n            elif bid==self.moving_body:\\n                self.geom_group[gid]=\'moving\'\\n        self.arm_indices=[r[\'qadr\'] for r in self.arm];self.arm_dofs=[r[\'vadr\'] for r in self.arm]\\n        self.low=np.array([r[\'low\']+.02 for r in self.arm]);self.high=np.array([r[\'high\']-.02 for r in self.arm])\\n        self.prop_reference={n:d.xpos[self.needed(mj.mjtObj.mjOBJ_BODY,\'development_\'+n)].copy() for n in (\'plate\',\'fork\')}\\n        self.report.update(scene=str(copied),scene_sha256=digest(copied),source_scene=str(self.source),\\n            mujoco_version=mj.__version__,python=sys.executable,timestep_s=self.dt,\\n            native_actuator_mapping=self.mapping,grasp_reference_local_m=self.local_point.tolist(),\\n            cup_outer_radius_m=radius,tip_gap_diagnostic_m=distances,\\n            requested_jaw_open_rad=self.open_q,maximum_closure_floor_rad=self.close_floor,\\n            thresholds=dict(lift_required_m=.025,lift_command_m=.040,release_position_m=.015,\\n              allowed_cup_tilt_deg=30,contact_fraction=.80,gripper_force_limit_nm=GRIPPER_TORQUE_CAP,\\n              penetration_limit_m=PENETRATION_LIMIT,contact_min_normal_n=CONTACT_MIN_N))\\n        self.event(\'scene_ready\',cup_diameter_mm=round(2000*radius,3),jaw_open_rad=self.open_q)\\n\\n    def contacts(self,data,allow_cup,planning=False):\\n        m,mj=self.model,self.mj\\n        f={\'fixed\':0.,\'moving\':0.};table=0.;issues=[]\\n        for i in range(data.ncon):\\n            c=data.contact[i]; a,b=int(c.geom1),int(c.geom2);dist=float(c.dist)\\n            if a<0 or b<0 or not math.isfinite(dist):raise RuntimeError(\'Unsupported contact\')\\n            ca,cb=a in self.cup_geoms,b in self.cup_geoms\\n            other=b if ca else a\\n            jaw=self.geom_group.get(other) if ca or cb else None\\n            robot=a in self.robot_geoms or b in self.robot_geoms\\n            intended=bool(allow_cup and jaw and (ca or cb))\\n            if planning and intended:\\n                # The planning cup is not attached or moved. Real cup contacts are checked at every executed step.\\n                continue\\n            force=np.zeros(6)\\n            if c.efc_address>=0:mj.mj_contactForce(m,data,i,force)\\n            normal=max(0.,float(force[0]))\\n            if jaw and (ca or cb) and dist<=.0005:f[jaw]+=normal\\n            if (ca or cb) and other==self.table:table+=normal\\n            if dist < -PENETRATION_LIMIT:issues.append(\'excess penetration\')\\n            if robot and dist<=0 and not intended:\\n                issues.append(f\'unexpected robot contact: {self.name(mj.mjtObj.mjOBJ_GEOM,a)} / {self.name(mj.mjtObj.mjOBJ_GEOM,b)}\')\\n            if intended and normal>25:issues.append(\'jaw contact normal force exceeds engineering bound\')\\n        return f,table,issues\\n\\n    def snapshot(self,label):\\n        if self.renderer is None:return\\n        try:\\n            self.renderer.update_scene(self.data,camera=self.view,scene_option=self.options)\\n            im=Image.fromarray(self.renderer.render().copy())\\n            draw=ImageDraw.Draw(im);draw.rectangle((0,0,640,45),fill=\'black\')\\n            draw.text((8,4),\'CONTACT-GRASP ENGINEERING TEST | not trained VLA / not table-setting\',fill=\'white\')\\n            draw.text((8,23),f\'{label} | simulated t={self.data.time:.2f}s\',fill=\'white\')\\n            p=self.out/f\'{label}.png\';im.save(p);self.snapshots[label]=str(p)\\n        except Exception as exc:self.report[\'graphics_error\']=f\'{type(exc).__name__}: {exc}\'\\n\\n    def tick(self,command,label,allow_cup=False):\\n        m,d,mj=self.model,self.data,self.mj;self.phase=label;self.budget()\\n        command=np.asarray(command,dtype=float)\\n        if command.shape!=(12,) or not np.isfinite(command).all():raise ValueError(\'Invalid 12-control command\')\\n        if np.any(np.abs(command-d.ctrl)>COMMAND_SPEED*self.dt*1.06+1e-8):\\n            raise RuntimeError(\'Command slew guard\')\\n        for r in self.mapping:\\n            if not r[\'low\']<=command[r[\'cid\']]<=r[\'high\']:raise ValueError(\'Command outside joint/control limits\')\\n        # BEFORE-action observation and the exact command applied in the following step.\\n        if (self.config.get(\'record_data\',False) and self.renderer is not None\\n            and self.steps%round(.1/self.dt)==0):\\n            try:\\n                obs_dir=self.out/\'observations\';obs_dir.mkdir(exist_ok=True)\\n                frame_id=len(self.video_rows);image_paths={}\\n                for cam in (\'top\',\'right_wrist_cam\'):\\n                    if mj.mj_name2id(m,mj.mjtObj.mjOBJ_CAMERA,cam)<0:raise ValueError(f\'Camera missing: {cam}\')\\n                    self.renderer.update_scene(d,camera=cam,scene_option=self.options)\\n                    pix=self.renderer.render().copy();file=obs_dir/f\'{frame_id:05d}_{cam}.jpg\'\\n                    Image.fromarray(pix).save(file,quality=85)\\n                    image_paths[cam]=str(file.relative_to(self.out))\\n                self.video_rows.append(dict(frame_id=frame_id,step=self.steps,time_s=float(d.time),images=image_paths,\\n                    instruction=\'Lift the cup, hold it, and set it back on the table.\',\\n                    state=[float(d.qpos[r[\'qadr\']]) for r in self.mapping],\\n                    action=[float(command[r[\'cid\']]) for r in self.mapping],action_order=[r[\'name\'] for r in self.mapping],\\n                    action_semantics=\'native absolute joint position targets, radians; applied immediately after observation\',\\n                    expert_source=\'privileged simulator-state engineering controller; not learned-policy output\'))\\n            except Exception as exc:\\n                self.report[\'graphics_error\']=f\'recording: {type(exc).__name__}: {exc}\'\\n                self.config[\'record_data\']=False\\n        d.ctrl[:]=command\\n        previous=float(d.time);mj.mj_step(m,d);self.steps+=1;mj.mj_forward(m,d)\\n        if not math.isclose(float(d.time),previous+self.dt,abs_tol=1e-8):raise RuntimeError(\'Unexpected simulation time/reset\')\\n        if not np.isfinite(d.qpos).all() or not np.isfinite(d.qvel).all():raise RuntimeError(\'Non-finite simulation\')\\n        if np.any(np.asarray(d.warning.number)>self.warnings0):raise RuntimeError(\'MuJoCo numerical/capacity warning\')\\n        f,support,issues=self.contacts(d,allow_cup)\\n        xyz=d.xpos[self.cup].copy();tilt=math.degrees(math.acos(float(np.clip(d.xmat[self.cup].reshape(3,3)[2,2],-1,1))))\\n        row=dict(step=self.steps,time_s=float(d.time),phase=label,cup_x=xyz[0],cup_y=xyz[1],cup_z=xyz[2],\\n            cup_tilt_deg=tilt,fixed_normal_n=f[\'fixed\'],moving_normal_n=f[\'moving\'],table_normal_n=support,\\n            two_jaw_contact=f[\'fixed\']>=CONTACT_MIN_N and f[\'moving\']>=CONTACT_MIN_N)\\n        for r in self.mapping:\\n            q,v=float(d.qpos[r[\'qadr\']]),float(d.qvel[r[\'vadr\']])\\n            row[r[\'name\']+\'_position_rad\']=q;row[r[\'name\']+\'_command_rad\']=float(command[r[\'cid\']])\\n            if not r[\'low\']-.03<=q<=r[\'high\']+.03:issues.append(\'joint range: \'+r[\'name\'])\\n            if abs(v)>3.:issues.append(\'joint speed: \'+r[\'name\'])\\n            # Contact blocks the jaw, so ordinary free-motion tracking does not apply to that one joint.\\n            limit=.65 if allow_cup and r[\'name\']==\'right_gripper\' else .12\\n            if abs(q-command[r[\'cid\']])>limit:issues.append(\'tracking: \'+r[\'name\'])\\n            if r[\'name\'].startswith(\'left_\') and abs(q-r[\'q0\'])>.15:issues.append(\'inactive arm drift\')\\n        for name,p0 in self.prop_reference.items():\\n            b=self.needed(mj.mjtObj.mjOBJ_BODY,\'development_\'+name)\\n            dist=float(np.linalg.norm(d.xpos[b]-p0));row[name+\'_displacement_m\']=dist\\n            if dist>.010:issues.append(\'other object moved: \'+name)\\n        if tilt>30.:issues.append(\'cup tilted beyond engineering bound\')\\n        if hasattr(self,\'cup_initial\') and np.linalg.norm(xyz[:2]-self.cup_initial[:2])>.035:\\n            issues.append(\'cup lateral displacement too large for set-back trial\')\\n        self.rows.append(row)\\n        if issues:raise RuntimeError(\'; \'.join(issues[:3]))\\n        return row\\n\\n    def move_jaw(self,target,label,allow_cup=False):\\n        start=self.data.ctrl.copy();end=start.copy();end[self.g[\'cid\']]=target\\n        seconds=math.ceil(max(.6,1.6*abs(target-start[self.g[\'cid\']])/COMMAND_SPEED)/self.dt)*self.dt\\n        n=round(seconds/self.dt);self.event(\'segment_start\',phase=label,simulated_seconds=seconds)\\n        for k in range(1,n+1):self.tick(start+smooth(k/n)*(end-start),label,allow_cup)\\n        self.event(\'segment_complete\',phase=label)\\n\\n    def hold(self,seconds,label,allow_cup=False):\\n        self.event(\'segment_start\',phase=label,simulated_seconds=seconds)\\n        for _ in range(round(seconds/self.dt)):self.tick(self.data.ctrl.copy(),label,allow_cup)\\n        self.event(\'segment_complete\',phase=label)\\n\\n    def execute_plan(self,plan,label,allow_cup=False):\\n        n=round(plan[\'duration_s\']/self.dt);base=self.data.ctrl.copy()\\n        self.event(\'segment_start\',phase=label,simulated_seconds=plan[\'duration_s\'])\\n        for k in range(1,n+1):\\n            cmd=base.copy();q=path_value(plan,smooth(k/n))\\n            for r,value in zip(self.arm,q):cmd[r[\'cid\']]=value\\n            self.tick(cmd,label,allow_cup)\\n        self.event(\'segment_complete\',phase=label)\\n\\n    def stable_rows(self,label,seconds=.25):\\n        rows=[r for r in self.rows if r[\'phase\']==label]\\n        if not rows:raise RuntimeError(\'No measured samples for \'+label)\\n        end=rows[-1][\'time_s\'];return [r for r in rows if r[\'time_s\']>=end-seconds-1e-9]\\n\\n    def close_on_cup(self):\\n        self.event(\'segment_start\',phase=\'close_gripper\')\\n        start=float(self.data.ctrl[self.g[\'cid\']]);n=round(4./self.dt);consecutive=0\\n        for k in range(1,n+1):\\n            cmd=self.data.ctrl.copy();cmd[self.g[\'cid\']]=start+smooth(k/n)*(self.close_floor-start)\\n            row=self.tick(cmd,\'close_gripper\',True)\\n            consecutive=consecutive+1 if row[\'two_jaw_contact\'] else 0\\n            if consecutive>=max(1,round(.10/self.dt)):\\n                self.gripper_close_target=float(cmd[self.g[\'cid\']]);break\\n        if self.gripper_close_target is None:\\n            raise RuntimeError(\'No sustained two-jaw contact; the cup will not be lifted\')\\n        self.hold(.4,\'grasp_hold\',True);rows=self.stable_rows(\'grasp_hold\')\\n        fraction=sum(bool(r[\'two_jaw_contact\']) for r in rows)/len(rows)\\n        self.report[\'grasp_contact_fraction\']=fraction\\n        if fraction<.80:raise RuntimeError(\'Two-jaw contact did not remain stable before lift\')\\n        self.report[\'grasp_verified\']=True;self.snapshot(\'grasped\')\\n        self.event(\'grasp_contact_verified\',fraction=fraction,command_rad=self.gripper_close_target)\\n"\nBASELINE_METHODS_SHA256 = \'149ed9d58b32f2a9ebb0dd762ecc10d107c33b6d140651d0aa6e177040acc577\'\nPRESERVED_METHOD_NAMES = (\'load\', \'contacts\', \'snapshot\', \'tick\', \'move_jaw\', \'hold\', \'execute_plan\', \'stable_rows\', \'close_on_cup\')\n\nBASE_CONSTANTS = {\'JOINT_SUFFIXES\': (\'shoulder_pan\', \'shoulder_lift\', \'elbow_flex\', \'wrist_flex\', \'wrist_roll\', \'gripper\'), \'COMMAND_SPEED\': 0.35, \'POSITION_TOL\': 0.0005, \'AXIS_TOL_DEG\': 2.0, \'ORIENTATION_WEIGHT\': 0.08, \'CONTACT_MIN_N\': 0.02, \'GRIPPER_TORQUE_CAP\': 0.25, \'PENETRATION_LIMIT\': 0.003, \'MAX_WALL_SECONDS\': 240.0}\n\n\ndef runtime_method_nodes(source):\n    """Parse exactly one reference class; reject ambiguous/duplicate methods."""\n    tree = ast.parse(source)\n    classes = [n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == "CupTrial"]\n    if len(classes) != 1:\n        raise ValueError("Exactly one CupTrial class is required")\n    result = {}\n    for node in classes[0].body:\n        if isinstance(node, (ast.FunctionDef, ast.AsyncFunctionDef)):\n            if node.name in result:\n                raise ValueError("Duplicate method: " + node.name)\n            result[node.name] = node\n    return result\n\n\ndef changed_runtime_methods(source, reference=BASELINE_METHODS_SOURCE):\n    """Return changed/missing reviewed methods, with no cross-version dump hashes."""\n    actual = runtime_method_nodes(source)\n    baseline = runtime_method_nodes(reference)\n    if set(baseline) != set(PRESERVED_METHOD_NAMES):\n        raise ValueError("The independent reference method set changed")\n    return [name for name in PRESERVED_METHOD_NAMES\n            if name not in actual or\n            ast.dump(actual[name], include_attributes=False) !=\n            ast.dump(baseline[name], include_attributes=False)]\n\ndef rz(a):\n    c,s=math.cos(a),math.sin(a)\n    return np.array([[c,-s,0],[s,c,0],[0,0,1.]])\n\ndef rx(a):\n    c,s=math.cos(a),math.sin(a)\n    return np.array([[1,0,0],[0,c,-s],[0,s,c]])\n\nclass FakeMJ:\n    mjtJoint=SimpleNamespace(mjJNT_FREE=0)\n    mjtObj=SimpleNamespace(mjOBJ_GEOM=1)\n    mjtGeom=SimpleNamespace(mjGEOM_BOX=6,mjGEOM_CYLINDER=5)\n    @staticmethod\n    def mj_kinematics(m,d):\n        d.xpos[1]=[0,0,d.qpos[0]]\n        d.xmat[1]=rz(d.qpos[1]).ravel()\n        d.xpos[2]=d.qpos[6:9]\n        w,x,y,z=d.qpos[9:13]\n        d.xmat[2]=np.array([[1-2*(y*y+z*z),2*(x*y-z*w),2*(x*z+y*w)],\n                          [2*(x*y+z*w),1-2*(x*x+z*z),2*(y*z-x*w)],\n                          [2*(x*z-y*w),2*(y*z+x*w),1-2*(x*x+y*y)]]).ravel()\n    @classmethod\n    def mj_forward(cls,m,d):\n        cls.mj_kinematics(m,d)\n        contacts=[]\n        # Two intended contacts for a cup held at a 25 mm body-frame offset.\n        if abs(d.xpos[2,2]-(d.xpos[1,2]-.025)) < .003:\n            contacts += [SimpleNamespace(geom1=0,geom2=2,dist=-.0001,efc_address=0),\n                         SimpleNamespace(geom1=1,geom2=2,dist=-.0001,efc_address=0)]\n        # Synthetic proxy reproduces the stale-object false collision; it is not a mesh test.\n        dist=abs((d.xpos[1,2]+.015)-d.xpos[2,2])-.012\n        if dist<=0:\n            contacts += [SimpleNamespace(geom1=3,geom2=2,dist=dist,efc_address=0)]\n        d.contact=contacts;d.ncon=len(contacts)\n    @staticmethod\n    def mj_jac(m,d,jp,jr,p,body):\n        jp[:]=0;jr[:]=0;jp[2,0]=1;jr[2,1]=1\n    @staticmethod\n    def mju_mat2Quat(out,flat):\n        # All synthetic carrier poses in these tests rotate only about Z.\n        R=np.asarray(flat).reshape(3,3);angle=math.atan2(R[1,0],R[0,0])\n        out[:]=[math.cos(angle/2),0,0,math.sin(angle/2)]\n    @staticmethod\n    def mj_contactForce(m,d,index,force):\n        force[:]=0;force[0]=1.\n    @staticmethod\n    def mj_id2name(m,kind,gid):\n        return {0:\'right_fixed_jaw_sph_tip1\',1:\'right_moving_jaw_sph_tip1\',\n                2:\'cup_wall_12\',3:\'right_fixed_jaw_box1\',4:\'development_table\'}[int(gid)]\n\n\ndef data():\n    q=np.zeros(13);q[0]=.14;q[5]=.3;q[6:9]=[0,0,.115];q[9]=1\n    d=SimpleNamespace(qpos=q,qvel=np.zeros(12),ctrl=np.array([.14,0,0,0,0,.28]),\n                      xpos=np.zeros((3,3)),xmat=np.tile(np.eye(3).ravel(),(3,1)),\n                      time=1.,contact=[],ncon=0)\n    return d\n\n\ndef trial(out):\n    t=mod.CupTrial.__new__(mod.CupTrial)\n    t.mj=FakeMJ\n    t.model=SimpleNamespace(nv=12,body_jntadr=np.array([-1,0,6]),\n        body_parentid=np.array([0,0,0]),body_jntnum=np.array([0,1,1]),\n        jnt_type=np.array([3]*6+[0]),jnt_qposadr=np.array([0,1,2,3,4,5,6]))\n    t.data=data();t.scratch=data();FakeMJ.mj_forward(t.model,t.data)\n    t.gripper_body=1;t.cup=2;t.table=4;t.local_point=np.zeros(3)\n    t.arm_indices=list(range(5));t.arm_dofs=list(range(5))\n    t.arm=[{\'cid\':i,\'qadr\':i} for i in range(5)]\n    t.low=np.full(5,-.5);t.high=np.full(5,.5);t.dt=.005\n    t.gripper_close_target=.28;t.cup_geoms={2};t.geom_group={0:\'fixed\',1:\'moving\'}\n    t.robot_geoms={0,1,3}\n    t.report={\'grasp_verified\':True,\'lift_verified\':True,\'carry_plans\':{},\n              \'planning_only_cup_pose_prediction\':False}\n    t.budget=lambda:None;t.event=lambda *a,**k:None;t.out=Path(out)\n    return t\n\nclass PoseTests(unittest.TestCase):\n    def test_relative_identity(self):\n        p,R=mod.relative_pose([1,2,3],np.eye(3),[1.1,2.2,3.3],np.eye(3))\n        np.testing.assert_allclose(p,[.1,.2,.3]);np.testing.assert_allclose(R,np.eye(3))\n    def test_translation_follows(self):\n        p,R=mod.predicted_pose([0,0,.1],np.eye(3),[0,0,-.025],np.eye(3))\n        np.testing.assert_allclose(p,[0,0,.075])\n    def test_rotation_follows(self):\n        p,R=mod.predicted_pose([1,0,0],rz(math.pi/2),[1,0,0],rx(.2))\n        np.testing.assert_allclose(p,[1,1,0],atol=1e-12)\n        np.testing.assert_allclose(R,rz(math.pi/2)@rx(.2))\n    def test_roundtrip_observed_pose(self):\n        p=np.array([.2,-.1,.4]);R=rz(.8)@rx(.4);c=np.array([.23,-.16,.38]);C=rx(-.3)\n        a,A=mod.relative_pose(p,R,c,C);cc,CC=mod.predicted_pose(p,R,a,A)\n        np.testing.assert_allclose(cc,c);np.testing.assert_allclose(CC,C,atol=1e-12)\n    def test_bad_shape(self):\n        with self.assertRaises(ValueError):mod.checked_pose([1,2],np.eye(3))\n    def test_nan_rejected(self):\n        with self.assertRaises(ValueError):mod.checked_pose([1,np.nan,0],np.eye(3))\n    def test_reflection_rejected(self):\n        with self.assertRaises(ValueError):mod.checked_pose([0,0,0],np.diag([1,1,-1]))\n    def test_non_rotation_rejected(self):\n        with self.assertRaises(ValueError):mod.checked_pose([0,0,0],2*np.eye(3))\n    def test_no_argument_mutation(self):\n        p=np.ones(3);R=rz(.2);c=np.array([1,2,3.]);C=rx(.1)\n        original=[x.copy() for x in (p,R,c,C)]\n        mod.relative_pose(p,R,c,C)\n        for got,expected in zip((p,R,c,C),original):np.testing.assert_array_equal(got,expected)\n\nclass SurfaceTests(unittest.TestCase):\n    def test_box_horizontal(self):\n        self.assertAlmostEqual(mod.primitive_plane_radius(\'box\',[.02,.03,.04],np.eye(3),[0,0,1]),.04)\n    def test_box_rotated(self):\n        self.assertAlmostEqual(mod.primitive_plane_radius(\'box\',[.02,.03,.04],rx(math.pi/2),[0,0,1]),.03)\n    def test_cylinder_upright(self):\n        self.assertAlmostEqual(mod.primitive_plane_radius(\'cylinder\',[.025,.003,0],np.eye(3),[0,0,1]),.003)\n    def test_cylinder_tilted(self):\n        a=.3;want=.025*math.sin(a)+.003*math.cos(a)\n        self.assertAlmostEqual(mod.primitive_plane_radius(\'cylinder\',[.025,.003,0],rx(a),[0,0,1]),want)\n    def test_unknown_primitive_rejected(self):\n        with self.assertRaises(ValueError):mod.primitive_plane_radius(\'mesh\',[1,1,1],np.eye(3),[0,0,1])\n    def test_negative_size_rejected(self):\n        with self.assertRaises(ValueError):mod.primitive_plane_radius(\'box\',[1,-1,1],np.eye(3),[0,0,1])\n    def test_zero_normal_rejected(self):\n        with self.assertRaises(ValueError):mod.primitive_plane_radius(\'box\',[1,1,1],np.eye(3),[0,0,0])\n    def test_drop_uses_measured_clearance(self):\n        np.testing.assert_allclose(mod.measured_lowering_target([.1,.1,.1],[0,0,1],.0378),[.1,.1,.0612])\n    def test_drop_has_unchanged_one_mm_allowance(self):\n        self.assertAlmostEqual(mod.measured_lowering_target([0,0,.1],[0,0,1],0)[2],.099)\n    def test_negative_clearance_rejected(self):\n        with self.assertRaises(ValueError):mod.measured_lowering_target([0,0,.1],[0,0,1],-.002)\n    def test_excess_drop_rejected(self):\n        with self.assertRaises(ValueError):mod.measured_lowering_target([0,0,.1],[0,0,1],.081)\n    def test_excess_compression_rejected(self):\n        with self.assertRaises(ValueError):mod.measured_lowering_target([0,0,.1],[0,0,1],.04,.003)\n    def test_nonfinite_drop_rejected(self):\n        with self.assertRaises(ValueError):mod.measured_lowering_target([0,0,.1],[0,0,1],float(\'nan\'))\n\nclass CarryTests(unittest.TestCase):\n    def setUp(self):\n        self.temp=tempfile.TemporaryDirectory();self.addCleanup(self.temp.cleanup)\n        self.t=trial(self.temp.name)\n    def test_capture_measured_reference(self):\n        ref=self.t.make_carry_reference(\'set_down\',True)\n        np.testing.assert_allclose(ref[\'position\'],[0,0,-.025])\n    def test_before_grasp_rejected(self):\n        self.t.report[\'grasp_verified\']=False\n        with self.assertRaises(RuntimeError):self.t.make_carry_reference(\'lift\',True)\n    def test_set_down_before_lift_rejected(self):\n        self.t.report[\'lift_verified\']=False\n        with self.assertRaises(RuntimeError):self.t.make_carry_reference(\'set_down\',True)\n    def test_wrong_stage_rejected(self):\n        with self.assertRaises(ValueError):self.t.make_carry_reference(\'retreat\',True)\n    def test_contacts_not_allowed_rejected(self):\n        with self.assertRaises(ValueError):self.t.make_carry_reference(\'lift\',False)\n    def test_lost_jaw_contact_rejected(self):\n        self.t.data.qpos[8]=.3;FakeMJ.mj_forward(self.t.model,self.t.data)\n        with self.assertRaises(RuntimeError):self.t.make_carry_reference(\'lift\',True)\n    def test_alias_object_rejected(self):\n        self.t.scratch=self.t.data\n        with self.assertRaises(RuntimeError):self.t.verify_scratch_separation()\n    def test_alias_array_rejected(self):\n        self.t.scratch.qpos=self.t.data.qpos.view()\n        with self.assertRaises(RuntimeError):self.t.verify_scratch_separation()\n    def test_nonfree_cup_rejected(self):\n        self.t.model.jnt_type[6]=3\n        with self.assertRaises(ValueError):self.t.make_carry_reference(\'lift\',True)\n    def test_copy_moves_execution_unchanged(self):\n        ref=self.t.make_carry_reference(\'set_down\',True)\n        before=self.t.data.qpos.copy();self.t.scratch.qpos[0]=.10\n        self.t.update_scratch_cup(ref);FakeMJ.mj_forward(self.t.model,self.t.scratch)\n        self.assertAlmostEqual(self.t.scratch.xpos[2,2],.075)\n        np.testing.assert_array_equal(self.t.data.qpos,before)\n    def test_prediction_not_incrementally_accumulated(self):\n        ref=self.t.make_carry_reference(\'lift\',True)\n        for z in [.2,.1,.14,.1]:\n            self.t.scratch.qpos[0]=z;self.t.update_scratch_cup(ref)\n            self.assertAlmostEqual(self.t.scratch.qpos[8],z-.025)\n    def test_proximal_contact_is_still_rejected(self):\n        d=self.t.data\n        d.contact=[SimpleNamespace(geom1=3,geom2=2,dist=-.001,efc_address=0)];d.ncon=1\n        _,_,issues=self.t.contacts(d,True,planning=True)\n        self.assertTrue(any(\'right_fixed_jaw_box1\' in x for x in issues))\n    def test_excess_intended_penetration_still_rejected(self):\n        ref=self.t.make_carry_reference(\'set_down\',True)\n        self.t.scratch.contact=[SimpleNamespace(geom1=0,geom2=2,dist=-.004,efc_address=0)];self.t.scratch.ncon=1\n        self.assertTrue(self.t.extra_carry_contact_issues(ref))\n    def test_stationary_cup_reproduces_false_collision(self):\n        with self.assertRaisesRegex(RuntimeError,\'sampled path rejected\'):\n            self.t.plan(np.array([0,0,.10]),np.array([0,0,1]),\'set_down\',True,carry_cup=False)\n    def test_carried_prediction_passes_same_synthetic_path(self):\n        before=self.t.data.qpos.copy()\n        p=self.t.plan(np.array([0,0,.10]),np.array([0,0,1]),\'set_down\',True,carry_cup=True)\n        self.assertTrue(p[\'carried_object_prediction\'][\'candidate_evaluations\']>0)\n        np.testing.assert_array_equal(self.t.data.qpos,before)\n    def test_noncarrying_plan_has_no_object_prediction(self):\n        self.t.data.qpos[8] = -.1  # A free, distant cup for the approach case.\n        FakeMJ.mj_forward(self.t.model, self.t.data)\n        p=self.t.plan(np.array([0,0,.15]),np.array([0,0,1]),\'pregrasp\',False)\n        self.assertIsNone(p[\'carried_object_prediction\'])\n        self.assertFalse(self.t.report[\'planning_only_cup_pose_prediction\'])\n\n    def configure_table_geometry(self):\n        t=self.t\n        t.model.geom_type=np.zeros(5,dtype=int);t.model.geom_type[2]=5;t.model.geom_type[4]=6\n        t.model.geom_size=np.zeros((5,3));t.model.geom_size[2]=[.025,.003,0];t.model.geom_size[4]=[.48,.38,.03]\n        t.data.geom_xmat=np.tile(np.eye(3).ravel(),(5,1))\n        t.data.geom_xpos=np.zeros((5,3));t.data.geom_xpos[2]=[0,0,.073]\n    def test_set_down_height_reads_actual_table_and_cup(self):\n        self.configure_table_geometry()\n        p=self.t.set_down_target(np.array([0,0,.09]))\n        np.testing.assert_allclose(p,[0,0,.099])\n        self.assertAlmostEqual(self.t.report[\'set_down_height\'][\'measured_bottom_clearance_m\'],.04)\n    def test_set_down_height_rejects_sloped_table(self):\n        self.configure_table_geometry();self.t.data.geom_xmat[4]=rx(.2).ravel()\n        with self.assertRaises(ValueError):self.t.set_down_target(np.array([0,0,.09]))\n    def test_set_down_height_does_not_change_execution(self):\n        self.configure_table_geometry();before=self.t.data.qpos.copy()\n        self.t.set_down_target(np.array([0,0,.09]))\n        np.testing.assert_array_equal(before,self.t.data.qpos)\n\nclass PreservationTests(unittest.TestCase):\n    def test_runtime_methods_unchanged(self):\n        actual = runtime_method_nodes(WORKER_TEXT)\n        reference = runtime_method_nodes(BASELINE_METHODS_SOURCE)\n        self.assertEqual(set(reference), set(PRESERVED_METHOD_NAMES))\n        for name in PRESERVED_METHOD_NAMES:\n            with self.subTest(method=name):\n                self.assertIn(name, actual)\n                self.assertTrue(\n                    ast.dump(actual[name], include_attributes=False) ==\n                    ast.dump(reference[name], include_attributes=False),\n                    "Reviewed runtime method changed: " + name,\n                )\n    def test_thresholds_unchanged(self):\n        for name,expected in BASE_CONSTANTS.items():\n            with self.subTest(constant=name):self.assertEqual(getattr(mod,name),expected)\n    def test_carry_enabled_only_for_lift_and_set_down(self):\n        tree=ast.parse(WORKER.read_text(encoding="utf-8"))\n        calls=[n for n in ast.walk(tree) if isinstance(n,ast.Call)\n               and isinstance(n.func,ast.Attribute) and n.func.attr==\'plan\'\n               and any(k.arg==\'carry_cup\' and isinstance(k.value,ast.Constant) and k.value.value is True for k in n.keywords)]\n        self.assertEqual([ast.literal_eval(n.args[2]) for n in calls],[\'lift\',\'set_down\'])\n    def test_no_execution_position_assignments_in_patch(self):\n        tree=ast.parse(WORKER.read_text(encoding="utf-8"))\n        cls=next(n for n in tree.body if isinstance(n,ast.ClassDef) and n.name==\'CupTrial\')\n        patched={\'verify_scratch_separation\',\'make_carry_reference\',\'update_scratch_cup\',\n                 \'extra_carry_contact_issues\',\'set_down_target\',\'plan\'}\n        for f in cls.body:\n            if not isinstance(f,ast.FunctionDef) or f.name not in patched:continue\n            for node in ast.walk(f):\n                if isinstance(node,(ast.Assign,ast.AugAssign,ast.AnnAssign)):\n                    targets=node.targets if isinstance(node,ast.Assign) else [node.target]\n                    for target in targets:\n                        text=ast.unparse(target)\n                        self.assertNotIn(\'self.data.qpos\',text)\n                        self.assertFalse(text.startswith(\'d.qpos\'))\n    def test_fixed_jaw_box1_not_whitelisted(self):\n        loadsrc=ast.get_source_segment(WORKER.read_text(encoding="utf-8"),next(n for n in ast.walk(ast.parse(WORKER.read_text(encoding="utf-8"))) if isinstance(n,ast.FunctionDef) and n.name==\'load\'))\n        self.assertNotIn("\'right_fixed_jaw_box1\'",loadsrc)\n\n\nclass PortablePreservationTests(unittest.TestCase):\n    def test_reference_snapshot_is_independent_and_intact(self):\n        self.assertEqual(hashlib.sha256(BASELINE_METHODS_SOURCE.encode("utf-8")).hexdigest(),\n                         BASELINE_METHODS_SHA256)\n        self.assertEqual(set(runtime_method_nodes(BASELINE_METHODS_SOURCE)), set(PRESERVED_METHOD_NAMES))\n\n    def test_all_reviewed_methods_match_the_original_source(self):\n        self.assertEqual(changed_runtime_methods(WORKER_TEXT), [])\n\n    def test_blank_lines_and_comments_do_not_change_structure(self):\n        altered = "# harmless formatting comment\\n\\n" + BASELINE_METHODS_SOURCE\n        self.assertEqual(changed_runtime_methods(altered), [])\n\n    def test_crlf_line_endings_do_not_change_structure(self):\n        self.assertEqual(changed_runtime_methods(BASELINE_METHODS_SOURCE.replace("\\n", "\\r\\n")), [])\n\n    def test_changed_contact_force_guard_is_detected(self):\n        old = "normal>25"\n        self.assertIn(old, BASELINE_METHODS_SOURCE)\n        altered = BASELINE_METHODS_SOURCE.replace(old, "normal>2500", 1)\n        self.assertIn("contacts", changed_runtime_methods(altered))\n\n    def test_changed_collision_predicate_is_detected(self):\n        old = "if robot and dist<=0 and not intended:"\n        self.assertIn(old, BASELINE_METHODS_SOURCE)\n        altered = BASELINE_METHODS_SOURCE.replace(old, "if False:", 1)\n        self.assertIn("contacts", changed_runtime_methods(altered))\n\n    def test_missing_runtime_method_is_detected(self):\n        altered = BASELINE_METHODS_SOURCE.replace("def hold(", "def hold_removed(", 1)\n        self.assertIn("hold", changed_runtime_methods(altered))\n\n    def test_duplicate_method_is_rejected(self):\n        altered = BASELINE_METHODS_SOURCE + "\\n    def contacts(self):\\n        return []\\n"\n        with self.assertRaises(ValueError):\n            runtime_method_nodes(altered)\n\n    def test_source_integrity_accepts_lf_and_crlf(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            for endings in ("\\n", "\\r\\n"):\n                path = Path(tmp) / "worker.py"\n                path.write_bytes(WORKER_TEXT.replace("\\n", endings).encode("utf-8"))\n                self.assertEqual(checked_worker_text(path), WORKER_TEXT)\n\n    def test_source_integrity_rejects_a_real_code_change(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            path = Path(tmp) / "worker.py"\n            self.assertIn("COMMAND_SPEED = 0.35", WORKER_TEXT)\n            altered = WORKER_TEXT.replace("COMMAND_SPEED = 0.35", "COMMAND_SPEED = 3.5", 1)\n            path.write_bytes(altered.encode("utf-8"))\n            with self.assertRaises(ValueError):\n                checked_worker_text(path)\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])\n    result=unittest.TextTestRunner(verbosity=1).run(suite)\n    report={\'kind\':\'OFFLINE_SYNTHETIC_REGRESSION_NOT_MUJOCO_PHYSICS\',\'tests_run\':result.testsRun,\n            \'failures\':len(result.failures),\'errors\':len(result.errors),\'skipped\':len(result.skipped),\n            \'passed\':result.wasSuccessful(),\'worker_sha256\':hashlib.sha256(WORKER.read_bytes()).hexdigest(),\n            \'worker_text_sha256_lf\':hashlib.sha256(WORKER_TEXT.encode(\'utf-8\')).hexdigest(),\n            \'python_version\':sys.version.split()[0],\n            \'preservation_check\':\'independent original-06 reference parsed by same interpreter\',\n            \'reference_source_sha256\':BASELINE_METHODS_SHA256,\n            \'preserved_runtime_methods\':list(PRESERVED_METHOD_NAMES),\n            \'robot_physics_executed\':False}\n    (WORKER.parent/\'regression_summary.json\').write_text(json.dumps(report,indent=2), encoding=\'utf-8\')\n    print(json.dumps(report,indent=2))\n    raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'test_transfer.py': '"""Offline checks only: synthetic numbers and objects, not MuJoCo execution."""\nfrom pathlib import Path\nfrom types import SimpleNamespace as NS\nfrom unittest.mock import patch\nimport ast\nimport hashlib\nimport json\nimport sys\nimport tempfile\nimport unittest\n\nimport numpy as np\nimport base06b as base\nimport transfer_worker as t\n\n\ndef row(position=(.155, .1, .034), table=1., fixed=0., moving=0., two=False):\n    return dict(zip((\'cup_x\',\'cup_y\',\'cup_z\'),position), table_normal_n=table,\n                fixed_normal_n=fixed,moving_normal_n=moving,two_jaw_contact=two)\n\n\nclass TargetTests(unittest.TestCase):\n    def test_target_offset(self):\n        np.testing.assert_allclose(t.target_from_initial([.13,.1,.034],[.025,0,0]),[.155,.1,.034])\n    def test_initial_is_not_mutated(self):\n        a=np.array([.13,.1,.034]);copy=a.copy();t.target_from_initial(a,[.025,0,0]);np.testing.assert_array_equal(a,copy)\n    def test_no_larger_workspace(self):\n        with self.assertRaises(ValueError):t.target_from_initial([0,0,0],[.1,0,0])\n    def test_no_zero_transfer(self):\n        with self.assertRaises(ValueError):t.target_from_initial([0,0,0],[0,0,0])\n    def test_invalid_target_shape(self):\n        with self.assertRaises(ValueError):t.target_from_initial([0,0],[.025,0,0])\n    def test_nan_target(self):\n        with self.assertRaises(ValueError):t.target_from_initial([float(\'nan\'),0,0],[.025,0,0])\n\n\nclass MetricTests(unittest.TestCase):\n    def setUp(self):self.initial=np.array([.13,.1,.034]);self.target=self.initial+[.025,0,0]\n    def check(self,rows):return t.placement_metrics(rows,self.initial,self.target)\n    def test_new_destination_passes(self):self.assertTrue(self.check([row()])[\'placement_verified\'])\n    def test_old_position_cannot_pass(self):self.assertFalse(self.check([row(self.initial)])[\'placement_verified\'])\n    def test_no_table_support_fails(self):self.assertFalse(self.check([row(table=0)])[\'placement_verified\'])\n    def test_still_held_fails(self):self.assertFalse(self.check([row(fixed=1,moving=1)])[\'placement_verified\'])\n    def test_wrong_y_fails(self):self.assertFalse(self.check([row((.155,.112,.034))])[\'placement_verified\'])\n    def test_airborne_fails(self):self.assertFalse(self.check([row((.155,.1,.074))])[\'placement_verified\'])\n    def test_one_good_frame_is_not_enough(self):\n        rows=[row()]+[row(self.initial) for _ in range(5)];self.assertFalse(self.check(rows)[\'placement_verified\'])\n    def test_empty_rows_rejected(self):\n        with self.assertRaises(ValueError):self.check([])\n    def test_nan_measurement_rejected(self):\n        with self.assertRaises(ValueError):self.check([row((float(\'nan\'),.1,.034))])\n    def test_wrong_reference_rejected(self):\n        with self.assertRaises(ValueError):t.placement_metrics([row()],self.initial,self.initial)\n    def test_transfer_hold_good(self):\n        r=t.transfer_hold_metrics([row((.155,.1,.074),0,1,1,True)],self.initial,self.target);self.assertTrue(r[\'transfer_verified\'])\n    def test_transfer_low_cup_fails(self):\n        r=t.transfer_hold_metrics([row((.155,.1,.040),0,1,1,True)],self.initial,self.target);self.assertFalse(r[\'transfer_verified\'])\n    def test_transfer_no_jaw_contact_fails(self):\n        r=t.transfer_hold_metrics([row((.155,.1,.074),0,0,0,False)],self.initial,self.target);self.assertFalse(r[\'transfer_verified\'])\n    def test_transfer_table_supported_fails(self):\n        r=t.transfer_hold_metrics([row((.155,.1,.074),1,1,1,True)],self.initial,self.target);self.assertFalse(r[\'transfer_verified\'])\n\n\nclass GuardTests(unittest.TestCase):\n    def test_original_base_integrity(self):\n        raw=Path(base.__file__).read_bytes().replace(b\'\\r\\n\',b\'\\n\')\n        self.assertEqual(hashlib.sha256(raw).hexdigest(),\'027f2a91e66be817776dc36d4f2c434e1f2f1e16fdfee6ed01ad58f86134463e\')\n    def test_unmodified_inherited_methods(self):\n        for method in (\'load\',\'contacts\',\'move_jaw\',\'hold\',\'execute_plan\',\'stable_rows\',\'close_on_cup\',\'update_scratch_cup\',\'extra_carry_contact_issues\',\'set_down_target\',\'plan\'):\n            with self.subTest(method=method):self.assertIs(getattr(t.TransferTrial,method),getattr(base.CupTrial,method))\n    def trial(self):\n        obj=object.__new__(t.TransferTrial);obj.video_rows=[];obj.loss_steps=0;obj.dt=.005;obj.cup_initial=np.array([.13,.1,.034]);return obj\n    def test_tick_calls_original_guard(self):\n        obj=self.trial()\n        with patch.object(base.CupTrial,\'tick\',side_effect=RuntimeError(\'original guard\')):\n            with self.assertRaisesRegex(RuntimeError,\'original guard\'):obj.tick(np.zeros(12),\'transfer\',True)\n    def test_new_instruction_even_after_failure(self):\n        obj=self.trial()\n        def fail(*args,**kwargs):obj.video_rows.append({\'instruction\':\'old\'});raise RuntimeError(\'stopped\')\n        with patch.object(base.CupTrial,\'tick\',side_effect=fail):\n            with self.assertRaises(RuntimeError):obj.tick(np.zeros(12),\'transfer\',True)\n        self.assertEqual(obj.video_rows[0][\'instruction\'],t.INSTRUCTION)\n    def test_carry_requires_grasp(self):\n        obj=self.trial();obj.verify_scratch_separation=lambda:None;obj.report={};obj.gripper_close_target=None\n        with self.assertRaisesRegex(RuntimeError,\'grasp\'):obj.make_carry_reference(\'transfer\',True)\n    def test_carry_requires_lift(self):\n        obj=self.trial();obj.verify_scratch_separation=lambda:None;obj.report={\'grasp_verified\':True};obj.gripper_close_target=.49\n        with self.assertRaisesRegex(RuntimeError,\'lift\'):obj.make_carry_reference(\'transfer\',True)\n    def test_carry_forbidden_phase(self):\n        obj=self.trial();obj.verify_scratch_separation=lambda:None\n        with self.assertRaises(ValueError):obj.make_carry_reference(\'retreat\',True)\n    def test_no_execution_position_assignment(self):\n        source=Path(t.__file__).read_text();tree=ast.parse(source)\n        for node in ast.walk(tree):\n            if isinstance(node,(ast.Assign,ast.AugAssign,ast.AnnAssign)):\n                targets=node.targets if isinstance(node,ast.Assign) else [node.target]\n                for target in targets:\n                    txt=ast.unparse(target)\n                    self.assertFalse(any(k in txt for k in (\'self.data.qpos\',\'self.data.qvel\',\'xfrc_applied\',\'qfrc_applied\')))\n    def test_transfer_checks_loss(self):\n        obj=self.trial();obj.loss_steps=19\n        with patch.object(base.CupTrial,\'tick\',return_value=row((.155,.1,.074),0,0,0,False)):\n            with self.assertRaisesRegex(RuntimeError,\'contact lost\'):obj.tick(np.zeros(12),\'transfer\',True)\n    def test_transfer_keeps_elevation_check(self):\n        obj=self.trial()\n        with patch.object(base.CupTrial,\'tick\',return_value=row((.155,.1,.04),0,1,1,True)):\n            with self.assertRaisesRegex(RuntimeError,\'elevation\'):obj.tick(np.zeros(12),\'transfer\',True)\n\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])\n    r=unittest.TextTestRunner(verbosity=1).run(suite)\n    summary=dict(kind=\'OFFLINE_TRANSFER_HELPERS_NOT_ROBOT_PHYSICS\',tests_run=r.testsRun,\n                 failures=len(r.failures),errors=len(r.errors),skipped=len(r.skipped),passed=r.wasSuccessful())\n    Path(__file__).with_name(\'transfer_test_summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2))\n    raise SystemExit(0 if r.wasSuccessful() else 1)\n', 'object_target_worker.py': '"""07B: cup-origin targeting for the 25 mm transfer, with bounded feedback.\n\nEngineering controller: reads privileged MuJoCo state, not camera perception/VLA.\nBase 06B and original 07 remain separate, unmodified modules. Physical motion uses\ninherited actuator execution. Only scratch state predicts carried-cup geometry.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport sys\nimport traceback\nimport numpy as np\nimport base06b as base\nimport transfer_worker as original\n\nMAX_CORRECTIONS = 2\nMAX_CORRECTION_STEP_M = 0.010\nMAX_CUMULATIVE_CORRECTION_M = 0.020\nREVISION = \'07b-cup-origin-target-v1\'\n\n\ndef checked_vector(value):\n    a = np.array(value, dtype=float, copy=True)\n    if a.shape != (3,) or not np.isfinite(a).all():\n        raise ValueError(\'Expected a finite 3D point\')\n    return a\n\n\ndef tool_to_cup_target(tool_goal, tool_now, cup_now, destination):\n    """Convert a requested lowering displacement to a cup-origin objective.\n\nHorizontal destination is not updated to wherever the cup happens to be.\n"""\n    goal, tool, cup, dest = map(checked_vector, (tool_goal, tool_now, cup_now, destination))\n    target = cup + (goal - tool)\n    target[:2] = dest[:2]\n    return target\n\n\ndef bounded_correction_target(cup_now, destination, height, used_distance):\n    cup, dest = map(checked_vector, (cup_now, destination))\n    if (not math.isfinite(float(height)) or not math.isfinite(float(used_distance))\n            or not 0.0 <= used_distance <= MAX_CUMULATIVE_CORRECTION_M + 1e-12):\n        raise ValueError(\'Invalid correction height/distance budget\')\n    error = dest[:2] - cup[:2]\n    norm = float(np.linalg.norm(error))\n    available = max(0.0, MAX_CUMULATIVE_CORRECTION_M - used_distance)\n    step = min(norm, MAX_CORRECTION_STEP_M, available)\n    if step <= 1e-9:\n        raise RuntimeError(\'No remaining useful correction distance\')\n    result = cup.copy()\n    result[:2] += error * (step / norm)\n    result[2] = height\n    return result, step\n\n\ndef maintaining_carry(metrics):\n    """A position-only retry is forbidden if grasp/elevation/support gates fail."""\n    values = [float(metrics[k]) for k in\n              (\'minimum_cup_rise_m\', \'two_jaw_contact_fraction\', \'max_table_normal_n\')]\n    if not all(math.isfinite(v) for v in values):\n        raise ValueError(\'Non-finite carry measurements\')\n    return (values[0] >= .025 and .80 <= values[1] <= 1.0\n            and 0 <= values[2] < base.CONTACT_MIN_N)\n\n\nclass ObjectTargetTransferTrial(original.TransferTrial):\n    def __init__(self, config):\n        super().__init__(config)\n        self.object_plans = []\n        self.corrections_used = 0\n        self.correction_distance = 0.0\n        self.transfer_height = None\n        self.transfer_axis = None\n        self.report.update(\n            task_revision=REVISION,\n            carry_position_objective=\'predicted cup body origin, not fixed gripper point\',\n            correction_source=\'current simulator body pose; NOT camera perception\',\n            maximum_position_corrections=MAX_CORRECTIONS,\n            maximum_correction_step_m=MAX_CORRECTION_STEP_M,\n            maximum_cumulative_correction_m=MAX_CUMULATIVE_CORRECTION_M,\n            correction_history=[], object_target_plans=self.object_plans,\n            acceptance_thresholds_changed=False,\n        )\n        self.report[\'notes\'].append(\n            \'07B directly targets the predicted cup origin for transfer/set-down. \'\n            \'At most two bounded position-only corrections are permitted while carry gates hold. \'\n            \'These corrections are engineering feedback, not the camera-based TableGuard feature.\')\n\n    def plan(self, target, axis, label, allow_cup=False, carry_cup=False):\n        # Retain the original checked approach, lift and retreat behaviors.\n        if label not in (\'transfer\', \'set_down\'):\n            return super().plan(target, axis, label, allow_cup, carry_cup)\n        if not allow_cup or not carry_cup:\n            raise ValueError(\'Object-target plans require verified carrying mode\')\n        cup = self.data.xpos[self.cup].copy()\n        if label == \'transfer\':\n            cup_goal = cup.copy()\n            cup_goal[:2] = self.target[:2]\n            self.transfer_height = float(cup_goal[2])\n            self.transfer_axis = base.normalize(axis).copy()\n        else:\n            # The base already calculated the measured bottom-clearance drop.\n            cup_goal = tool_to_cup_target(target, self.gripper_point(self.data), cup, self.target)\n        plan = self.plan_cup(cup_goal, axis, label)\n        plan[\'original_gripper_target_m\'] = checked_vector(target).tolist()\n        return plan\n\n    def plan_cup(self, target, axis, label):\n        """IK at the carried cup\'s predicted origin, using the gripper-body Jacobian.\n\nDo NOT use mj_jacBody(cup): the free cup\'s actual Jacobian is not a robot-arm\nkinematic chain. The point is treated as attached to the gripper only for prediction.\n"""\n        if label not in (\'transfer\', \'set_down\'):\n            raise ValueError(\'Object targeting is restricted to transfer/set-down\')\n        m, d, mj = self.model, self.data, self.mj\n        self.verify_scratch_separation()\n        target = checked_vector(target)\n        axis = base.normalize(axis)\n        if np.linalg.norm(target[:2] - self.cup_initial[:2]) > .035 + 1e-10:\n            raise ValueError(\'Cup target exceeds the retained 35 mm lateral envelope\')\n        if label == \'transfer\' and target[2] - self.cup_initial[2] < .025:\n            raise ValueError(\'Transfer target would violate the retained lift requirement\')\n        reference = self.make_carry_reference(label, True)\n        scratch = self.scratch\n        scratch.qpos[:] = d.qpos  # Scratch only. Never modify execution qpos.\n        seed = np.array([d.ctrl[r[\'cid\']] for r in self.arm], dtype=float)\n        jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))\n        index = len(self.object_plans)\n        evidence = dict(\n            label=label, ordinal=index, target_cup_origin_m=target.tolist(),\n            tool_axis=axis.tolist(), source=\'simulator geometry; not vision\',\n            success=False, waypoints=[], carry_reference=reference[\'record\'],\n        )\n        self.object_plans.append(evidence)\n        self.event(\'object_target_plan_start\', phase=label, plan_index=index,\n                   target_m=target.tolist())\n\n        def evaluate(q):\n            self.budget()\n            scratch.qpos[self.arm_indices] = q\n            self.update_scratch_cup(reference)\n            mj.mj_forward(m, scratch)\n            p = scratch.xpos[self.cup].copy()\n            tool_axis = scratch.xmat[self.gripper_body].reshape(3, 3)[:, 2].copy()\n            mj.mj_jac(m, scratch, jp, jr, p, self.gripper_body)\n            return (p, tool_axis, jp[:, self.arm_dofs].copy(),\n                    (-base.skew(tool_axis) @ jr[:, self.arm_dofs]).copy())\n\n        p0, a0, _, _ = evaluate(seed)\n        path = [seed.copy()]\n        for k in range(1, 41):\n            fraction = k / 40\n            goal = p0 + fraction * (target - p0)\n            goal_axis = base.normalize((1 - fraction) * a0 + fraction * axis)\n            q, diag = base.solve_pose(evaluate, goal, goal_axis, seed, self.low, self.high, self.budget)\n            evidence[\'waypoints\'].append(dict(waypoint=k, **diag))\n            if not diag[\'converged\']:\n                raise RuntimeError(f\'{label}: cup-origin IK waypoint {k} failed: {diag}\')\n            count = max(1, math.ceil(float(np.max(np.abs(q - seed))) / .02))\n            for alpha in np.linspace(0, 1, count + 1)[1:]:\n                evaluate(seed + alpha * (q - seed))\n                _, _, issues = self.contacts(scratch, True, planning=True)\n                issues += self.extra_carry_contact_issues(reference)\n                if issues:\n                    raise RuntimeError(f\'{label}: cup-origin sampled path rejected: {issues[:2]}\')\n            path.append(q.copy())\n            seed = q\n            if k % 10 == 0:\n                self.event(\'object_target_waypoint\', phase=label, waypoint=k,\n                           predicted_error_mm=round(1000 * diag[\'position_error_m\'], 4))\n        plan = base.parameterize(path, self.dt)\n        end, end_axis, _, _ = evaluate(seed)\n        residual = float(np.linalg.norm(end - target))\n        if residual > base.POSITION_TOL + 1e-10:\n            raise RuntimeError(\'Final predicted cup position does not meet original IK tolerance\')\n        evidence.update(success=True, predicted_endpoint_m=end.tolist(),\n                        predicted_endpoint_error_m=residual, duration_s=plan[\'duration_s\'])\n        plan.update(target_cup_origin_m=target.tolist(), target_axis=axis.tolist(),\n                    diagnostics=evidence[\'waypoints\'], carried_object_prediction=reference[\'record\'])\n        base.write_json(self.out / f\'object_plan_{index:02d}_{label}.json\', evidence)\n        self.event(\'object_target_plan_complete\', phase=label, plan_index=index,\n                   predicted_error_mm=round(residual * 1000, 4), duration_s=plan[\'duration_s\'])\n        return plan\n\n    def hold(self, seconds, label, allow_cup=False):\n        super().hold(seconds, label, allow_cup)\n        if label != \'transfer_hold\':\n            return\n        if not allow_cup or self.transfer_height is None or self.transfer_axis is None:\n            raise RuntimeError(\'Transfer feedback requires the current verified object-target plan\')\n        while True:\n            metrics = original.transfer_hold_metrics(\n                self.stable_rows(\'transfer_hold\'), self.cup_initial, self.target)\n            self.report[\'transfer_measurements\'] = metrics\n            self.report[\'transfer_position_diagnostics\'] = dict(\n                desired_cup_position_m=self.target.tolist(),\n                actual_cup_position_m=self.data.xpos[self.cup].copy().tolist(),\n                horizontal_error_vector_m=(self.target[:2]-self.data.xpos[self.cup, :2]).tolist())\n            self.report[\'correction_history\'].append(dict(\n                corrections_completed=self.corrections_used,\n                simulation_time_s=float(self.data.time), **metrics))\n            error = float(metrics[\'maximum_horizontal_target_error_m\'])\n            self.event(\'object_target_measured\', corrections_completed=self.corrections_used,\n                       horizontal_error_mm=round(error * 1000, 4),\n                       minimum_rise_mm=round(metrics[\'minimum_cup_rise_m\'] * 1000, 4))\n            if metrics[\'transfer_verified\']:\n                return\n            if not maintaining_carry(metrics):\n                raise RuntimeError(\'Carry elevation/contact/support failed; no position correction permitted\')\n            if not math.isfinite(error):\n                raise RuntimeError(\'Non-finite transfer-position error\')\n            if self.corrections_used >= MAX_CORRECTIONS:\n                raise RuntimeError(f\'Cup horizontal error {1000*error:.3f} mm exceeds 8 mm after \'\n                                   f\'{MAX_CORRECTIONS} bounded corrections\')\n            cup_goal, distance = bounded_correction_target(\n                self.data.xpos[self.cup], self.target, self.transfer_height, self.correction_distance)\n            self.event(\'bounded_cup_correction\', number=self.corrections_used + 1,\n                       planned_horizontal_step_mm=round(1000 * distance, 4))\n            plan = self.plan_cup(cup_goal, self.transfer_axis, \'transfer\')\n            self.corrections_used += 1\n            self.correction_distance += distance\n            self.report[\'corrections_used\'] = self.corrections_used\n            self.report[\'cumulative_planned_correction_m\'] = self.correction_distance\n            # Same labels retain the original transfer-specific per-step guards.\n            super().execute_plan(plan, \'transfer\', True)\n            super().hold(.5, \'transfer_hold\', True)\n\n    def finish(self):\n        self.report[\'corrections_used\'] = self.corrections_used\n        self.report[\'cumulative_planned_correction_m\'] = self.correction_distance\n        super().finish()\n\n\ndef main(config):\n    original.target_from_initial([0, 0, 0], config[\'offset_m\'])\n    if not isinstance(config.get(\'record_data\'), bool):\n        raise ValueError(\'record_data must be boolean\')\n    trial = ObjectTargetTransferTrial(config)\n    try:\n        trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.event(\'stopped\', reason=trial.report[\'stops\'][-1])\n        (trial.out / \'failure_traceback.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n        if trial.renderer is not None:\n            trial.snapshot(\'stopped\')\n    finally:\n        trial.finish()\n    print(\'TRANSFER 07B RESULT \' + json.dumps({k: trial.report.get(k) for k in (\n        \'completed\', \'transfer_test_passed\', \'grasp_verified\', \'lift_verified\',\n        \'transfer_verified\', \'placement_verified\', \'release_verified\', \'corrections_used\',\n        \'transfer_measurements\', \'placement_measurements\', \'stops\', \'report_file\')},\n        allow_nan=False), flush=True)\n    return 0 if trial.report.get(\'transfer_test_passed\') else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_object_target.py': '"""Offline geometry, bounded-feedback and preservation tests; NOT robot physics."""\nfrom pathlib import Path\nfrom types import SimpleNamespace\nimport ast\nimport contextlib\nimport hashlib\nimport io\nimport json\nimport math\nimport tempfile\nimport unittest\nfrom unittest.mock import patch, Mock\nimport numpy as np\nimport base06b as base\nimport transfer_worker as old\nimport object_target_worker as fix\n\n\ndef rotation(yaw=0., pitch=0.):\n    cy,sy,cp,sp=np.cos(yaw),np.sin(yaw),np.cos(pitch),np.sin(pitch)\n    return np.array([[cy,-sy,0],[sy,cy,0],[0,0,1]]) @ np.array([[cp,0,sp],[0,1,0],[-sp,0,cp]])\n\n\nclass TargetTests(unittest.TestCase):\n    def test_user_report_rejected_by_unchanged_metric(self):\n        rows=[dict(cup_x=.025-.012082477230204298,cup_y=0,cup_z=.03742273199512661,\n                   table_normal_n=0,two_jaw_contact=True)]\n        value=old.transfer_hold_metrics(rows,np.zeros(3),np.array([.025,0,0]))\n        self.assertFalse(value[\'transfer_verified\'])\n        self.assertTrue(fix.maintaining_carry(value))\n    def test_original_position_cannot_pass_final_destination(self):\n        rows=[dict(cup_x=0,cup_y=0,cup_z=0,table_normal_n=1,fixed_normal_n=0,moving_normal_n=0)]\n        self.assertFalse(old.placement_metrics(rows,np.zeros(3),np.array([.025,0,0]))[\'placement_verified\'])\n    def test_acceptance_tolerance_remains_eight_mm(self):\n        self.assertEqual(old.PLACEMENT_TOL_M,.008)\n    def test_target_conversion_keeps_measured_lowering(self):\n        p=fix.tool_to_cup_target([.13,.1,.07],[.13,.1,.11],[.14,.09,.08],[.155,.1,.04])\n        np.testing.assert_allclose(p,[.155,.1,.04])\n    def test_target_does_not_follow_wrong_horizontal_measurement(self):\n        p=fix.tool_to_cup_target([.1,0,.06],[.1,0,.1],[.012,0,.075],[.025,0,.035])\n        self.assertEqual(p[0],.025)\n    def test_target_inputs_not_modified(self):\n        x=np.array([.1,.2,.3]);a=x.copy()\n        fix.tool_to_cup_target(x,x,x,x)\n        np.testing.assert_array_equal(x,a)\n    def test_nonfinite_target_rejected(self):\n        with self.assertRaises(ValueError):fix.checked_vector([0,float(\'nan\'),0])\n    def test_bad_target_shape_rejected(self):\n        with self.assertRaises(ValueError):fix.checked_vector([0,0])\n    def test_point_offset_rotates(self):\n        cup_local=np.array([.02,0,0]);oldR=rotation();newR=rotation(math.pi/4)\n        delta=np.array([.025,0,0]);initial=oldR@cup_local\n        predicted_with_old_tool_offset=delta+newR@cup_local\n        desired=initial+delta\n        self.assertGreater(np.linalg.norm(predicted_with_old_tool_offset-desired),.008)\n        correct_tool=desired-newR@cup_local\n        np.testing.assert_allclose(correct_tool+newR@cup_local,desired)\n    def test_corrected_cup_point_jacobian_has_rotation_term(self):\n        r=np.array([.03,-.012,.018]);R=rotation(.6,.4);world=R@r\n        axis=np.array([0,0,1.]);analytic=np.cross(axis,world)\n        eps=1e-6;numeric=(rotation(.6+eps,.4)@r-rotation(.6-eps,.4)@r)/(2*eps)\n        np.testing.assert_allclose(analytic,numeric,atol=1e-9)\n    def test_step_bounded_ten_mm(self):\n        t,d=fix.bounded_correction_target([0,0,.04],[.025,0,0],.04,0)\n        self.assertAlmostEqual(d,.01);self.assertAlmostEqual(t[0],.01)\n    def test_step_never_overshoots_destination(self):\n        t,d=fix.bounded_correction_target([.024,0,.04],[.025,0,0],.04,0)\n        self.assertAlmostEqual(d,.001);self.assertAlmostEqual(t[0],.025)\n    def test_remaining_budget_applied(self):\n        t,d=fix.bounded_correction_target([0,0,.04],[.025,0,0],.04,.017)\n        self.assertAlmostEqual(d,.003)\n    def test_exhausted_budget_rejected(self):\n        with self.assertRaises(RuntimeError):fix.bounded_correction_target([0,0,.04],[.025,0,0],.04,.02)\n    def test_negative_budget_rejected(self):\n        with self.assertRaises(ValueError):fix.bounded_correction_target([0,0,.04],[.025,0,0],.04,-.01)\n    def test_invalid_height_rejected(self):\n        with self.assertRaises(ValueError):fix.bounded_correction_target([0,0,.04],[.025,0,0],float(\'nan\'),0)\n    def test_carry_loss_forbids_position_only_retry(self):\n        x=dict(minimum_cup_rise_m=.04,two_jaw_contact_fraction=.3,max_table_normal_n=0)\n        self.assertFalse(fix.maintaining_carry(x))\n    def test_table_support_forbids_position_only_retry(self):\n        x=dict(minimum_cup_rise_m=.04,two_jaw_contact_fraction=1,max_table_normal_n=.1)\n        self.assertFalse(fix.maintaining_carry(x))\n    def test_insufficient_elevation_forbids_retry(self):\n        x=dict(minimum_cup_rise_m=.020,two_jaw_contact_fraction=1,max_table_normal_n=0)\n        self.assertFalse(fix.maintaining_carry(x))\n\n\nclass SyntheticMJ:\n    """5-coordinate kinematic surrogate only; does not simulate any physics."""\n    def __init__(self,trial):self.trial=trial;self.jac_body_ids=[]\n    def mj_forward(self,model,data):self.trial.update_transforms(data)\n    def mj_jac(self,model,data,jp,jr,point,body):\n        self.jac_body_ids.append(body)\n        q=data.qpos;R=rotation(q[3],q[4]);local=R.T@(point-q[:3]);eps=1e-6\n        jp[:]=0;jr[:]=0;jp[:,:3]=np.eye(3)\n        jp[:,3]=(rotation(q[3]+eps,q[4])@local-rotation(q[3]-eps,q[4])@local)/(2*eps)\n        jp[:,4]=(rotation(q[3],q[4]+eps)@local-rotation(q[3],q[4]-eps)@local)/(2*eps)\n        jr[:,3]=[0,0,1];jr[:,4]=[-np.sin(q[3]),np.cos(q[3]),0]\n\n\nclass KinematicProbe(fix.ObjectTargetTransferTrial):\n    def __init__(self,out):\n        self.out=Path(out);self.model=SimpleNamespace(nv=5)\n        self.gripper_body=0;self.cup=1;self.offset=np.array([.025,.014,-.025])\n        seed=np.array([.1,.1,.14,.25,.4])\n        self.data=SimpleNamespace(qpos=seed.copy(),ctrl=seed.copy(),xpos=np.zeros((2,3)),xmat=np.zeros((2,9)),time=0.)\n        self.scratch=SimpleNamespace(qpos=seed.copy(),xpos=np.zeros((2,3)),xmat=np.zeros((2,9)))\n        self.update_transforms(self.data);self.update_transforms(self.scratch)\n        self.cup_initial=self.data.xpos[1].copy()-[0,0,.037]\n        self.target=self.cup_initial+[.025,0,0]\n        self.mj=SyntheticMJ(self);self.arm=[{\'cid\':i} for i in range(5)]\n        self.arm_indices=list(range(5));self.arm_dofs=list(range(5));self.low=np.array([-1]*5);self.high=np.array([1]*5)\n        self.dt=.005;self.object_plans=[];self.report={\'carry_plans\':{}};self.block=False;self.penetration=False\n        self.events=[];self.contact_calls=0\n    def update_transforms(self,d):\n        R=rotation(d.qpos[3],d.qpos[4]);d.xpos[0]=d.qpos[:3];d.xmat[0]=R.reshape(9)\n        d.xpos[1]=d.qpos[:3]+R@self.offset;d.xmat[1]=R.reshape(9)\n    def make_carry_reference(self,label,allow_cup):\n        self.verify_scratch_separation()\n        return {\'record\':{\'label\':label,\'source\':\'synthetic\'}}\n    def update_scratch_cup(self,ref):self.update_transforms(self.scratch)\n    def budget(self):pass\n    def event(self,name,**kw):self.events.append((name,kw))\n    def contacts(self,*a,**kw):\n        self.contact_calls+=1;return {},0.,([\'synthetic collision\'] if self.block else [])\n    def extra_carry_contact_issues(self,*a):return [\'penetration\'] if self.penetration else []\n\n\nclass PlannerTests(unittest.TestCase):\n    def setUp(self):self.temp=tempfile.TemporaryDirectory();self.t=KinematicProbe(self.temp.name)\n    def tearDown(self):self.temp.cleanup()\n    def target(self):\n        target=self.t.data.xpos[self.t.cup].copy();target[:2]=self.t.target[:2];return target\n    def axis(self):return self.t.data.xmat[0].reshape(3,3)[:,2]\n    def test_predicted_cup_origin_reaches_target(self):\n        p=self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n        self.assertLessEqual(self.t.object_plans[-1][\'predicted_endpoint_error_m\'],base.POSITION_TOL)\n        self.assertGreater(len(p[\'path\']),1)\n    def test_actual_execution_state_unchanged(self):\n        oldq=self.t.data.qpos.copy();oldpos=self.t.data.xpos.copy();oldctrl=self.t.data.ctrl.copy()\n        self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n        np.testing.assert_array_equal(oldq,self.t.data.qpos);np.testing.assert_array_equal(oldpos,self.t.data.xpos)\n        np.testing.assert_array_equal(oldctrl,self.t.data.ctrl)\n    def test_jacobian_uses_gripper_not_free_cup(self):\n        self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n        self.assertEqual(set(self.t.mj.jac_body_ids),{self.t.gripper_body})\n    def test_sampled_collision_still_rejects(self):\n        self.t.block=True\n        with self.assertRaisesRegex(RuntimeError,\'sampled path rejected\'):\n            self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n    def test_penetration_still_rejects(self):\n        self.t.penetration=True\n        with self.assertRaisesRegex(RuntimeError,\'penetration\'):\n            self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n    def test_joint_intermediate_checks_run(self):\n        self.t.plan_cup(self.target(),self.axis(),\'transfer\');self.assertGreaterEqual(self.t.contact_calls,40)\n    def test_plan_saved_as_new_diagnostic(self):\n        self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n        self.assertTrue((Path(self.temp.name)/\'object_plan_00_transfer.json\').is_file())\n    def test_lateral_envelope_not_enlarged(self):\n        p=self.target();p[0]+=1\n        with self.assertRaisesRegex(ValueError,\'35 mm\'):self.t.plan_cup(p,self.axis(),\'transfer\')\n    def test_no_lowered_transfer_target(self):\n        p=self.target();p[2]=self.t.cup_initial[2]+.01\n        with self.assertRaisesRegex(ValueError,\'lift requirement\'):self.t.plan_cup(p,self.axis(),\'transfer\')\n    def test_no_unverified_label(self):\n        with self.assertRaises(ValueError):self.t.plan_cup(self.target(),self.axis(),\'invented\')\n    def test_no_shared_execution_scratch(self):\n        self.t.scratch=self.t.data\n        with self.assertRaisesRegex(RuntimeError,\'separate\'):self.t.plan_cup(self.target(),self.axis(),\'transfer\')\n    def test_target_transfer_mapping_is_cup_not_tool(self):\n        # Exercise the dispatch without calling the surrogate pose solver.\n        with patch.object(self.t,\'plan_cup\',return_value={}) as plan:\n            self.t.plan([.8,.7,.6],self.axis(),\'transfer\',True,True)\n            np.testing.assert_allclose(plan.call_args.args[0][:2],self.t.target[:2])\n            self.assertAlmostEqual(plan.call_args.args[0][2],self.t.data.xpos[1,2])\n    def test_uncarried_transfer_rejected(self):\n        with self.assertRaises(ValueError):self.t.plan(self.target(),self.axis(),\'transfer\',False,False)\n\n\ndef hold_probe(position=.012, tracking_fraction=1., support=0.,two_jaw=True):\n    obj=object.__new__(fix.ObjectTargetTransferTrial)\n    obj.transfer_height=.037;obj.transfer_axis=np.array([0,0,1.]);obj.cup_initial=np.zeros(3)\n    obj.target=np.array([.025,0,0]);obj.cup=0\n    obj.data=SimpleNamespace(xpos=np.array([[position,0,.037]]),time=1.)\n    obj.corrections_used=0;obj.correction_distance=0.;obj.report={\'correction_history\':[]}\n    obj.event=lambda *a,**k:None\n    obj.stable_rows=lambda *a,**k:[dict(cup_x=obj.data.xpos[0,0],cup_y=0,cup_z=.037,\n                                      table_normal_n=support,two_jaw_contact=two_jaw)]\n    obj.plan_cup=Mock(side_effect=lambda target,*a,**k:{\'target\':target.copy()})\n    def execute(_self,plan,*a):\n        obj.data.xpos[0]+=(plan[\'target\']-obj.data.xpos[0])*tracking_fraction\n    return obj,execute\n\n\nclass FeedbackTests(unittest.TestCase):\n    def test_one_bounded_feedback_step_can_fix_residual(self):\n        obj,execute=hold_probe()\n        with patch.object(base.CupTrial,\'hold\'),patch.object(base.CupTrial,\'execute_plan\',execute):\n            obj.hold(.5,\'transfer_hold\',True)\n        self.assertEqual(obj.corrections_used,1)\n        self.assertTrue(obj.report[\'transfer_measurements\'][\'transfer_verified\'])\n    def test_acceptable_result_not_rerun(self):\n        obj,execute=hold_probe(.024)\n        with patch.object(base.CupTrial,\'hold\'),patch.object(base.CupTrial,\'execute_plan\',execute):obj.hold(.5,\'transfer_hold\',True)\n        obj.plan_cup.assert_not_called()\n    def test_no_contact_loss_retry(self):\n        obj,execute=hold_probe(two_jaw=False)\n        with patch.object(base.CupTrial,\'hold\'),self.assertRaisesRegex(RuntimeError,\'no position correction\'):\n            obj.hold(.5,\'transfer_hold\',True)\n        obj.plan_cup.assert_not_called()\n    def test_table_support_prevents_correction(self):\n        obj,execute=hold_probe(support=.1)\n        with patch.object(base.CupTrial,\'hold\'),self.assertRaises(RuntimeError):obj.hold(.5,\'transfer_hold\',True)\n        obj.plan_cup.assert_not_called()\n    def test_two_correction_limit_enforced(self):\n        obj,execute=hold_probe(tracking_fraction=0.)\n        with patch.object(base.CupTrial,\'hold\'),patch.object(base.CupTrial,\'execute_plan\',execute),self.assertRaisesRegex(RuntimeError,\'after 2\'):\n            obj.hold(.5,\'transfer_hold\',True)\n        self.assertEqual(obj.corrections_used,2)\n        self.assertLessEqual(obj.correction_distance,.020)\n    def test_unrelated_hold_delegates_only(self):\n        obj,_=hold_probe()\n        with patch.object(base.CupTrial,\'hold\') as h:obj.hold(.75,\'initial_settle\',False)\n        h.assert_called_once_with(.75,\'initial_settle\',False);obj.plan_cup.assert_not_called()\n\n\nclass PreservationTests(unittest.TestCase):\n    def test_tick_inherited_without_override(self):self.assertIs(fix.ObjectTargetTransferTrial.tick,old.TransferTrial.tick)\n    def test_execute_inherited_without_override(self):self.assertIs(fix.ObjectTargetTransferTrial.execute,old.TransferTrial.execute)\n    def test_contact_rules_inherited(self):self.assertIs(fix.ObjectTargetTransferTrial.contacts,base.CupTrial.contacts)\n    def test_execution_command_function_unchanged(self):self.assertIs(fix.ObjectTargetTransferTrial.execute_plan,base.CupTrial.execute_plan)\n    def test_original_sources_match_verified_hashes(self):\n        for path,digest in [(\'base06b.py\',\'027f2a91e66be817776dc36d4f2c434e1f2f1e16fdfee6ed01ad58f86134463e\'),\n                            (\'transfer_worker.py\',\'92e0bcebfd45f7501940ad2f16ec97491349c645c04914d86e9019c1d539f849\')]:\n            self.assertEqual(hashlib.sha256((Path(__file__).parent/path).read_bytes().replace(b\'\\r\\n\',b\'\\n\')).hexdigest(),digest)\n    def test_no_state_force_or_weld_override(self):\n        tree=ast.parse(Path(fix.__file__).read_text(encoding=\'utf-8\'))\n        for node in ast.walk(tree):\n            if isinstance(node,(ast.Assign,ast.AugAssign,ast.AnnAssign)):\n                for t in getattr(node,\'targets\',[getattr(node,\'target\',None)]):\n                    if t is None:continue\n                    text=ast.unparse(t)\n                    self.assertFalse(any(v in text for v in (\'data.qpos\',\'data.xpos\',\'qfrc_applied\',\'xfrc_applied\',\'eq_active\')))\n    def test_original_thresholds_remain(self):\n        self.assertEqual(base.GRIPPER_TORQUE_CAP,.25);self.assertEqual(base.PENETRATION_LIMIT,.003)\n        self.assertEqual(base.COMMAND_SPEED,.35);self.assertEqual(base.POSITION_TOL,.0005)\n\n\nif __name__==\'__main__\':\n    # unittest\'s module loader must use the current __main__ module.\n    import sys\n    suite=unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])\n    result=unittest.TextTestRunner(verbosity=1).run(suite)\n    report={\'kind\':\'OFFLINE_SYNTHETIC_OBJECT_TARGET_TESTS_NOT_MUJOCO_PHYSICS\',\n            \'tests_run\':result.testsRun,\'failures\':len(result.failures),\'errors\':len(result.errors),\n            \'skipped\':len(result.skipped),\'passed\':result.wasSuccessful(),\'python\':sys.version.split()[0]}\n    (Path(__file__).parent/\'object_target_test_summary.json\').write_text(json.dumps(report,indent=2),encoding=\'utf-8\')\n    print(json.dumps(report,indent=2));raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'cup_orientation_worker.py': '"""07C: predict and regulate the carried cup\'s up-axis, not the gripper z-axis.\n\nOriginal base06b, transfer_worker and object_target_worker are kept unchanged.\nThis is privileged simulator-state engineering control, not a learned policy.\nOnly scratch qpos is assigned. Execution retains the 30-degree tilt stop and\nall inherited guards. A 25-degree sampled planning ceiling adds 5 degrees of\nmargin; it is not a safety proof. Contact slip remains possible in execution.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport sys\nimport traceback\nimport numpy as np\nimport base06b as base\nimport transfer_worker as original\nimport object_target_worker as previous\n\nREVISION = \'07c-cup-up-axis-v1\'\nPLANNING_TILT_LIMIT_DEG = 25.0\nRUNTIME_TILT_LIMIT_DEG = 30.0  # Informational: enforced in the unchanged base tick.\n\n\ndef angle_deg(a, b):\n    a, b = base.normalize(a), base.normalize(b)\n    return math.degrees(math.acos(float(np.clip(np.dot(a, b), -1.0, 1.0))))\n\n\ndef tilt_deg(axis):\n    return angle_deg(axis, np.array([0., 0., 1.]))\n\n\ndef carried_point_and_axis(parent_p, parent_R, local_p, local_R):\n    """Pure transform helper: return CUP origin and CUP up-axis in world space."""\n    p, R = base.predicted_pose(parent_p, parent_R, local_p, local_R)\n    return p, R[:, 2].copy()\n\n\ndef require_planning_tilt(axis):\n    tilt = tilt_deg(axis)\n    if tilt > PLANNING_TILT_LIMIT_DEG + 1e-9:\n        raise RuntimeError(\n            f\'Predicted cup tilt {tilt:.3f} deg exceeds the \'\n            f\'{PLANNING_TILT_LIMIT_DEG:.1f} deg planning ceiling; no candidate motion executed\')\n    return tilt\n\n\ndef up_axis_jacobian(axis, angular_jacobian):\n    a = base.normalize(axis)\n    J = np.asarray(angular_jacobian, dtype=float)\n    if J.ndim != 2 or J.shape[0] != 3 or not np.isfinite(J).all():\n        raise ValueError(\'Expected a finite 3-by-N angular Jacobian\')\n    return -base.skew(a) @ J\n\n\nclass CupOrientationTrial(previous.ObjectTargetTransferTrial):\n    """Retain target/feedback semantics, change only carrying orientation objective."""\n    def __init__(self, config):\n        super().__init__(config)\n        self.cup_up_reference = None\n        self.initial_carry_relative_p = None\n        self.initial_carry_relative_up = None\n        self.report.update(\n            task_revision=REVISION,\n            carry_orientation_objective=\'cup up-axis measured after lift; cup yaw remains free\',\n            planning_tilt_limit_deg=PLANNING_TILT_LIMIT_DEG,\n            runtime_tilt_limit_deg=RUNTIME_TILT_LIMIT_DEG,\n            orientation_target_source=\'simulator cup rotation; NOT camera perception\',\n            actual_transfer_physics_verified_in_preparation=False,\n        )\n        self.report[\'notes\'].append(\n            \'07C constrains the predicted CUP up-axis rather than the gripper z-axis. \'\n            \'The measured post-lift up-axis is kept fixed for transfer, corrections and set-down. \'\n            \'Sampled planning is rejected above 25 degrees; the unchanged execution guard stops above 30. \'\n            \'This is not full six-dimensional pose control or a guarantee against contact slip.\')\n\n    def plan_cup(self, target, axis, label):\n        # The inherited signature supplies a gripper-axis argument. It is not used\n        # as the orientation target for the CUP; only its validity is checked.\n        if label not in (\'transfer\', \'set_down\'):\n            raise ValueError(\'Cup-axis plans are limited to transfer and set_down\')\n        requested_tool_axis = base.normalize(axis)\n        m, d, mj = self.model, self.data, self.mj\n        self.verify_scratch_separation()\n        goal = previous.checked_vector(target)\n        if np.linalg.norm(goal[:2] - self.cup_initial[:2]) > .035 + 1e-10:\n            raise ValueError(\'Cup target exceeds retained 35 mm lateral bound\')\n        if label == \'transfer\' and goal[2] - self.cup_initial[2] < .025:\n            raise ValueError(\'Cup transfer target fails the retained elevation requirement\')\n        reference = self.make_carry_reference(label, True)\n        measured_up = base.normalize(d.xmat[self.cup].reshape(3, 3)[:, 2])\n        entry_tilt = require_planning_tilt(measured_up)\n        if self.cup_up_reference is None:\n            if label != \'transfer\' or not self.report.get(\'lift_verified\'):\n                raise RuntimeError(\'Capture the cup up-axis only after a verified lift\')\n            self.cup_up_reference = measured_up.copy()\n            self.initial_carry_relative_p = np.array(reference[\'position\'], copy=True)\n            self.initial_carry_relative_up = np.array(reference[\'rotation\'][:, 2], copy=True)\n            self.report[\'cup_up_reference_world\'] = self.cup_up_reference.tolist()\n            self.report[\'cup_tilt_at_transfer_entry_deg\'] = entry_tilt\n            self.event(\'cup_orientation_reference\', tilt_deg=round(entry_tilt, 4),\n                       cup_up_axis=self.cup_up_reference.tolist())\n        desired_up = self.cup_up_reference.copy()\n        require_planning_tilt(desired_up)\n        scratch = self.scratch\n        scratch.qpos[:] = d.qpos  # Planning buffer only; execution never assigned.\n        seed = np.array([d.ctrl[r[\'cid\']] for r in self.arm], dtype=float)\n        jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))\n        ordinal = len(self.object_plans)\n        evidence = dict(\n            label=label, ordinal=ordinal, target_cup_origin_m=goal.tolist(),\n            target_cup_up_world=desired_up.tolist(), entry_actual_cup_tilt_deg=entry_tilt,\n            supplied_tool_axis_not_used_as_cup_target=requested_tool_axis.tolist(),\n            predicted_tilt_ceiling_deg=PLANNING_TILT_LIMIT_DEG,\n            orientation_scope=\'cup axis (2 orientation constraints); yaw free\',\n            success=False, waypoints=[], sampled_maximum_cup_tilt_deg=None,\n            carry_reference=reference[\'record\'])\n        self.object_plans.append(evidence)\n        plan_file = self.out / f\'cup_pose_plan_{ordinal:02d}_{label}.json\'\n        self.event(\'cup_pose_plan_start\', phase=label, plan_index=ordinal,\n                   entry_cup_tilt_deg=round(entry_tilt, 4))\n\n        def evaluate(q):\n            self.budget()\n            scratch.qpos[self.arm_indices] = q\n            self.update_scratch_cup(reference)\n            mj.mj_forward(m, scratch)\n            p = scratch.xpos[self.cup].copy()\n            cup_up = scratch.xmat[self.cup].reshape(3, 3)[:, 2].copy()\n            # Predict a carried point on the GRIPPER chain, not the free-joint cup chain.\n            mj.mj_jac(m, scratch, jp, jr, p, self.gripper_body)\n            return p, cup_up, jp[:, self.arm_dofs].copy(), up_axis_jacobian(cup_up, jr[:, self.arm_dofs])\n\n        def check_sample():\n            cup_up = scratch.xmat[self.cup].reshape(3, 3)[:, 2]\n            angle = require_planning_tilt(cup_up)\n            old = evidence[\'sampled_maximum_cup_tilt_deg\']\n            evidence[\'sampled_maximum_cup_tilt_deg\'] = angle if old is None else max(old, angle)\n            _, _, issues = self.contacts(scratch, True, planning=True)\n            issues += self.extra_carry_contact_issues(reference)\n            if issues:\n                raise RuntimeError(f\'{label}: cup-pose sampled contact rejected: {issues[:2]}\')\n\n        try:\n            p0, a0, _, _ = evaluate(seed)\n            check_sample()\n            path = [seed.copy()]\n            for k in range(1, 41):\n                fraction = k / 40\n                waypoint_p = p0 + fraction * (goal - p0)\n                waypoint_up = base.normalize((1. - fraction) * a0 + fraction * desired_up)\n                q, diag = base.solve_pose(evaluate, waypoint_p, waypoint_up, seed,\n                                         self.low, self.high, self.budget)\n                evidence[\'waypoints\'].append(dict(waypoint=k, **diag))\n                if not diag[\'converged\']:\n                    raise RuntimeError(f\'{label}: cup-axis IK waypoint {k} failed: {diag}\')\n                intervals = max(1, math.ceil(float(np.max(np.abs(q - seed))) / .02))\n                for alpha in np.linspace(0, 1, intervals + 1)[1:]:\n                    evaluate(seed + alpha * (q - seed))\n                    check_sample()\n                path.append(q.copy())\n                seed = q\n                if k % 10 == 0:\n                    self.event(\'cup_pose_waypoint\', phase=label, waypoint=k,\n                               error_mm=round(diag[\'position_error_m\'] * 1000, 4),\n                               cup_axis_error_deg=round(diag[\'axis_error_deg\'], 4))\n            plan = base.parameterize(path, self.dt)\n            end_p, end_up, _, _ = evaluate(seed)\n            check_sample()\n            residual = float(np.linalg.norm(end_p - goal))\n            axis_error = angle_deg(end_up, desired_up)\n            if residual > base.POSITION_TOL + 1e-10 or axis_error > base.AXIS_TOL_DEG + 1e-9:\n                raise RuntimeError(\'Predicted endpoint fails original position/axis IK tolerances\')\n            evidence.update(success=True, predicted_endpoint_m=end_p.tolist(),\n                            predicted_endpoint_error_m=residual,\n                            predicted_endpoint_cup_tilt_deg=tilt_deg(end_up),\n                            predicted_endpoint_cup_axis_error_deg=axis_error,\n                            duration_s=plan[\'duration_s\'])\n            plan.update(target_cup_origin_m=goal.tolist(), target_cup_up_world=desired_up.tolist(),\n                        diagnostics=evidence[\'waypoints\'], carried_object_prediction=reference[\'record\'])\n            self.event(\'cup_pose_plan_complete\', phase=label, plan_index=ordinal,\n                       predicted_cup_tilt_deg=round(tilt_deg(end_up), 4), duration_s=plan[\'duration_s\'])\n            return plan\n        except Exception as exc:\n            evidence[\'failure\'] = f\'{type(exc).__name__}: {exc}\'\n            raise\n        finally:\n            base.write_json(plan_file, evidence)\n\n    def tick(self, command, label, allow_cup=False):\n        before = len(self.rows)\n        try:\n            return super().tick(command, label, allow_cup)\n        except Exception as exc:\n            if len(self.rows) > before:\n                last = self.rows[-1]\n                self.report[\'stop_diagnostics\'] = dict(\n                    phase=last[\'phase\'], simulation_time_s=float(last[\'time_s\']),\n                    actual_cup_tilt_deg=float(last[\'cup_tilt_deg\']),\n                    fixed_normal_n=float(last[\'fixed_normal_n\']),\n                    moving_normal_n=float(last[\'moving_normal_n\']),\n                    table_normal_n=float(last[\'table_normal_n\']),\n                    cup_xyz_m=[float(last[\'cup_\' + c]) for c in \'xyz\'], reason=str(exc))\n            raise\n        finally:\n            if len(self.rows) > before:\n                row = self.rows[-1]\n                # Extra evaluator-only diagnostics, never added to policy observations.\n                d = self.data\n                cup_up = d.xmat[self.cup].reshape(3, 3)[:, 2].copy()\n                Rg = d.xmat[self.gripper_body].reshape(3, 3)\n                for c, value in zip(\'xyz\', cup_up):\n                    row[\'cup_up_\' + c] = float(value)\n                row[\'cup_axis_change_from_lift_deg\'] = None\n                row[\'cup_gripper_relative_axis_change_deg\'] = None\n                row[\'cup_gripper_relative_position_change_m\'] = None\n                if self.cup_up_reference is not None:\n                    row[\'cup_axis_change_from_lift_deg\'] = angle_deg(cup_up, self.cup_up_reference)\n                    row[\'cup_gripper_relative_axis_change_deg\'] = angle_deg(\n                        Rg.T @ cup_up, self.initial_carry_relative_up)\n                    local_p = Rg.T @ (d.xpos[self.cup] - d.xpos[self.gripper_body])\n                    row[\'cup_gripper_relative_position_change_m\'] = float(\n                        np.linalg.norm(local_p - self.initial_carry_relative_p))\n\n    def finish(self):\n        phases = {}\n        for row in self.rows:\n            entry = phases.setdefault(row[\'phase\'], dict(samples=0, maximum_cup_tilt_deg=0.0))\n            entry[\'samples\'] += 1\n            entry[\'maximum_cup_tilt_deg\'] = max(entry[\'maximum_cup_tilt_deg\'], float(row[\'cup_tilt_deg\']))\n        self.report[\'tilt_by_phase\'] = phases\n        self.report[\'maximum_actual_cup_tilt_deg\'] = max(\n            (float(row[\'cup_tilt_deg\']) for row in self.rows), default=None)\n        if self.rows:\n            last = self.rows[-1]\n            self.report[\'last_execution_sample\'] = {k: last.get(k) for k in (\n                \'phase\', \'time_s\', \'cup_tilt_deg\', \'cup_axis_change_from_lift_deg\',\n                \'cup_gripper_relative_axis_change_deg\', \'cup_gripper_relative_position_change_m\')}\n        super().finish()\n\n\ndef main(config):\n    original.target_from_initial([0, 0, 0], config[\'offset_m\'])\n    if not isinstance(config.get(\'record_data\'), bool):\n        raise ValueError(\'record_data must be boolean\')\n    trial = CupOrientationTrial(config)\n    try:\n        trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.event(\'stopped\', reason=trial.report[\'stops\'][-1])\n        (trial.out / \'failure_traceback.txt\').write_text(traceback.format_exc(), encoding=\'utf-8\')\n        if trial.renderer is not None:\n            trial.snapshot(\'stopped\')\n    finally:\n        trial.finish()\n    print(\'TRANSFER 07C RESULT \' + json.dumps({k: trial.report.get(k) for k in (\n        \'completed\', \'transfer_test_passed\', \'grasp_verified\', \'lift_verified\',\n        \'transfer_verified\', \'placement_verified\', \'release_verified\', \'corrections_used\',\n        \'maximum_actual_cup_tilt_deg\', \'transfer_measurements\', \'placement_measurements\',\n        \'stop_diagnostics\', \'stops\', \'report_file\')}, allow_nan=False), flush=True)\n    return 0 if trial.report.get(\'transfer_test_passed\') else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_cup_orientation.py': '"""Synthetic tests: no MuJoCo physics, assets, or invented robot outcomes."""\nfrom pathlib import Path\nimport ast\nimport copy\nimport json\nimport math\nimport tempfile\nimport types\nimport unittest\nimport sys\nimport numpy as np\nimport cup_orientation_worker as patch\nimport object_target_worker as previous\nimport transfer_worker as original\nimport base06b as base\n\n\ndef rx(t):\n    c,s=math.cos(t),math.sin(t)\n    return np.array([[1,0,0],[0,c,-s],[0,s,c]])\n\n\ndef ry(t):\n    c,s=math.cos(t),math.sin(t)\n    return np.array([[c,0,s],[0,1,0],[-s,0,c]])\n\n\ndef rz(t):\n    c,s=math.cos(t),math.sin(t)\n    return np.array([[c,-s,0],[s,c,0],[0,0,1]])\n\n\nclass GeometryTests(unittest.TestCase):\n    def test_upright(self): self.assertAlmostEqual(patch.tilt_deg([0,0,1]),0)\n    def test_tilt_angle(self): self.assertAlmostEqual(patch.tilt_deg(ry(math.radians(24))[:,2]),24)\n    def test_limits_are_distinct(self): self.assertEqual((patch.PLANNING_TILT_LIMIT_DEG,patch.RUNTIME_TILT_LIMIT_DEG),(25,30))\n    def test_planning_boundary_accepted(self): patch.require_planning_tilt(ry(math.radians(25))[:,2])\n    def test_excess_planning_tilt_rejected(self):\n        with self.assertRaises(RuntimeError): patch.require_planning_tilt(ry(math.radians(25.1))[:,2])\n    def test_nan_rejected(self):\n        with self.assertRaises(ValueError): patch.tilt_deg([np.nan,0,1])\n    def test_zero_rejected(self):\n        with self.assertRaises(ValueError): patch.tilt_deg([0,0,0])\n    def test_bad_shape(self):\n        with self.assertRaises(ValueError): patch.tilt_deg([1,0])\n    def test_transform_uses_cup_axis(self):\n        p,a=patch.carried_point_and_axis(np.zeros(3),ry(.4),[.01,0,0],ry(-.4))\n        np.testing.assert_allclose(a,[0,0,1],atol=1e-12)\n        self.assertGreater(patch.tilt_deg(ry(.4)[:,2]),20)\n    def test_inputs_preserved(self):\n        p=np.zeros(3);R=ry(.2);a=np.array([.01,0,0]);A=ry(-.2)\n        old=[x.copy() for x in (p,R,a,A)]\n        patch.carried_point_and_axis(p,R,a,A)\n        for x,y in zip((p,R,a,A),old): np.testing.assert_array_equal(x,y)\n    def test_fixed_tool_axis_does_not_fix_cup_axis(self):\n        T=ry(math.radians(25));L=ry(math.radians(-25));T2=T@rz(math.pi/2)\n        np.testing.assert_allclose(T[:,2],T2[:,2],atol=1e-12)\n        self.assertAlmostEqual(patch.tilt_deg((T@L)[:,2]),0)\n        self.assertGreater(patch.tilt_deg((T2@L)[:,2]),30)\n    def test_axis_jacobian_finite_difference(self):\n        q=np.array([.17,-.24]);u=(rx(q[0])@ry(q[1]))[:,2]\n        W=np.column_stack(([1,0,0],rx(q[0])@np.array([0,1,0])))\n        J=patch.up_axis_jacobian(u,W)\n        for j in range(2):\n            dq=np.zeros(2);dq[j]=1e-6\n            ap=(rx(q[0]+dq[0])@ry(q[1]+dq[1]))[:,2]\n            am=(rx(q[0]-dq[0])@ry(q[1]-dq[1]))[:,2]\n            np.testing.assert_allclose(J[:,j],(ap-am)/2e-6,atol=1e-9)\n    def test_bad_jacobian(self):\n        with self.assertRaises(ValueError): patch.up_axis_jacobian([0,0,1],np.zeros((2,5)))\n    def test_rotation_check(self):\n        with self.assertRaises(ValueError): patch.carried_point_and_axis([0,0,0],np.ones((3,3)),[0,0,0],np.eye(3))\n\n\nclass FakeMJ:\n    """Analytical five-coordinate kinematics only; no integration/contact engine."""\n    def mj_forward(self,m,s):\n        R=rx(s.qpos[3])@ry(s.qpos[4]);p=s.qpos[:3]\n        s.xpos[:]=p\n        s.xmat[0]=(R@ry(math.radians(25))).ravel()\n        s.xmat[1]=R.ravel()\n    def mj_jac(self,m,s,jp,jr,p,bid):\n        if bid!=0: raise AssertionError(\'Jacobian must use the gripper chain\')\n        jp[:]=0;jr[:]=0;jp[:,:3]=np.eye(3)\n        jr[:,3]=[1,0,0];jr[:,4]=rx(s.qpos[3])@np.array([0,1,0])\n\n\nclass PlannerTests(unittest.TestCase):\n    def setUp(self):\n        self.tmp=tempfile.TemporaryDirectory();self.out=Path(self.tmp.name)\n        t=patch.CupOrientationTrial({\'scene\':str(self.out/\'not_a_real_scene.xml\'),\'output\':str(self.out),\'record_data\':False})\n        t.model=types.SimpleNamespace(nv=5);t.mj=FakeMJ()\n        q=np.array([0.,0.,.04,0.,0.]);ctrl=np.r_[q,np.zeros(7)]\n        t.data=types.SimpleNamespace(qpos=q.copy(),ctrl=ctrl,xpos=np.zeros((2,3)),xmat=np.zeros((2,9)))\n        t.scratch=copy.deepcopy(t.data);t.mj.mj_forward(t.model,t.data)\n        t.arm=[{\'cid\':i} for i in range(5)];t.arm_indices=list(range(5));t.arm_dofs=list(range(5))\n        t.gripper_body=0;t.cup=1;t.cup_initial=np.zeros(3);t.low=np.array([-.1,-.1,.001,-1.,-1.]);t.high=np.ones(5)\n        t.dt=.005;t.report[\'grasp_verified\']=True;t.report[\'lift_verified\']=True\n        t.make_carry_reference=lambda label,allowed:{\'position\':np.zeros(3),\'rotation\':ry(-math.radians(25)), \'record\':{}}\n        t.update_scratch_cup=lambda ref:t.mj.mj_forward(t.model,t.scratch)\n        t.contacts=lambda s,*a,**k:({\'fixed\':1.,\'moving\':1.},0.,[])\n        t.extra_carry_contact_issues=lambda ref:[]\n        t.event=lambda *a,**k:None\n        self.t=t\n    def tearDown(self): self.tmp.cleanup()\n    def test_plans_cup_axis_not_supplied_tool_axis(self):\n        plan=self.t.plan_cup([.025,0,.04],ry(math.radians(25))[:,2],\'transfer\')\n        np.testing.assert_allclose(plan[\'target_cup_up_world\'],[0,0,1],atol=1e-12)\n        self.assertLessEqual(self.t.object_plans[-1][\'predicted_endpoint_cup_tilt_deg\'],2)\n    def test_actual_state_unchanged(self):\n        old=copy.deepcopy(self.t.data)\n        self.t.plan_cup([.025,0,.04],ry(.4)[:,2],\'transfer\')\n        for k in (\'qpos\',\'ctrl\',\'xpos\',\'xmat\'):np.testing.assert_array_equal(getattr(self.t.data,k),getattr(old,k))\n    def test_waypoints_and_saving(self):\n        self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n        record=json.loads((self.out/\'cup_pose_plan_00_transfer.json\').read_text())\n        self.assertEqual(len(record[\'waypoints\']),40);self.assertTrue(record[\'success\'])\n        self.assertLessEqual(record[\'sampled_maximum_cup_tilt_deg\'],25)\n    def test_entry_tilt_rejected(self):\n        self.t.data.qpos[3]=math.radians(26);self.t.mj.mj_forward(self.t.model,self.t.data)\n        with self.assertRaises(RuntimeError): self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n    def test_missing_verified_lift(self):\n        self.t.report[\'lift_verified\']=False\n        with self.assertRaises(RuntimeError): self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n    def test_illegal_label(self):\n        with self.assertRaises(ValueError): self.t.plan_cup([.025,0,.04],[0,0,1],\'retreat\')\n    def test_outside_lateral_bound(self):\n        with self.assertRaises(ValueError): self.t.plan_cup([.036,0,.04],[0,0,1],\'transfer\')\n    def test_lowered_elevation_rejected(self):\n        with self.assertRaises(ValueError): self.t.plan_cup([.025,0,.024],[0,0,1],\'transfer\')\n    def test_contacts_not_bypassed(self):\n        self.t.contacts=lambda *a,**k:({},0,[\'collision\'])\n        with self.assertRaisesRegex(RuntimeError,\'contact rejected\'): self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n        self.assertIn(\'failure\',json.loads((self.out/\'cup_pose_plan_00_transfer.json\').read_text()))\n    def test_penetration_guard_called(self):\n        self.t.extra_carry_contact_issues=lambda ref:[\'penetration\']\n        with self.assertRaisesRegex(RuntimeError,\'penetration\'): self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n    def test_shared_state_rejected(self):\n        self.t.scratch=self.t.data\n        with self.assertRaises(RuntimeError): self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n    def test_fixed_cup_reference_survives_replan(self):\n        self.t.plan_cup([.025,0,.04],[0,0,1],\'transfer\')\n        old=self.t.cup_up_reference.copy()\n        self.t.data.qpos[3]=math.radians(3);self.t.mj.mj_forward(self.t.model,self.t.data)\n        p=self.t.plan_cup([.02,0,.035],[1,0,0],\'set_down\')\n        np.testing.assert_array_equal(self.t.cup_up_reference,old)\n        np.testing.assert_array_equal(p[\'target_cup_up_world\'],old)\n\n\nclass GuardTests(unittest.TestCase):\n    def test_runtime_inherited(self):\n        for method in (\'contacts\',\'load\',\'move_jaw\',\'execute_plan\',\'close_on_cup\'):\n            self.assertIs(getattr(patch.CupOrientationTrial,method),getattr(previous.ObjectTargetTransferTrial,method))\n    def test_no_execution_assignment(self):\n        tree=ast.parse(Path(patch.__file__).read_text())\n        for node in ast.walk(tree):\n            if isinstance(node,(ast.Assign,ast.AugAssign,ast.AnnAssign)):\n                targets=node.targets if isinstance(node,ast.Assign) else [node.target]\n                for t in targets:\n                    text=ast.unparse(t)\n                    self.assertNotIn(\'self.data.qpos\',text)\n                    self.assertNotIn(\'self.data.xpos\',text)\n                    self.assertNotIn(\'self.data.ctrl\',text)\n                    self.assertNotIn(\'xfrc_applied\',text)\n    def test_acceptance_and_force_constants(self):\n        self.assertEqual(original.PLACEMENT_TOL_M,.008)\n        self.assertEqual(base.GRIPPER_TORQUE_CAP,.25)\n        np.testing.assert_array_equal(original.OFFSET_M,[.025,0,0])\n    def test_position_feedback_budget_unchanged(self):\n        self.assertEqual(previous.MAX_CORRECTIONS,2)\n        self.assertEqual(previous.MAX_CORRECTION_STEP_M,.010)\n    def test_base_tilt_guard_present(self):\n        source=Path(base.__file__).read_text()\n        self.assertIn("if tilt>30.:issues.append(\'cup tilted beyond engineering bound\')",source)\n    def test_no_simulator_import_during_helpers(self):\n        self.assertNotIn(\'mujoco\',sys.modules)\n\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])\n    result=unittest.TextTestRunner(verbosity=1).run(suite)\n    summary=dict(kind=\'OFFLINE_SYNTHETIC_CUP_AXIS_TESTS_NOT_ROBOT_PHYSICS\',tests_run=result.testsRun,\n                 failures=len(result.failures),errors=len(result.errors),skipped=len(result.skipped),passed=result.wasSuccessful(),\n                 python=sys.version.split()[0],actual_mujoco_physics_executed=False)\n    Path(__file__).with_name(\'cup_orientation_test_summary.json\').write_text(json.dumps(summary,indent=2),encoding=\'utf-8\')\n    print(json.dumps(summary,indent=2))\n    raise SystemExit(0 if result.wasSuccessful() else 2)\n', 'two_object_worker.py': '"""08: continuous right-cup then left-fork engineering sequence.\n\nThe cup stage calls the unchanged 07C implementation. Fork execution uses the\nsame physical MjData after cup release, not a reset or a replayed final image.\nNo learned-policy, camera reasoning or official challenge result is claimed.\nAll object pose assignment is confined to scratch IK state. Real motion uses\njoint position commands and mj_step. No weld, external lift or collision disable.\n"""\nfrom pathlib import Path\nimport csv\nimport json\nimport math\nimport sys\nimport time\nimport traceback\nimport xml.etree.ElementTree as ET\nimport numpy as np\nfrom PIL import Image, ImageDraw\nimport base06b as base\nimport transfer_worker as cup_metrics\nimport cup_orientation_worker as cup07c\n\nREVISION = \'08-cup-then-left-fork-v1\'\nFORK_OFFSET = np.array([-0.025, 0., 0.])\nFORK_TOL = .008\nOTHER_TOL = .010\nCUP_PRESERVE_TOL = .008\nJAW_CAP = .25\nFULL_INSTRUCTION = \'Place the cup 25 mm to the right and the fork 25 mm to the left of their starting positions.\'\n\n\ndef rotation_vector(R):\n    """SO(3) logarithm, including an explicit near-pi branch."""\n    _, R = base.checked_pose(np.zeros(3), R)\n    a = math.acos(float(np.clip((np.trace(R)-1.)/2., -1., 1.)))\n    v = np.array([R[2,1]-R[1,2], R[0,2]-R[2,0], R[1,0]-R[0,1]])\n    if a < 1e-7:\n        return .5*v\n    if math.pi-a < 1e-5:\n        vals, vecs = np.linalg.eigh((R+R.T)/2.)\n        axis = vecs[:, np.argmax(vals)]\n        if np.dot(axis, v) < 0: axis = -axis\n        return axis*a\n    return v*(a/(2.*math.sin(a)))\n\n\ndef rotation_exp(v):\n    v = np.asarray(v, dtype=float)\n    if v.shape != (3,) or not np.isfinite(v).all(): raise ValueError(\'Invalid rotation vector\')\n    a = float(np.linalg.norm(v)); K = base.skew(v)\n    if a < 1e-7: return np.eye(3)+K+.5*K@K\n    return np.eye(3)+(math.sin(a)/a)*K+((1-math.cos(a))/a**2)*K@K\n\n\ndef upright_frame(length_direction):\n    y = np.asarray(length_direction, dtype=float).copy()\n    if y.shape != (3,) or not np.isfinite(y).all(): raise ValueError(\'Invalid fork direction\')\n    y[2] = 0.; y = base.normalize(y)\n    z = np.array([0., 0., 1.]); x = base.normalize(np.cross(y,z))\n    return np.column_stack([x,y,z])\n\n\ndef support_radius(kind, size, R, normal):\n    if kind != \'capsule\': return base.primitive_plane_radius(kind, size, R, normal)\n    _, R = base.checked_pose(np.zeros(3), R)\n    s = np.asarray(size,dtype=float)\n    if s.shape != (3,) or not np.isfinite(s).all() or s[0] <= 0 or s[1] < 0:\n        raise ValueError(\'Invalid capsule size\')\n    n = base.normalize(normal)\n    return float(s[0]+s[1]*abs(np.dot(R[:,2],n)))\n\n\ndef fork_placement_metrics(rows, initial, target, initial_R):\n    if not rows: raise ValueError(\'No final fork samples\')\n    p0, target = np.asarray(initial), np.asarray(target)\n    if p0.shape != (3,) or target.shape != (3,) or not np.allclose(target-p0,FORK_OFFSET,atol=1e-9,rtol=0):\n        raise ValueError(\'Fork destination must be initial position minus 25 mm world X\')\n    xyz = np.array([[r[\'fork_\'+a] for a in \'xyz\'] for r in rows],dtype=float)\n    vals = np.array([[r[\'fork_table_normal_n\'],r[\'fork_fixed_normal_n\'],r[\'fork_moving_normal_n\'],\n                      r[\'fork_orientation_error_deg\']] for r in rows],dtype=float)\n    if not np.isfinite(xyz).all() or not np.isfinite(vals).all(): raise ValueError(\'Invalid final measurements\')\n    error = float(np.linalg.norm(xyz-target,axis=1).max())\n    progress = float((p0[0]-xyz[:,0]).min())\n    fraction = float(np.mean((vals[:,0]>=base.CONTACT_MIN_N)&(vals[:,1]<base.CONTACT_MIN_N)&(vals[:,2]<base.CONTACT_MIN_N)))\n    angle = float(vals[:,3].max())\n    return dict(maximum_destination_error_m=error,minimum_leftward_displacement_m=progress,\n                supported_and_released_fraction=fraction,maximum_orientation_error_deg=angle,\n                placement_verified=bool(error<=FORK_TOL and progress>=.025-FORK_TOL and fraction>=.8 and angle<=15.))\n\n\ndef solve_frame(evaluate, target, Rtarget, seed, low, high, budget):\n    ptarget,Rtarget = base.checked_pose(target,Rtarget)\n    q,lo,hi = [np.array(x,dtype=float,copy=True) for x in (seed,low,high)]\n    if q.ndim!=1 or lo.shape!=q.shape or hi.shape!=q.shape or not np.isfinite([q,lo,hi]).all() or np.any(lo>=hi) or np.any(q<lo) or np.any(q>hi):\n        raise ValueError(\'Invalid frame IK bounds\')\n    for k in range(180):\n        budget(); p,R,jp,jr=evaluate(q)\n        ep=ptarget-p; er=rotation_vector(Rtarget@R.T)\n        pe=float(np.linalg.norm(ep)); ae=math.degrees(float(np.linalg.norm(er)))\n        if pe<=base.POSITION_TOL and ae<=base.AXIS_TOL_DEG:\n            return q,dict(converged=True,iterations=k+1,position_error_m=pe,rotation_error_deg=ae)\n        J=np.vstack([jp,.08*jr]); e=np.r_[ep,.08*er]\n        dq=np.linalg.solve(J.T@J+1e-6*np.eye(len(q)),J.T@e)\n        dq*=min(1.,.045/max(float(np.max(np.abs(dq))),1e-12))\n        improved=False\n        for scale in (1.,.5,.25,.125,.0625):\n            candidate=np.clip(q+scale*dq,lo,hi)\n            pc,Rc,_,_=evaluate(candidate)\n            ec=np.r_[ptarget-pc,.08*rotation_vector(Rtarget@Rc.T)]\n            if np.linalg.norm(ec)<np.linalg.norm(e)-1e-11:\n                q=candidate;improved=True;break\n        if not improved: break\n    p,R,_,_=evaluate(q)\n    return q,dict(converged=False,iterations=k+1,position_error_m=float(np.linalg.norm(ptarget-p)),\n                  rotation_error_deg=math.degrees(float(np.linalg.norm(rotation_vector(Rtarget@R.T)))))\n\n\nclass CombinedTrial(cup07c.CupOrientationTrial):\n    def __init__(self,config):\n        super().__init__(config)\n        self.fork_stage=None\n        self.report.update(kind=\'ENGINEERING_TWO_OBJECT_SEQUENCE_NOT_TRAINED_VLA\',\n            task_revision=REVISION,sequence_test_passed=False,cup_stage_passed=False,\n            fork_grasp_verified=False,fork_lift_verified=False,fork_transfer_verified=False,\n            fork_placement_verified=False,cup_preserved_after_fork=False,\n            full_instruction_label=FULL_INSTRUCTION,instruction_interpreted=False,\n            sequence_continuity=\'same model and MjData; cup stage then fork stage, no reset\',\n            demonstration_episode_count=1)\n\n    def load(self):\n        super().load()\n        if self.steps!=0 or self.data.time!=0: raise RuntimeError(\'Effort setup must precede physics execution\')\n        aid=self.needed(self.mj.mjtObj.mjOBJ_ACTUATOR,\'left_gripper\')\n        original=self.model.actuator_forcerange[aid].copy()\n        if not self.model.actuator_forcelimited[aid] or original[0]>-JAW_CAP or original[1]<JAW_CAP:\n            raise ValueError(\'Cannot verify the left effort cap only reduces the original effort\')\n        scene=Path(self.report[\'scene\'])\n        tree=ET.parse(scene); servo=tree.getroot().find("actuator/position[@name=\'left_gripper\']")\n        if servo is None: raise ValueError(\'Expected left position servo\')\n        servo.set(\'forcerange\',f\'{-JAW_CAP} {JAW_CAP}\');servo.set(\'forcelimited\',\'true\')\n        tree.write(scene,encoding=\'utf-8\',xml_declaration=True)\n        self.model.actuator_forcerange[aid]=[-JAW_CAP,JAW_CAP]\n        self.report[\'scene_sha256\']=base.digest(scene)\n        self.report[\'left_gripper_setup\']=dict(previous_force_range_nm=original.tolist(),\n            new_force_range_nm=[-JAW_CAP,JAW_CAP],time_s=0,\n            scope=\'new copied scene and model only; no geometry/friction changes\')\n        if self.config.get(\'record_data\'):\n            for name in (\'top\',\'left_wrist_cam\',\'right_wrist_cam\'):\n                self.needed(self.mj.mjtObj.mjOBJ_CAMERA,name)\n\n    def snapshot(self, label):\n        if self.renderer is None: return\n        try:\n            self.renderer.update_scene(self.data,camera=\'top\',scene_option=self.options)\n            im=Image.fromarray(self.renderer.render().copy());draw=ImageDraw.Draw(im)\n            draw.rectangle((0,0,im.width,43),fill=\'black\')\n            draw.text((8,4),\'TWO-OBJECT ENGINEERING SEQUENCE | no learned policy / camera decisions\',fill=\'white\')\n            draw.text((8,23),f\'{label} | t={self.data.time:.2f}s | cup +25 mm X; fork -25 mm X\',fill=\'white\')\n            path=self.out/f\'{label}.png\';im.save(path);self.snapshots[label]=str(path)\n        except Exception as exc:self.report[\'graphics_error\']=\'snapshot: \'+str(exc)\n\n    def camera_file(self, camera, frame):\n        folder=self.out/\'observations\';folder.mkdir(exist_ok=True)\n        self.renderer.update_scene(self.data,camera=camera,scene_option=self.options)\n        pixels=self.renderer.render().copy()\n        path=folder/f\'{frame:05d}_{camera}.jpg\'\n        Image.fromarray(pixels).save(path,quality=85)\n        return str(path.relative_to(self.out))\n\n    def tick(self,command,label,allow_cup=False):\n        # This method is used only for the unchanged cup stage.\n        n=len(self.video_rows); left=None; before=self.steps\n        if self.config.get(\'record_data\') and self.renderer is not None and before%round(.1/self.dt)==0:\n            try: left=self.camera_file(\'left_wrist_cam\',n)\n            except Exception as exc:\n                self.report[\'graphics_error\']=\'left-camera recording: \'+str(exc)\n                self.config[\'record_data\']=False\n        try: return super().tick(command,label,allow_cup)\n        finally:\n            for obs in self.video_rows[n:]:\n                if obs[\'step\']!=before: raise RuntimeError(\'Pre-action camera alignment failed\')\n                if left: obs[\'images\'][\'left_wrist_cam\']=left\n                obs[\'instruction\']=FULL_INSTRUCTION;obs[\'stage\']=\'cup_transfer\'\n                obs[\'demonstration_task\']=\'two_object_scripted_engineering_sequence\'\n\n    def execute(self):\n        # Calls the same successful 07C cup manipulation, in a new continuous episode.\n        super().execute()\n        self.report[\'cup_stage_passed\']=bool(self.report.get(\'transfer_test_passed\'))\n        self.report[\'cup_stage_end_time_s\']=float(self.data.time)\n        self.report[\'cup_stage_placement\']=dict(self.report.get(\'placement_measurements\',{}))\n        if not self.report[\'cup_stage_passed\']: raise RuntimeError(\'Cup must pass before left-arm motion\')\n        self.snapshot(\'cup_placed\')\n        self.report[\'completed\']=False;self.report[\'grasp_test_passed\']=False\n        self.fork_stage=ForkStage(self)\n        self.fork_stage.execute()\n        rows=self.fork_stage.stable(\'fork_final_settle\',.5)\n        cup=cup_metrics.placement_metrics(rows,self.cup_initial,self.target)\n        self.report[\'cup_final_placement\']=cup\n        self.report[\'cup_preserved_after_fork\']=bool(cup[\'placement_verified\'])\n        if not self.report[\'cup_preserved_after_fork\']:\n            raise RuntimeError(\'Cup destination goal failed after fork placement\')\n        self.report[\'sequence_test_passed\']=True\n        self.report[\'completed\']=True;self.report[\'grasp_test_passed\']=True\n        self.snapshot(\'sequence_final\')\n\n    def finish(self):\n        # One trace and one manifest across both stages, including partial episodes.\n        if self.renderer is not None:\n            try:self.renderer.close()\n            except Exception as exc:self.report[\'graphics_error\']=self.report[\'graphics_error\'] or str(exc)\n        modified=self.source.is_file() and base.digest(self.source)!=self.config[\'scene_sha256\']\n        if modified:self.report[\'stops\'].append(\'Original source scene changed\')\n        passed=bool(self.report.get(\'sequence_test_passed\') and self.report.get(\'completed\') and not self.report[\'stops\'] and not modified)\n        self.report.update(sequence_test_passed=passed,completed=passed,source_scene_modified=modified,\n            physics_steps=self.steps,simulation_time_s=float(self.data.time) if self.data is not None else 0,\n            snapshots=self.snapshots,timing_events=self.events,recorded_observation_action_pairs=len(self.video_rows))\n        if self.rows:\n            columns=list(dict.fromkeys(k for r in self.rows for k in r))\n            path=self.out/\'sequence_trace.csv\'\n            with path.open(\'w\',newline=\'\',encoding=\'utf-8\') as f:\n                w=csv.DictWriter(f,fieldnames=columns);w.writeheader();w.writerows(self.rows)\n            self.report[\'trace_file\']=str(path)\n        if self.video_rows:\n            stride=round(.1/self.dt)\n            for obs in self.video_rows:\n                selected=self.rows[obs[\'step\']:obs[\'step\']+stride]\n                obs[\'action_chunk\']=[[float(row[r[\'name\']+\'_command_rad\']) for r in self.mapping] for row in selected]\n                obs[\'action_chunk_dt_s\']=self.dt;obs[\'action_chunk_valid_length\']=len(selected)\n                obs[\'episode_outcome\']=\'passed_engineering_sequence\' if passed else \'failed_or_partial_engineering_sequence\'\n            path=self.out/\'expert_observations.jsonl\'\n            with path.open(\'w\',encoding=\'utf-8\') as f:\n                for obs in self.video_rows:f.write(json.dumps(obs,allow_nan=False)+\'\\n\')\n            self.report[\'observation_manifest\']=str(path)\n        self.report[\'recording_complete\']=bool(passed and self.video_rows and self.report.get(\'graphics_error\') is None and\n            all(set(o[\'images\'])=={\'top\',\'left_wrist_cam\',\'right_wrist_cam\'} for o in self.video_rows))\n        self.report[\'training_dataset_ready\']=False\n        self.report[\'wall_time_s_before_report\']=time.monotonic()-self.started\n        self.report[\'report_file\']=str(self.out/\'sequence_report.json\')\n        base.write_json(self.out/\'sequence_report.json\',self.report)\n\n\nclass ForkStage:\n    """Left-arm phase using the SAME MjData after the cup stage."""\n    def __init__(self,owner):\n        self.o=owner;self.m=owner.model;self.d=owner.data;self.mj=owner.mj\n        self.arm=[owner.byname[\'left_\'+j] for j in base.JOINT_SUFFIXES[:-1]]\n        self.g=owner.byname[\'left_gripper\'];self.qa=[r[\'qadr\'] for r in self.arm];self.va=[r[\'vadr\'] for r in self.arm]\n        self.lo=np.array([r[\'low\']+.02 for r in self.arm]);self.hi=np.array([r[\'high\']-.02 for r in self.arm])\n        self.fork=owner.needed(self.mj.mjtObj.mjOBJ_BODY,\'development_fork\')\n        self.gripper=owner.needed(self.mj.mjtObj.mjOBJ_BODY,\'left_gripper\')\n        self.moving=owner.needed(self.mj.mjtObj.mjOBJ_BODY,\'left_moving_jaw_so101_v1\')\n        self.handle=owner.needed(self.mj.mjtObj.mjOBJ_GEOM,\'fork_handle\')\n        self.fixed_tip=owner.needed(self.mj.mjtObj.mjOBJ_GEOM,\'left_fixed_jaw_sph_tip1\')\n        self.scratch=self.mj.MjData(self.m)\n        self.geoms={g for g in range(self.m.ngeom) if int(self.m.geom_bodyid[g])==self.fork}\n        if int(self.m.geom_type[self.handle])!=int(self.mj.mjtGeom.mjGEOM_CAPSULE):raise ValueError(\'Expected capsule fork handle\')\n        self.radius=float(self.m.geom_size[self.handle,0])\n        if not .003<=self.radius<=.005:raise ValueError(\'Fork handle differs from reviewed 6–10 mm diameter\')\n        if int(self.m.geom_type[self.fixed_tip])!=int(self.mj.mjtGeom.mjGEOM_SPHERE):raise ValueError(\'Expected spherical fixed fingertip\')\n        jid=int(self.m.body_jntadr[self.fork])\n        if int(self.m.body_jntnum[self.fork])!=1 or int(self.m.jnt_type[jid])!=int(self.mj.mjtJoint.mjJNT_FREE):\n            raise ValueError(\'Fork must have one free joint\')\n        self.qadr=int(self.m.jnt_qposadr[jid])\n        self.initial=self.d.xpos[self.fork].copy();self.initial_R=self.d.xmat[self.fork].reshape(3,3).copy()\n        self.target=self.initial+FORK_OFFSET\n        self.cup_reference=self.d.xpos[owner.cup].copy()\n        self.plate=owner.needed(self.mj.mjtObj.mjOBJ_BODY,\'development_plate\')\n        self.plate_reference=self.d.xpos[self.plate].copy()\n        self.right_q={r[\'name\']:float(self.d.qpos[r[\'qadr\']]) for r in owner.mapping if r[\'name\'].startswith(\'right_\')}\n        self.local_point=self.m.geom_pos[self.fixed_tip].copy()+np.array([self.radius+self.m.geom_size[self.fixed_tip,0]+.0015,0,0])\n        self.open_q=.22;self.close_floor=-.155\n        if not self.g[\'low\']+.01<self.close_floor<self.open_q<self.g[\'high\']-.01:raise ValueError(\'Fork jaw limits incompatible\')\n        self.jaws={}\n        for gid in range(self.m.ngeom):\n            bid=int(self.m.geom_bodyid[gid]);name=owner.name(self.mj.mjtObj.mjOBJ_GEOM,gid)\n            mesh=\'\'\n            if int(self.m.geom_type[gid])==int(self.mj.mjtGeom.mjGEOM_MESH):mesh=owner.name(self.mj.mjtObj.mjOBJ_MESH,int(self.m.geom_dataid[gid]))\n            if bid==self.gripper and (name.startswith(\'left_fixed_jaw_sph_\') or name in\n                {\'left_fixed_jaw_box3\',\'left_fixed_jaw_box4\',\'left_fixed_jaw_box5\',\'left_fixed_jaw_box6\',\'left_fixed_jaw_box7\'} or mesh==\'wrist_roll_follower_so101_gripper_part0_v1\'):\n                self.jaws[gid]=\'fixed\'\n            elif bid==self.moving:self.jaws[gid]=\'moving\'\n        if set(self.jaws.values())!={\'fixed\',\'moving\'}:raise ValueError(\'Missing left contact groups\')\n        self.close_target=None;self.carry_R=None;self.start_row=len(owner.rows);self.loss_steps=0\n        owner.report.update(fork_initial_position_m=self.initial.tolist(),fork_destination_position_m=self.target.tolist(),\n            fork_grasp_settings=dict(handle_radius_m=self.radius,opening_rad=self.open_q,closure_floor_rad=self.close_floor,\n                grasp_height_offset_m=.0008,tool_frame=\'vertical tool; local Y parallel to projected handle axis\',\n                full_rotation_ik=\'bounded local solve; five arm joints may not reach every requested frame\'),\n            fork_limits=dict(placement_m=FORK_TOL,orientation_deg=15,lateral_m=.035,tilt_deg=30,\n                minimum_carry_rise_m=.025,cup_preservation_m=CUP_PRESERVE_TOL,plate_preservation_m=OTHER_TOL))\n        self.check_workspace()\n\n    def check_workspace(self):\n        R=self.d.geom_xmat[self.o.table].reshape(3,3)\n        if not np.allclose(R[:,2],[0,0,1],atol=1e-6):raise ValueError(\'Horizontal table required\')\n        local=R.T@(self.target-self.d.geom_xpos[self.o.table])\n        # Conservative footprint radius includes every fork geom.\n        radius=max(float(np.linalg.norm(self.d.geom_xpos[g]-self.initial)+self.m.geom_rbound[g]) for g in self.geoms)\n        if np.any(self.m.geom_size[self.o.table,:2]-np.abs(local[:2])-radius < .005):raise RuntimeError(\'Fork destination too near table edge\')\n        if np.linalg.norm(self.target[:2]-self.d.xpos[self.plate,:2])<radius+.065+.005:raise RuntimeError(\'Fork destination too near plate\')\n        if np.linalg.norm(self.target[:2]-self.d.xpos[self.o.cup,:2])<radius+self.o.radius+.005:raise RuntimeError(\'Fork destination too near cup\')\n\n    def point(self,data):\n        return data.xpos[self.gripper]+data.xmat[self.gripper].reshape(3,3)@self.local_point\n\n    def contact_state(self,data,allow,planning=False):\n        ff={\'fixed\':0.,\'moving\':0.};cf={\'fixed\':0.,\'moving\':0.};support=0.;cup_support=0.;issues=[]\n        for i in range(data.ncon):\n            c=data.contact[i];a,b=int(c.geom1),int(c.geom2);dist=float(c.dist)\n            if a<0 or b<0 or not math.isfinite(dist):raise RuntimeError(\'Invalid rigid contact\')\n            touches=a in self.geoms or b in self.geoms;other=b if a in self.geoms else a\n            jaw=self.jaws.get(other) if touches else None\n            intended=bool(allow and jaw)\n            force=np.zeros(6)\n            if not planning and c.efc_address>=0:self.mj.mj_contactForce(self.m,data,i,force)\n            normal=max(0.,float(force[0]))\n            if not math.isfinite(normal):issues.append(\'non-finite contact force\')\n            if touches and jaw and dist<=.0005:ff[jaw]+=normal\n            if touches and other==self.o.table:support+=normal\n            cup=a in self.o.cup_geoms or b in self.o.cup_geoms;cother=b if a in self.o.cup_geoms else a\n            if cup and cother==self.o.table:cup_support+=normal\n            cjaw=self.o.geom_group.get(cother) if cup else None\n            if cjaw and dist<=.0005:cf[cjaw]+=normal\n            if dist < -base.PENETRATION_LIMIT:issues.append(\'excess penetration\')\n            if (a in self.o.robot_geoms or b in self.o.robot_geoms) and dist<=0 and not intended:\n                issues.append(\'unexpected robot contact: \'+self.o.name(self.mj.mjtObj.mjOBJ_GEOM,a)+\' / \'+self.o.name(self.mj.mjtObj.mjOBJ_GEOM,b))\n            if intended and normal>25:issues.append(\'finger normal force beyond engineering limit\')\n        return ff,support,cf,cup_support,issues\n\n    def tick(self,command,label,allow=False):\n        o,m,d,mj=self.o,self.m,self.d,self.mj;o.budget();command=np.asarray(command,dtype=float)\n        if command.shape!=(12,) or not np.isfinite(command).all():raise ValueError(\'Invalid action\')\n        if np.any(np.abs(command-d.ctrl)>base.COMMAND_SPEED*o.dt*1.06+1e-8):raise RuntimeError(\'Command slew guard\')\n        for r in o.mapping:\n            if not r[\'low\']<=command[r[\'cid\']]<=r[\'high\']:raise RuntimeError(\'Command outside bounds\')\n        if o.config.get(\'record_data\') and o.renderer is not None and o.steps%round(.1/o.dt)==0:\n            try:\n                frame=len(o.video_rows);images={cam:o.camera_file(cam,frame) for cam in (\'top\',\'left_wrist_cam\',\'right_wrist_cam\')}\n                o.video_rows.append(dict(frame_id=frame,step=o.steps,time_s=float(d.time),images=images,\n                    instruction=FULL_INSTRUCTION,stage=\'left_fork_placement\',demonstration_task=\'two_object_scripted_engineering_sequence\',\n                    state=[float(d.qpos[r[\'qadr\']]) for r in o.mapping],action=[float(command[r[\'cid\']]) for r in o.mapping],\n                    action_order=[r[\'name\'] for r in o.mapping],action_semantics=\'native absolute joint position targets, radians; applied after observation\',\n                    expert_source=\'privileged simulator-state engineering controller; not learned-policy output\'))\n            except Exception as exc:\n                o.report[\'graphics_error\']=\'fork recording: \'+str(exc);o.config[\'record_data\']=False\n        d.ctrl[:]=command;t=float(d.time);mj.mj_step(m,d);o.steps+=1;mj.mj_forward(m,d)\n        if not math.isclose(float(d.time),t+o.dt,abs_tol=1e-8):raise RuntimeError(\'Unexpected simulation time/reset\')\n        if not np.isfinite(d.qpos).all() or not np.isfinite(d.qvel).all():raise RuntimeError(\'Non-finite state\')\n        ff,fs,cf,cs,issues=self.contact_state(d,allow)\n        if np.any(np.asarray(d.warning.number)>o.warnings0):issues.append(\'MuJoCo warning\')\n        fpos=d.xpos[self.fork].copy();R=d.xmat[self.fork].reshape(3,3)\n        tilt=cup07c.tilt_deg(R[:,2]);rot=math.degrees(float(np.linalg.norm(rotation_vector(R@self.initial_R.T))))\n        cup_xyz=d.xpos[o.cup].copy();cup_change=float(np.linalg.norm(cup_xyz-self.cup_reference))\n        plate_change=float(np.linalg.norm(d.xpos[self.plate]-self.plate_reference))\n        row=dict(step=o.steps,time_s=float(d.time),phase=label,cup_x=float(cup_xyz[0]),cup_y=float(cup_xyz[1]),cup_z=float(cup_xyz[2]),\n            cup_tilt_deg=cup07c.tilt_deg(d.xmat[o.cup].reshape(3,3)[:,2]),fixed_normal_n=cf[\'fixed\'],moving_normal_n=cf[\'moving\'],table_normal_n=cs,\n            two_jaw_contact=cf[\'fixed\']>=base.CONTACT_MIN_N and cf[\'moving\']>=base.CONTACT_MIN_N,\n            fork_x=float(fpos[0]),fork_y=float(fpos[1]),fork_z=float(fpos[2]),fork_tilt_deg=tilt,fork_orientation_error_deg=rot,\n            fork_fixed_normal_n=ff[\'fixed\'],fork_moving_normal_n=ff[\'moving\'],fork_table_normal_n=fs,\n            fork_two_jaw_contact=min(ff.values())>=base.CONTACT_MIN_N,\n            cup_displacement_during_fork_m=cup_change,plate_displacement_during_fork_m=plate_change)\n        for r in o.mapping:\n            q,v=float(d.qpos[r[\'qadr\']]),float(d.qvel[r[\'vadr\']]);cmd=float(command[r[\'cid\']])\n            row[r[\'name\']+\'_position_rad\']=q;row[r[\'name\']+\'_command_rad\']=cmd\n            if not r[\'low\']-.03<=q<=r[\'high\']+.03:issues.append(\'Joint range: \'+r[\'name\'])\n            if abs(v)>3.:issues.append(\'Joint speed: \'+r[\'name\'])\n            limit=.65 if allow and r[\'name\']==\'left_gripper\' else .12\n            if abs(q-cmd)>limit:issues.append(\'Tracking: \'+r[\'name\'])\n            if r[\'name\'] in self.right_q and abs(q-self.right_q[r[\'name\']])>.15:issues.append(\'Inactive right arm drift\')\n        if tilt>30:issues.append(\'fork tilt above 30 degrees\')\n        if np.linalg.norm(fpos[:2]-self.initial[:2])>.035:issues.append(\'fork lateral excursion above 35 mm\')\n        if cup_change>CUP_PRESERVE_TOL:issues.append(\'placed cup disturbed during fork stage\')\n        if plate_change>OTHER_TOL:issues.append(\'plate disturbed during fork stage\')\n        if label in (\'fork_transfer\',\'fork_transfer_hold\'):\n            if fpos[2]-self.initial[2]<.025:issues.append(\'fork lost carry elevation\')\n            if fs>=base.CONTACT_MIN_N:issues.append(\'fork supported by table during transfer\')\n            self.loss_steps=0 if row[\'fork_two_jaw_contact\'] else self.loss_steps+1\n            if self.loss_steps>=round(.10/o.dt):issues.append(\'fork lost two-jaw contact\')\n        o.rows.append(row)\n        if issues:\n            o.report[\'fork_stop_diagnostics\']=dict(phase=label,time_s=float(d.time),fork_position_m=fpos.tolist(),tilt_deg=tilt,\n                fixed_normal_n=ff[\'fixed\'],moving_normal_n=ff[\'moving\'],table_normal_n=fs,issues=issues[:6])\n            raise RuntimeError(\'; \'.join(issues[:3]))\n        return row\n\n    def hold(self,seconds,label,allow=False):\n        self.o.event(\'fork_hold\',phase=label)\n        for _ in range(round(seconds/self.o.dt)):self.tick(self.d.ctrl.copy(),label,allow)\n\n    def jaw(self,target,label,allow=False):\n        start=self.d.ctrl.copy();end=start.copy();end[self.g[\'cid\']]=target\n        duration=max(.6,1.6*abs(target-start[self.g[\'cid\']])/base.COMMAND_SPEED)\n        n=math.ceil(duration/self.o.dt)\n        for k in range(1,n+1):self.tick(start+base.smooth(k/n)*(end-start),label,allow)\n\n    def stable(self,label,seconds=.25):\n        rows=[r for r in self.o.rows[self.start_row:] if r[\'phase\']==label]\n        if not rows:raise RuntimeError(\'No measurements for \'+label)\n        t=rows[-1][\'time_s\'];return [r for r in rows if r[\'time_s\']>=t-seconds-1e-9]\n\n    def plan(self,target,Rtarget,label,allow=False,carry=False):\n        o,m,d,mj=self.o,self.m,self.d,self.mj;o.event(\'fork_plan_start\',phase=label)\n        s=self.scratch\n        if s is d or np.shares_memory(s.qpos,d.qpos):raise RuntimeError(\'Scratch/execution overlap\')\n        target,Rtarget=base.checked_pose(target,Rtarget)\n        s.qpos[:]=d.qpos\n        reference=None\n        if carry:\n            if not o.report[\'fork_grasp_verified\']:raise RuntimeError(\'Carry requires verified fork grasp\')\n            ff,_,_,_,issues=self.contact_state(d,True)\n            if issues or min(ff.values())<base.CONTACT_MIN_N:raise RuntimeError(\'No stable fork grasp at carry-plan entry\')\n            reference=base.relative_pose(d.xpos[self.gripper],d.xmat[self.gripper].reshape(3,3),d.xpos[self.fork],d.xmat[self.fork].reshape(3,3))\n        jp,jr=np.zeros((3,m.nv)),np.zeros((3,m.nv))\n        def evaluate(q):\n            o.budget();s.qpos[self.qa]=q\n            if reference:\n                mj.mj_kinematics(m,s)\n                p,R=base.predicted_pose(s.xpos[self.gripper],s.xmat[self.gripper].reshape(3,3),*reference)\n                quat=np.empty(4);mj.mju_mat2Quat(quat,np.ascontiguousarray(R).reshape(9))\n                s.qpos[self.qadr:self.qadr+3]=p;s.qpos[self.qadr+3:self.qadr+7]=quat/np.linalg.norm(quat)\n            mj.mj_forward(m,s)\n            p=s.xpos[self.fork].copy() if carry else self.point(s)\n            R=s.xmat[self.fork if carry else self.gripper].reshape(3,3).copy()\n            mj.mj_jac(m,s,jp,jr,p,self.gripper)\n            return p,R,jp[:,self.va].copy(),jr[:,self.va].copy()\n        seed=np.array([d.ctrl[r[\'cid\']] for r in self.arm]);p0,R0,_,_=evaluate(seed)\n        w=rotation_vector(Rtarget@R0.T);path=[seed.copy()];diags=[]\n        for k in range(1,41):\n            alpha=k/40;Rp=rotation_exp(alpha*w)@R0\n            q,diag=solve_frame(evaluate,p0+alpha*(target-p0),Rp,seed,self.lo,self.hi,o.budget)\n            diags.append(diag)\n            if not diag[\'converged\']:\n                base.write_json(o.out/f\'plan_{label}_failure.json\',dict(target_m=target.tolist(),waypoint=k,diagnostics=diags))\n                raise RuntimeError(f\'{label}: local fork frame IK failed at waypoint {k}: {diag}\')\n            n=max(1,math.ceil(float(np.max(np.abs(q-seed)))/.02))\n            for t in np.linspace(0,1,n+1)[1:]:\n                evaluate(seed+t*(q-seed));_,_,_,_,issues=self.contact_state(s,allow,planning=True)\n                if carry and cup07c.tilt_deg(s.xmat[self.fork].reshape(3,3)[:,2])>25:issues.append(\'predicted fork tilt above 25 degrees\')\n                if issues:raise RuntimeError(f\'{label}: sampled collision rejection: {issues[:3]}\')\n            path.append(q.copy());seed=q\n        plan=base.parameterize(path,o.dt);plan.update(target_m=target.tolist(),target_rotation=Rtarget.tolist(),carry_prediction_only=carry,diagnostics=diags)\n        base.write_json(o.out/f\'plan_{label}.json\',plan)\n        o.event(\'fork_plan_ready\',phase=label,duration_s=plan[\'duration_s\'])\n        return plan\n\n    def move(self,plan,label,allow=False):\n        n=round(plan[\'duration_s\']/self.o.dt);start=self.d.ctrl.copy()\n        self.o.event(\'fork_motion\',phase=label)\n        for k in range(1,n+1):\n            q=base.path_value(plan,base.smooth(k/n));cmd=start.copy()\n            for r,val in zip(self.arm,q):cmd[r[\'cid\']]=val\n            self.tick(cmd,label,allow)\n\n    def grasp(self):\n        start=float(self.d.ctrl[self.g[\'cid\']]);n=round(4./self.o.dt);consecutive=0\n        for k in range(1,n+1):\n            cmd=self.d.ctrl.copy();cmd[self.g[\'cid\']]=start+base.smooth(k/n)*(self.close_floor-start)\n            row=self.tick(cmd,\'fork_close\',True)\n            consecutive=consecutive+1 if row[\'fork_two_jaw_contact\'] else 0\n            if consecutive>=round(.10/self.o.dt):self.close_target=float(cmd[self.g[\'cid\']]);break\n        if self.close_target is None:raise RuntimeError(\'Fork lacks sustained two-jaw contact; no lift executed\')\n        self.hold(.4,\'fork_grasp_hold\',True)\n        frac=float(np.mean([r[\'fork_two_jaw_contact\'] for r in self.stable(\'fork_grasp_hold\')]))\n        self.o.report[\'fork_grasp_contact_fraction\']=frac\n        if frac<.8:raise RuntimeError(\'Fork grasp contact not stable\')\n        self.o.report[\'fork_grasp_verified\']=True;self.carry_R=self.d.xmat[self.fork].reshape(3,3).copy()\n        self.o.snapshot(\'fork_grasped\')\n\n    def carry_metrics(self,label):\n        rows=self.stable(label,.5 if label==\'fork_lift_hold\' else .25)\n        p=np.array([[r[\'fork_\'+c] for c in \'xyz\'] for r in rows])\n        return dict(minimum_rise_m=float((p[:,2]-self.initial[2]).min()),\n            maximum_horizontal_destination_error_m=float(np.linalg.norm(p[:,:2]-self.target[:2],axis=1).max()),\n            two_jaw_contact_fraction=float(np.mean([r[\'fork_two_jaw_contact\'] for r in rows])),\n            maximum_table_support_n=max(r[\'fork_table_normal_n\'] for r in rows))\n\n    def bottom_clearance(self):\n        top=self.d.geom_xpos[self.o.table]+np.array([0,0,self.m.geom_size[self.o.table,2]])\n        gaps=[]\n        for g in self.geoms:\n            typ=int(self.m.geom_type[g]);kind={int(self.mj.mjtGeom.mjGEOM_CAPSULE):\'capsule\',int(self.mj.mjtGeom.mjGEOM_BOX):\'box\'}.get(typ)\n            if kind is None:raise ValueError(\'Unsupported fork collision geometry\')\n            r=support_radius(kind,self.m.geom_size[g],self.d.geom_xmat[g].reshape(3,3),[0,0,1])\n            gaps.append(float(self.d.geom_xpos[g,2]-top[2]-r))\n        return min(gaps)\n\n    def execute(self):\n        o=self.o;o.event(\'left_fork_stage_start\',same_data=True,start_simulation_time_s=float(self.d.time))\n        o.snapshot(\'fork_start\')\n        R=upright_frame(self.initial_R[:,1])\n        point=self.initial+self.initial_R@np.array([0.,-.020,0.])+np.array([0,0,.0008])\n        pre=point.copy();pre[2]=max(.14,point[2]+.08)\n        self.jaw(self.open_q,\'fork_open\')\n        p=self.plan(pre,R,\'fork_pregrasp\');self.move(p,\'fork_pregrasp\');self.hold(.25,\'fork_pregrasp_hold\')\n        o.snapshot(\'fork_pregrasp\')\n        p=self.plan(point,R,\'fork_lower\',True);self.move(p,\'fork_lower\',True);self.hold(.25,\'fork_approach_hold\',True)\n        if np.linalg.norm((self.d.xpos[self.fork]-self.initial)[:2])>.010:raise RuntimeError(\'Fork moved too far before closure\')\n        self.grasp()\n        lifted=self.d.xpos[self.fork].copy()+[0,0,.040]\n        p=self.plan(lifted,self.carry_R,\'fork_lift\',True,True);self.move(p,\'fork_lift\',True);self.hold(.75,\'fork_lift_hold\',True)\n        lift=self.carry_metrics(\'fork_lift_hold\');o.report[\'fork_lift_measurements\']=lift\n        if lift[\'minimum_rise_m\']<.025 or lift[\'two_jaw_contact_fraction\']<.8 or lift[\'maximum_table_support_n\']>=base.CONTACT_MIN_N:\n            raise RuntimeError(\'Fork lift not verified\')\n        o.report[\'fork_lift_verified\']=True;o.snapshot(\'fork_lifted\')\n        goal=self.d.xpos[self.fork].copy();goal[:2]=self.target[:2]\n        p=self.plan(goal,self.carry_R,\'fork_transfer\',True,True);self.move(p,\'fork_transfer\',True);self.hold(.5,\'fork_transfer_hold\',True)\n        trans=self.carry_metrics(\'fork_transfer_hold\');o.report[\'fork_transfer_measurements\']=trans\n        if trans[\'minimum_rise_m\']<.025 or trans[\'two_jaw_contact_fraction\']<.8 or trans[\'maximum_table_support_n\']>=base.CONTACT_MIN_N or trans[\'maximum_horizontal_destination_error_m\']>FORK_TOL:\n            raise RuntimeError(\'Fork destination carry failed; no set-down executed\')\n        o.report[\'fork_transfer_verified\']=True;o.snapshot(\'fork_transferred\')\n        clearance=self.bottom_clearance()\n        goal=base.measured_lowering_target(self.d.xpos[self.fork],[0,0,1],clearance)\n        goal[:2]=self.target[:2]\n        p=self.plan(goal,self.carry_R,\'fork_set_down\',True,True);self.move(p,\'fork_set_down\',True);self.hold(.5,\'fork_support_hold\',True)\n        support=float(np.mean([r[\'fork_table_normal_n\']>=base.CONTACT_MIN_N for r in self.stable(\'fork_support_hold\')]))\n        o.report[\'fork_support_fraction\']=support\n        if support<.8:raise RuntimeError(\'Fork table support missing before release\')\n        self.jaw(self.open_q,\'fork_release\',True);o.snapshot(\'fork_released\')\n        retreat=self.point(self.d).copy();retreat[2]=max(.14,retreat[2]+.05)\n        p=self.plan(retreat,R,\'fork_retreat\',True);self.move(p,\'fork_retreat\',True);self.hold(.75,\'fork_final_settle\')\n        rows=self.stable(\'fork_final_settle\',.5)\n        metrics=fork_placement_metrics(rows,self.initial,self.target,self.initial_R)\n        o.report[\'fork_placement_measurements\']=metrics\n        o.report[\'preservation_during_fork\']=dict(\n            maximum_cup_displacement_m=max(r[\'cup_displacement_during_fork_m\'] for r in o.rows[self.start_row:]),\n            maximum_plate_displacement_m=max(r[\'plate_displacement_during_fork_m\'] for r in o.rows[self.start_row:]))\n        if not metrics[\'placement_verified\']:raise RuntimeError(\'Fork destination release/orientation check failed\')\n        o.report[\'fork_placement_verified\']=True\n\n\ndef main(config):\n    trial=CombinedTrial(config)\n    try:trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.report[\'completed\']=False;trial.report[\'sequence_test_passed\']=False\n        trial.event(\'stopped\',reason=trial.report[\'stops\'][-1])\n        (trial.out/\'failure_traceback.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n        if trial.renderer is not None:trial.snapshot(\'stopped\')\n    finally:trial.finish()\n    print(\'SEQUENCE RESULT \'+json.dumps({k:trial.report.get(k) for k in (\'completed\',\'sequence_test_passed\',\'cup_stage_passed\',\n        \'fork_grasp_verified\',\'fork_lift_verified\',\'fork_placement_verified\',\'cup_preserved_after_fork\',\'stops\',\'report_file\')},allow_nan=False),flush=True)\n    return 0 if trial.report[\'sequence_test_passed\'] else 2\n\n\nif __name__==\'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_sequence.py': '"""Offline math, contact rules and data-preservation tests. Not MuJoCo physics."""\nimport unittest\nfrom pathlib import Path\nfrom types import SimpleNamespace as NS\nfrom unittest.mock import patch\nimport ast\nimport json\nimport math\nimport tempfile\nimport numpy as np\nimport two_object_worker as w\n\nclass Geometry(unittest.TestCase):\n    def test_identity_log(self): np.testing.assert_allclose(w.rotation_vector(np.eye(3)),0,atol=1e-12)\n    def test_identity_exp(self): np.testing.assert_allclose(w.rotation_exp(np.zeros(3)),np.eye(3),atol=1e-12)\n    def test_log_exp(self):\n        for v in ([.1,.2,.3],[-.8,.4,.2],[0,0,3.0]):\n            np.testing.assert_allclose(w.rotation_vector(w.rotation_exp(v)),v,atol=1e-9)\n    def test_pi_log(self):\n        R=w.rotation_exp(np.array([1,2,3])/np.sqrt(14)*math.pi)\n        np.testing.assert_allclose(w.rotation_exp(w.rotation_vector(R)),R,atol=1e-6)\n    def test_invalid_rotation(self):\n        with self.assertRaises(ValueError): w.rotation_vector(np.zeros((3,3)))\n    def test_invalid_exp(self):\n        with self.assertRaises(ValueError): w.rotation_exp([0,float(\'nan\'),0])\n    def test_upright(self):np.testing.assert_allclose(w.upright_frame([0,1,0]),np.eye(3))\n    def test_upright_projected(self):\n        R=w.upright_frame([.5,1,.1]);np.testing.assert_allclose(R[:,2],[0,0,1]);self.assertAlmostEqual(np.linalg.det(R),1)\n    def test_zero_direction(self):\n        with self.assertRaises(ValueError):w.upright_frame([0,0,1])\n    def test_capsule_vertical(self):self.assertAlmostEqual(w.support_radius(\'capsule\',[.004,.0325,0],np.eye(3),[0,0,1]),.0365)\n    def test_capsule_horizontal(self):self.assertAlmostEqual(w.support_radius(\'capsule\',[.004,.0325,0],w.rotation_exp([math.pi/2,0,0]),[0,0,1]),.004)\n    def test_box_support(self):self.assertAlmostEqual(w.support_radius(\'box\',[.012,.006,.003],np.eye(3),[0,0,1]),.003)\n    def test_invalid_capsule(self):\n        with self.assertRaises(ValueError):w.support_radius(\'capsule\',[-.1,.2,0],np.eye(3),[0,0,1])\n    def test_ik_five_dim_feasible(self):\n        def fk(q):return q[:3],w.rotation_exp([0,q[3],q[4]]),np.c_[np.eye(3),np.zeros((3,2))],np.c_[np.zeros((3,3)),[0,1,0],[0,0,1]]\n        q,d=w.solve_frame(fk,[.01,.02,.03],np.eye(3),np.zeros(5),-np.ones(5),np.ones(5),lambda:None)\n        self.assertTrue(d[\'converged\']);np.testing.assert_allclose(q[:3],[.01,.02,.03],atol=.0005)\n    def test_ik_impossible_not_marked_pass(self):\n        def fk(q):return np.zeros(3),np.eye(3),np.zeros((3,5)),np.zeros((3,5))\n        _,d=w.solve_frame(fk,[.1,0,0],np.eye(3),np.zeros(5),-np.ones(5),np.ones(5),lambda:None)\n        self.assertFalse(d[\'converged\'])\n\nclass Metrics(unittest.TestCase):\n    def rows(self,x=-.135,normal=.2,fixed=0,angle=0):\n        return [dict(fork_x=x,fork_y=.1,fork_z=.034,fork_table_normal_n=normal,\n                     fork_fixed_normal_n=fixed,fork_moving_normal_n=0,fork_orientation_error_deg=angle)]\n    def metrics(self,rows):return w.fork_placement_metrics(rows,[-.11,.1,.034],[-.135,.1,.034],np.eye(3))\n    def test_good(self):self.assertTrue(self.metrics(self.rows())[\'placement_verified\'])\n    def test_old_position_fails(self):self.assertFalse(self.metrics(self.rows(x=-.11))[\'placement_verified\'])\n    def test_no_support_fails(self):self.assertFalse(self.metrics(self.rows(normal=0))[\'placement_verified\'])\n    def test_still_held_fails(self):self.assertFalse(self.metrics(self.rows(fixed=.1))[\'placement_verified\'])\n    def test_rotated_fails(self):self.assertFalse(self.metrics(self.rows(angle=25))[\'placement_verified\'])\n    def test_empty_fails(self):\n        with self.assertRaises(ValueError):self.metrics([])\n    def test_nan_fails(self):\n        with self.assertRaises(ValueError):self.metrics(self.rows(x=float(\'nan\')))\n    def test_changed_destination_fails(self):\n        with self.assertRaises(ValueError):w.fork_placement_metrics(self.rows(),[-.11,.1,.034],[-.11,.1,.034],np.eye(3))\n\nclass ContactRules(unittest.TestCase):\n    def stage(self,a,b,dist=-.0001,force=.1):\n        t=w.ForkStage.__new__(w.ForkStage);t.m=NS();t.geoms={10};t.jaws={20:\'fixed\',21:\'moving\'}\n        t.o=NS(table=40,cup_geoms={30},geom_group={22:\'fixed\'},robot_geoms={20,21,22,23},name=lambda k,i:str(i))\n        t.mj=NS(mjtObj=NS(mjOBJ_GEOM=1),mj_contactForce=lambda m,d,i,v:v.__setitem__(0,force))\n        return t,NS(ncon=1,contact=[NS(geom1=a,geom2=b,dist=dist,efc_address=0)])\n    def test_intended_contact_allowed(self):\n        t,d=self.stage(10,20);ff,_,_,_,issues=t.contact_state(d,True);self.assertEqual(issues,[]);self.assertAlmostEqual(ff[\'fixed\'],.1)\n    def test_forbidden_phase(self):\n        t,d=self.stage(10,20);self.assertTrue(t.contact_state(d,False)[-1])\n    def test_other_arm_not_allowed(self):\n        t,d=self.stage(10,22);self.assertTrue(t.contact_state(d,True)[-1])\n    def test_fork_table_supported(self):\n        t,d=self.stage(10,40);ff,s,_,_,issues=t.contact_state(d,False);self.assertEqual(issues,[]);self.assertEqual(s,.1)\n    def test_deep_intended_collision_not_ignored(self):\n        t,d=self.stage(10,20,dist=-.004);self.assertTrue(t.contact_state(d,True,planning=True)[-1])\n    def test_high_force_not_ignored(self):\n        t,d=self.stage(10,20,force=26);self.assertTrue(t.contact_state(d,True)[-1])\n    def test_cup_support_independent(self):\n        t,d=self.stage(30,40);_,fs,_,cs,issues=t.contact_state(d,True);self.assertEqual(fs,0);self.assertEqual(cs,.1);self.assertEqual(issues,[])\n    def test_robot_table_not_allowed(self):\n        t,d=self.stage(20,40);self.assertTrue(t.contact_state(d,True)[-1])\n\nclass Evidence(unittest.TestCase):\n    def test_execution_no_teleport(self):\n        tree=ast.parse(Path(w.__file__).read_text())\n        for node in ast.walk(tree):\n            if isinstance(node,(ast.Assign,ast.AugAssign,ast.AnnAssign)):\n                targets=node.targets if isinstance(node,ast.Assign) else [node.target]\n                for t in targets:\n                    text=ast.unparse(t)\n                    self.assertNotIn(\'d.qpos\',text);self.assertNotIn(\'data.qpos\',text)\n                    self.assertNotIn(\'xfrc_applied\',text);self.assertNotIn(\'qfrc_applied\',text)\n    def test_combined_continuous_data(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            s=Path(tmp)/\'source.xml\';s.write_text(\'original\')\n            cfg=dict(scene=str(s),scene_sha256=w.base.digest(s),output=tmp,record_data=False)\n            t=w.CombinedTrial(cfg);data=NS(time=34.,xpos=np.zeros((3,3)));t.data=data\n            t.snapshot=lambda label:None;t.cup_initial=np.array([0,0,0]);t.target=np.array([.025,0,0])\n            row=dict(cup_x=.025,cup_y=0,cup_z=0,table_normal_n=.1,fixed_normal_n=0,moving_normal_n=0)\n            def cup(self):self.report.update(transfer_test_passed=True,completed=True,grasp_test_passed=True)\n            class Fork:\n                def __init__(self,owner):self.owner=owner;assert owner.data is data\n                def execute(self):self.owner.data.time+=5;self.owner.report[\'fork_placement_verified\']=True\n                def stable(self,*args):return [row]\n            with patch.object(w.cup07c.CupOrientationTrial,\'execute\',cup),patch.object(w,\'ForkStage\',Fork):t.execute()\n            self.assertIs(t.data,data);self.assertEqual(t.data.time,39);self.assertTrue(t.report[\'sequence_test_passed\'])\n    def test_failure_remains_partial(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            s=Path(tmp)/\'source.xml\';s.write_text(\'original\')\n            cfg=dict(scene=str(s),scene_sha256=w.base.digest(s),output=tmp,record_data=False)\n            t=w.CombinedTrial(cfg);t.data=NS(time=1.);t.dt=.005;t.mapping=[]\n            t.report[\'stops\']=[\'test\'];t.report[\'cup_stage_passed\']=True\n            t.finish();r=json.loads((Path(tmp)/\'sequence_report.json\').read_text())\n            self.assertFalse(r[\'sequence_test_passed\']);self.assertFalse(r[\'completed\']);self.assertEqual(s.read_text(),\'original\')\n    def test_writes_separate_recorded_labels(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            s=Path(tmp)/\'source.xml\';s.write_text(\'original\')\n            cfg=dict(scene=str(s),scene_sha256=w.base.digest(s),output=tmp,record_data=True)\n            t=w.CombinedTrial(cfg);t.data=NS(time=.005);t.dt=.005;t.mapping=[dict(name=\'test\')]\n            t.rows=[{\'test_command_rad\':.2}];t.steps=1\n            t.video_rows=[dict(step=0,images={\'top\':\'a\',\'left_wrist_cam\':\'b\',\'right_wrist_cam\':\'c\'})]\n            t.finish();obs=json.loads((Path(tmp)/\'expert_observations.jsonl\').read_text())\n            self.assertEqual(obs[\'action_chunk\'],[[.2]])\n            self.assertEqual(obs[\'episode_outcome\'],\'failed_or_partial_engineering_sequence\')\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromModule(__import__(__name__))\n    result=unittest.TextTestRunner(verbosity=1).run(suite)\n    summary=dict(kind=\'OFFLINE_SYNTHETIC_NOT_ROBOT_PHYSICS\',tests_run=result.testsRun,failures=len(result.failures),\n                 errors=len(result.errors),skipped=len(result.skipped),passed=result.wasSuccessful())\n    Path(\'sequence_test_summary.json\').write_text(json.dumps(summary,indent=2))\n    print(json.dumps(summary));raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'sequence_08b_worker.py': '"""08B expert-demonstration patch, not a learned-policy submission.\n\nKeep the original 08 cup stage, physical execution, recorder, and all contact/\nforce/range/final-goal checks. Replace unreachable Cartesian pose interpolation\nfor free approaches with endpoint-verified joint-space connections. Select a\nreachable pregrasp height while retaining at least 30 mm tip-reference clearance.\nDuring carrying constrain fork origin and up-axis (5 task DoF), and keep the\nexisting tilt and final full-orientation checks. No physical grasp success is\nassumed. Only scratch state is assigned for planning.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport sys\nimport traceback\nimport numpy as np\nimport base06b as base\nimport two_object_worker as original\nimport cup_orientation_worker as cup07c\n\nREVISION = \'08b-reachable-endpoints-fork-up-axis-v1\'\nMIN_APPROACH_CLEARANCE_M = .030\nPLANNING_TILT_DEG = 25.\nPLANNING_ROTATION_DEG = 15.\nJOINT_SAMPLE_STEP_RAD = .020\n\n\ndef candidate_heights(requested_z, reference_z):\n    """A bounded, declared approach-height search, not a larger joint range."""\n    if not all(math.isfinite(float(x)) for x in (requested_z, reference_z)):\n        raise ValueError(\'Finite heights required\')\n    minimum = float(reference_z) + MIN_APPROACH_CLEARANCE_M\n    if requested_z < minimum - 1e-10:\n        raise ValueError(\'Requested approach is below the 30 mm reference clearance\')\n    candidates = [float(requested_z)]\n    candidates += [z for z in (.12, .10, .08, .07) if minimum <= z < requested_z - 1e-9]\n    if not any(abs(z-minimum) < 1e-9 for z in candidates):\n        candidates.append(minimum)\n    return sorted(candidates, reverse=True)\n\n\ndef sample_connection(start, end):\n    start, end = np.asarray(start, dtype=float), np.asarray(end, dtype=float)\n    if start.shape != (5,) or end.shape != (5,) or not np.isfinite([start, end]).all():\n        raise ValueError(\'Two finite five-joint endpoints required\')\n    steps = max(1, int(math.ceil(float(np.max(np.abs(end-start))) / JOINT_SAMPLE_STEP_RAD)))\n    return np.array([start + a*(end-start) for a in np.linspace(0., 1., steps+1)])\n\n\nclass ForkStage08B(original.ForkStage):\n    def __init__(self, owner):\n        super().__init__(owner)\n        if len(self.arm) != 5:\n            raise ValueError(\'Fork pose planner requires the inspected five arm joints\')\n        owner.report[\'fork_planner_revision\'] = REVISION\n        owner.report[\'fork_planning_changes\'] = dict(\n            free_approach=\'verify attainable full-frame endpoint, then sample a joint-space path\',\n            min_pregrasp_reference_clearance_m=MIN_APPROACH_CLEARANCE_M,\n            carrying=\'fork origin plus fork up-axis; rotation about up-axis not an equality constraint\',\n            final_full_orientation_limit_deg=15.,\n            pose_tolerance_m=base.POSITION_TOL,axis_tolerance_deg=base.AXIS_TOL_DEG,\n            cup_execution_contact_force_and_joint_guards=\'unchanged\',\n            purpose=\'expert demonstration generation only; NOT the final VLA controller\')\n        owner.report[\'fork_pregrasp_candidates\'] = []\n        self.approach_reference = (self.initial + self.initial_R @ np.array([0., -.020, 0.])\n                                   + np.array([0., 0., .0008]))\n\n    def _evaluator(self, carry):\n        o, m, d, mj, s = self.o, self.m, self.d, self.mj, self.scratch\n        if s is d or np.shares_memory(s.qpos, d.qpos):\n            raise RuntimeError(\'Scratch/execution overlap\')\n        s.qpos[:] = d.qpos  # Planning copy, never physical state.\n        reference = None\n        if carry:\n            if not o.report.get(\'fork_grasp_verified\'):\n                raise RuntimeError(\'Carry requires a measured fork grasp\')\n            ff, _, _, _, issues = self.contact_state(d, True)\n            if issues or min(ff.values()) < base.CONTACT_MIN_N:\n                raise RuntimeError(\'Fork contact is invalid at carry-plan entry\')\n            reference = base.relative_pose(d.xpos[self.gripper], d.xmat[self.gripper].reshape(3,3),\n                                           d.xpos[self.fork], d.xmat[self.fork].reshape(3,3))\n        jp, jr = np.zeros((3, m.nv)), np.zeros((3, m.nv))\n        def evaluate(q):\n            o.budget()\n            s.qpos[self.qa] = q\n            if reference is not None:\n                mj.mj_kinematics(m, s)\n                p, R = base.predicted_pose(s.xpos[self.gripper], s.xmat[self.gripper].reshape(3,3), *reference)\n                quat = np.empty(4)\n                mj.mju_mat2Quat(quat, np.ascontiguousarray(R).reshape(9))\n                norm = float(np.linalg.norm(quat))\n                if not np.isfinite(quat).all() or norm < 1e-10:\n                    raise RuntimeError(\'Invalid predicted fork quaternion\')\n                s.qpos[self.qadr:self.qadr+3] = p\n                s.qpos[self.qadr+3:self.qadr+7] = quat / norm\n            mj.mj_forward(m, s)\n            p = s.xpos[self.fork].copy() if carry else self.point(s)\n            R = s.xmat[self.fork if carry else self.gripper].reshape(3,3).copy()\n            mj.mj_jac(m, s, jp, jr, p, self.gripper)\n            return p, R, jp[:,self.va].copy(), jr[:,self.va].copy()\n        return evaluate\n\n    def _issues(self, allow, carry):\n        issues = list(self.contact_state(self.scratch, allow, planning=True)[-1])\n        if carry:\n            R = self.scratch.xmat[self.fork].reshape(3,3)\n            if cup07c.tilt_deg(R[:,2]) > PLANNING_TILT_DEG:\n                issues.append(\'predicted fork tilt above 25 degrees\')\n            angle = math.degrees(float(np.linalg.norm(original.rotation_vector(R @ self.initial_R.T))))\n            if angle > PLANNING_ROTATION_DEG:\n                issues.append(\'predicted full fork orientation above existing 15 degree final bound\')\n        return issues\n\n    def _record_plan(self, plan, target, Rtarget, label, carry, diagnostics, planning_mode):\n        plan.update(target_m=np.asarray(target).tolist(),target_rotation=np.asarray(Rtarget).tolist(),\n                    carry_prediction_only=bool(carry),diagnostics=diagnostics,planning_mode=planning_mode,\n                    planner_revision=REVISION)\n        base.write_json(self.o.out/f\'plan_{label}.json\', plan)\n        self.o.event(\'fork_plan_ready\',phase=label,duration_s=plan[\'duration_s\'],mode=planning_mode)\n        return plan\n\n    def _free_endpoint_plan(self, target, Rtarget, label, allow):\n        o = self.o\n        evaluate = self._evaluator(False)\n        start = np.array([self.d.ctrl[r[\'cid\']] for r in self.arm])\n        p0, _, _, _ = evaluate(start)\n        # Solve an attainable low grasp endpoint first; use it only as a solver seed.\n        # No pose is applied to physical execution during this check.\n        ref_z = self.approach_reference[2] if label == \'fork_pregrasp\' else p0[2]\n        low_point = np.array(target,copy=True); low_point[2] = ref_z\n        low_q, low_diag = original.solve_frame(evaluate, low_point, Rtarget, start,\n                                               self.lo, self.hi, o.budget)\n        seed_list = [low_q, start] if low_diag.get(\'converged\') else [start]\n        attempts = []\n        for height in candidate_heights(float(target[2]), float(ref_z)):\n            endpoint = np.array(target, copy=True); endpoint[2] = height\n            for seed_index, seed in enumerate(seed_list):\n                q, diag = original.solve_frame(evaluate, endpoint, Rtarget, seed,\n                                                self.lo, self.hi, o.budget)\n                record = dict(height_m=height,seed_index=seed_index,**diag)\n                if not diag.get(\'converged\'):\n                    record[\'accepted\'] = False; attempts.append(record); continue\n                # Check actual joint-space configurations; do not interpolate six independent pose coordinates.\n                path = sample_connection(start,q)\n                issue = []\n                for sample in path:\n                    evaluate(sample)\n                    issue = self._issues(allow,False)\n                    if issue: break\n                record[\'sampled_configurations\'] = len(path)\n                record[\'issues\'] = issue[:6]; record[\'accepted\'] = not bool(issue)\n                attempts.append(record)\n                if issue: continue\n                o.report[label+\'_selection\'] = dict(\n                    requested_target_m=np.asarray(target).tolist(),selected_target_m=endpoint.tolist(),\n                    reference_clearance_m=height-ref_z,attempts=attempts,\n                    source=\'reachability of current model in scratch data; not a camera estimate\',\n                    meaning=\'free-space path changed; grasp endpoint and final-goal tests are unchanged\')\n                base.write_json(o.out/f\'{label}_selection.json\',o.report[label+\'_selection\'])\n                o.event(\'fork_endpoint_selected\',phase=label,height_m=height,reference_clearance_m=height-ref_z)\n                return self._record_plan(base.parameterize(path,o.dt),endpoint,Rtarget,label,False,\n                                         [diag],\'reachable_endpoint_joint_connection\')\n        failure = dict(requested_target_m=np.asarray(target).tolist(),reference_z=float(ref_z),\n                       minimum_clearance_m=MIN_APPROACH_CLEARANCE_M,attempts=attempts)\n        o.report[\'fork_plan_failure\'] = dict(phase=label,**failure)\n        base.write_json(o.out/f\'plan_{label}_failure.json\', failure)\n        raise RuntimeError(f\'{label}: no endpoint/path passed the bounded height search; see plan failure JSON\')\n\n    def plan(self, target, Rtarget, label, allow=False, carry=False):\n        target, Rtarget = base.checked_pose(target,Rtarget)\n        self.o.event(\'fork_plan_start\',phase=label,revision=REVISION)\n        if not carry and label in (\'fork_pregrasp\',\'fork_retreat\'):\n            return self._free_endpoint_plan(target,Rtarget,label,allow)\n        if not carry:\n            # Lowering uses the original full-frame planner after reaching a compatible pregrasp pose.\n            return super().plan(target,Rtarget,label,allow,carry)\n        if label not in (\'fork_lift\',\'fork_transfer\',\'fork_set_down\'):\n            raise ValueError(\'Unsupported carried-fork phase\')\n        evaluate = self._evaluator(True)\n        seed = np.array([self.d.ctrl[r[\'cid\']] for r in self.arm])\n        p0, R0, _, _ = evaluate(seed)\n        a0, desired = R0[:,2].copy(), Rtarget[:,2].copy()\n        if cup07c.tilt_deg(a0) > PLANNING_TILT_DEG:\n            raise RuntimeError(\'Initial carried fork exceeds planning tilt bound\')\n        def axis_eval(q):\n            p, R, jp, jr = evaluate(q)\n            a = R[:,2]\n            return p, a.copy(), jp, -base.skew(a) @ jr\n        path, diags = [seed.copy()], []\n        for k in range(1,41):\n            alpha = k/40\n            axis = base.normalize((1-alpha)*a0 + alpha*desired)\n            q, diag = base.solve_pose(axis_eval,p0+alpha*(target-p0),axis,seed,\n                                      self.lo,self.hi,self.o.budget)\n            diags.append(diag)\n            if not diag.get(\'converged\'):\n                fail = dict(phase=label,waypoint=k,diagnostics=diags,mode=\'fork_origin_plus_up_axis\')\n                self.o.report[\'fork_plan_failure\'] = fail\n                base.write_json(self.o.out/f\'plan_{label}_failure.json\', fail)\n                raise RuntimeError(f\'{label}: five-task-DoF fork solve failed at waypoint {k}: {diag}\')\n            for sample in sample_connection(seed,q):\n                evaluate(sample); issues = self._issues(allow,True)\n                if issues:\n                    self.o.report[\'fork_plan_failure\'] = dict(phase=label,waypoint=k,issues=issues[:6])\n                    raise RuntimeError(f\'{label}: sampled path rejected: {issues[:3]}\')\n            path.append(q.copy());seed=q\n        return self._record_plan(base.parameterize(path,self.o.dt),target,Rtarget,label,True,\n                                 diags,\'fork_origin_plus_up_axis_final_orientation_guarded\')\n\n\nclass CombinedTrial08B(original.CombinedTrial):\n    def __init__(self,config):\n        super().__init__(config)\n        self.report.update(task_revision=REVISION,cup_preservation_evaluated=False,\n                           cup_preserved_after_fork=None,fork_placement_evaluated=False)\n        self.report[\'notes\'].append(\'08B free approach endpoint/height selection and five-task-DoF carrying, for expert data only.\')\n\n    def execute(self):\n        # Same cup implementation and same physical state; no reset between objects.\n        cup07c.CupOrientationTrial.execute(self)\n        self.report[\'cup_stage_passed\'] = bool(self.report.get(\'transfer_test_passed\'))\n        self.report[\'cup_stage_end_time_s\'] = float(self.data.time)\n        self.report[\'cup_stage_placement\'] = dict(self.report.get(\'placement_measurements\',{}))\n        if not self.report[\'cup_stage_passed\']:\n            raise RuntimeError(\'Cup must pass before left-arm motion\')\n        self.snapshot(\'cup_placed\')\n        self.report[\'completed\']=False;self.report[\'grasp_test_passed\']=False\n        self.fork_stage=ForkStage08B(self)\n        self.fork_stage.execute()\n        self.report[\'fork_placement_evaluated\']=True\n        rows=self.fork_stage.stable(\'fork_final_settle\',.5)\n        cup=original.cup_metrics.placement_metrics(rows,self.cup_initial,self.target)\n        self.report[\'cup_final_placement\']=cup\n        self.report[\'cup_preservation_evaluated\']=True\n        self.report[\'cup_preserved_after_fork\']=bool(cup[\'placement_verified\'])\n        if not self.report[\'cup_preserved_after_fork\']:\n            raise RuntimeError(\'Cup destination failed after fork placement\')\n        self.report[\'sequence_test_passed\']=True;self.report[\'completed\']=True\n        self.report[\'grasp_test_passed\']=True;self.snapshot(\'sequence_final\')\n\n    def finish(self):\n        if self.fork_stage is not None:\n            rows=self.rows[self.fork_stage.start_row:]\n            self.report[\'fork_execution_rows\']=len(rows)\n            self.report[\'fork_placement_evaluated\']=bool(self.report.get(\'fork_placement_measurements\'))\n            self.report[\'preservation_observed_so_far\']={\n                key:max((float(row[key]) for row in rows if key in row),default=None)\n                for key in (\'cup_displacement_during_fork_m\',\'plate_displacement_during_fork_m\')}\n        super().finish()\n\n\ndef main(config):\n    trial=CombinedTrial08B(config)\n    try:trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.report[\'completed\']=False;trial.report[\'sequence_test_passed\']=False\n        trial.event(\'stopped\',reason=trial.report[\'stops\'][-1])\n        (trial.out/\'failure_traceback.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n        if trial.renderer is not None:trial.snapshot(\'stopped\')\n    finally:trial.finish()\n    print(\'SEQUENCE RESULT \'+json.dumps({k:trial.report.get(k) for k in (\n        \'sequence_test_passed\',\'cup_stage_passed\',\'fork_grasp_verified\',\'fork_lift_verified\',\n        \'fork_placement_verified\',\'cup_preservation_evaluated\',\'cup_preserved_after_fork\',\n        \'stops\',\'report_file\')},allow_nan=False),flush=True)\n    return 0 if trial.report[\'sequence_test_passed\'] else 2\n\n\nif __name__==\'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_sequence_08b.py': '"""08B numerical/control-preservation tests; no robot physics execution."""\nimport ast\nimport json\nimport math\nfrom pathlib import Path\nimport tempfile\nimport unittest\nfrom types import SimpleNamespace as NS\nfrom unittest.mock import patch\nimport numpy as np\nimport sequence_08b_worker as w\n\nclass HeightAndPathTests(unittest.TestCase):\n    def test_height_order_and_minimum(self):\n        hs=w.candidate_heights(.14,.0348)\n        self.assertEqual(hs,sorted(hs,reverse=True));self.assertEqual(hs[0],.14)\n        self.assertTrue(all(z>=.0648-1e-10 for z in hs));self.assertAlmostEqual(hs[-1],.0648)\n    def test_high_reference(self):\n        self.assertEqual(w.candidate_heights(.2,.13),[.2,.16])\n    def test_low_requested_rejected(self):\n        with self.assertRaises(ValueError):w.candidate_heights(.04,.0348)\n    def test_nan_rejected(self):\n        with self.assertRaises(ValueError):w.candidate_heights(float(\'nan\'),.03)\n    def test_no_duplicate_height(self):\n        hs=w.candidate_heights(.10,.04);self.assertEqual(len(hs),len(set(hs)))\n    def test_joint_samples(self):\n        a=np.zeros(5);b=np.array([.3,-.22,.01,.51,-.7]);p=w.sample_connection(a,b)\n        np.testing.assert_allclose(p[0],a);np.testing.assert_allclose(p[-1],b)\n        self.assertLessEqual(np.abs(np.diff(p,axis=0)).max(),.02+1e-12)\n    def test_joint_samples_do_not_mutate(self):\n        a=np.zeros(5);b=np.ones(5);w.sample_connection(a,b)\n        np.testing.assert_array_equal(a,0);np.testing.assert_array_equal(b,1)\n    def test_bad_shape(self):\n        with self.assertRaises(ValueError):w.sample_connection(np.zeros(6),np.ones(6))\n    def test_bad_value(self):\n        with self.assertRaises(ValueError):w.sample_connection(np.zeros(5),[0,0,0,0,float(\'nan\')])\n\nclass TaskTests(unittest.TestCase):\n    def test_axis_removes_only_yaw_equality(self):\n        R=w.original.rotation_exp([0,0,.20])\n        np.testing.assert_allclose(R[:,2],[0,0,1])\n        self.assertGreater(np.linalg.norm(w.original.rotation_vector(R)),math.radians(2))\n    def test_five_task_dof_solve(self):\n        def evaluate(q):\n            # Three translations plus two rotations: a genuine five-dimensional task.\n            Rx=w.original.rotation_exp([q[3],0,0]);Ry=w.original.rotation_exp([0,q[4],0])\n            R=Rx@Ry;a=R[:,2];jr=np.c_[np.zeros((3,3)),[1,0,0],Rx@np.array([0,1,0])]\n            jp=np.c_[np.eye(3),np.zeros((3,2))]\n            return q[:3].copy(),a,jp,-w.base.skew(a)@jr\n        axis=w.original.rotation_exp([.08,0,0])@w.original.rotation_exp([0,-.06,0])@np.array([0,0,1])\n        q,diag=w.base.solve_pose(evaluate,[.01,.02,.03],axis,np.zeros(5),-np.ones(5),np.ones(5),lambda:None)\n        self.assertTrue(diag[\'converged\'])\n    def test_final_orientation_still_rejected(self):\n        r=dict(fork_x=-.135,fork_y=.1,fork_z=.034,fork_table_normal_n=.1,\n               fork_fixed_normal_n=0,fork_moving_normal_n=0,fork_orientation_error_deg=20)\n        out=w.original.fork_placement_metrics([r],[-.11,.1,.034],[-.135,.1,.034],np.eye(3))\n        self.assertFalse(out[\'placement_verified\'])\n    def test_runtime_guards_inherited(self):\n        for name in (\'tick\',\'contact_state\',\'jaw\',\'hold\',\'grasp\',\'execute\',\'carry_metrics\'):\n            self.assertIs(getattr(w.ForkStage08B,name),getattr(w.original.ForkStage,name))\n    def test_cup_methods_inherited(self):\n        for name in (\'tick\',\'load\',\'plan\',\'plan_object\',\'move_jaw\',\'close_on_cup\'):\n            if hasattr(w.original.CombinedTrial,name):\n                self.assertIs(getattr(w.CombinedTrial08B,name),getattr(w.original.CombinedTrial,name))\n    def test_placement_constants_unchanged(self):\n        self.assertEqual(w.original.FORK_TOL,.008);self.assertEqual(w.original.CUP_PRESERVE_TOL,.008)\n        self.assertEqual(w.base.GRIPPER_TORQUE_CAP,.25);self.assertEqual(w.base.POSITION_TOL,.0005)\n        self.assertEqual(w.base.AXIS_TOL_DEG,2.0)\n    def test_unreachable_axis_not_marked_success(self):\n        def bad(q):return np.zeros(3),np.array([0,0,1.]),np.zeros((3,5)),np.zeros((3,5))\n        _,diag=w.base.solve_pose(bad,[1,0,0],[0,0,1],np.zeros(5),-np.ones(5),np.ones(5),lambda:None)\n        self.assertFalse(diag[\'converged\'])\n\nclass PlannerTests(unittest.TestCase):\n    def stage(self,tmp,blocked=False):\n        t=w.ForkStage08B.__new__(w.ForkStage08B)\n        t.o=NS(out=Path(tmp),report={},dt=.005,budget=lambda:None,event=lambda *a,**k:None)\n        t.d=NS(ctrl=np.zeros(5));t.arm=[{\'cid\':i} for i in range(5)]\n        t.lo=-np.ones(5);t.hi=np.ones(5);t.approach_reference=np.array([.01,.02,.0348])\n        t._evaluator=lambda carry:lambda q:(q[:3].copy(),np.eye(3),np.c_[np.eye(3),np.zeros((3,2))],np.zeros((3,5)))\n        t._issues=lambda allow,carry:[\'blocked\'] if blocked else []\n        return t\n    def fake_solver(self,ev,target,R,seed,lo,hi,budget):\n        q=np.r_[target,[0,0]]\n        ok=bool(target[2]<=.071)\n        return q,dict(converged=ok,position_error_m=0. if ok else .02,rotation_error_deg=0.,iterations=2)\n    def test_height_fallback_keeps_endpoint(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            t=self.stage(tmp)\n            with patch.object(w.original,\'solve_frame\',self.fake_solver):\n                p=t._free_endpoint_plan(np.array([.01,.02,.14]),np.eye(3),\'fork_pregrasp\',False)\n            self.assertAlmostEqual(p[\'target_m\'][2],.07)\n            self.assertTrue(t.o.report[\'fork_pregrasp_selection\'][\'reference_clearance_m\']>=.03)\n            np.testing.assert_array_equal(t.d.ctrl,np.zeros(5))\n    def test_all_collisions_rejected(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            t=self.stage(tmp,True)\n            with patch.object(w.original,\'solve_frame\',self.fake_solver),self.assertRaises(RuntimeError):\n                t._free_endpoint_plan(np.array([.01,.02,.14]),np.eye(3),\'fork_pregrasp\',False)\n            self.assertTrue((Path(tmp)/\'plan_fork_pregrasp_failure.json\').is_file())\n    def test_no_endpoint_converged(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            t=self.stage(tmp)\n            def failed(*a,**kw):return np.zeros(5),dict(converged=False)\n            with patch.object(w.original,\'solve_frame\',failed),self.assertRaises(RuntimeError):\n                t._free_endpoint_plan(np.array([.01,.02,.14]),np.eye(3),\'fork_pregrasp\',False)\n    def test_planned_fork_tilt_rejected(self):\n        t=w.ForkStage08B.__new__(w.ForkStage08B);t.fork=0;t.initial_R=np.eye(3)\n        t.scratch=NS(xmat=np.array([w.original.rotation_exp([math.radians(28),0,0]).ravel()]))\n        t.contact_state=lambda *a,**k:({},0,{},0,[])\n        self.assertTrue(t._issues(True,True))\n    def test_planned_yaw_over_15_rejected(self):\n        t=w.ForkStage08B.__new__(w.ForkStage08B);t.fork=0;t.initial_R=np.eye(3)\n        t.scratch=NS(xmat=np.array([w.original.rotation_exp([0,0,math.radians(16)]).ravel()]))\n        t.contact_state=lambda *a,**k:({},0,{},0,[])\n        self.assertIn(\'predicted full fork orientation above existing 15 degree final bound\',t._issues(True,True))\n    def test_existing_contact_issue_retained(self):\n        t=w.ForkStage08B.__new__(w.ForkStage08B);t.scratch=NS()\n        t.contact_state=lambda *a,**k:({},0,{},0,[\'forbidden contact\'])\n        self.assertEqual(t._issues(False,False),[\'forbidden contact\'])\n\nclass EvidenceTests(unittest.TestCase):\n    def test_no_execution_position_or_force_assignment(self):\n        tree=ast.parse(Path(w.__file__).read_text())\n        for n in ast.walk(tree):\n            if isinstance(n,(ast.Assign,ast.AugAssign,ast.AnnAssign)):\n                for t in n.targets if isinstance(n,ast.Assign) else [n.target]:\n                    s=ast.unparse(t)\n                    for bad in (\'d.qpos\',\'data.qpos\',\'qfrc_applied\',\'xfrc_applied\'):\n                        self.assertNotIn(bad,s)\n    def test_preservation_unassessed_stays_none(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            p=Path(tmp)/\'source\';p.write_text(\'source\')\n            t=w.CombinedTrial08B(dict(scene=str(p),output=tmp,scene_sha256=w.base.digest(p),record_data=False))\n            self.assertIsNone(t.report[\'cup_preserved_after_fork\'])\n            self.assertFalse(t.report[\'cup_preservation_evaluated\'])\n            t.report[\'stops\']=[\'planned test failure\'];t.finish()\n            r=json.loads((Path(tmp)/\'sequence_report.json\').read_text())\n            self.assertFalse(r[\'sequence_test_passed\']);self.assertIsNone(r[\'cup_preserved_after_fork\'])\n    def test_same_data_and_cup_first(self):\n        with tempfile.TemporaryDirectory() as tmp:\n            p=Path(tmp)/\'source\';p.write_text(\'source\')\n            t=w.CombinedTrial08B(dict(scene=str(p),output=tmp,scene_sha256=w.base.digest(p),record_data=False))\n            d=NS(time=34.);t.data=d;t.cup_initial=np.zeros(3);t.target=np.array([.025,0,0]);t.snapshot=lambda label:None\n            order=[]\n            def cup(self):order.append(\'cup\');self.report[\'transfer_test_passed\']=True\n            class Fork:\n                def __init__(self,o):self.o=o;assert o.data is d\n                def execute(self):order.append(\'fork\');self.o.data.time+=2;self.o.report[\'fork_placement_verified\']=True\n                def stable(self,*a):return [dict(cup_x=.025,cup_y=0,cup_z=0,table_normal_n=.1,fixed_normal_n=0,moving_normal_n=0)]\n            with patch.object(w.cup07c.CupOrientationTrial,\'execute\',cup),patch.object(w,\'ForkStage08B\',Fork):t.execute()\n            self.assertEqual(order,[\'cup\',\'fork\']);self.assertIs(t.data,d)\n            self.assertTrue(t.report[\'sequence_test_passed\']);self.assertTrue(t.report[\'cup_preservation_evaluated\'])\n\nif __name__==\'__main__\':\n    result=unittest.TextTestRunner(verbosity=1).run(unittest.defaultTestLoader.loadTestsFromModule(__import__(__name__)))\n    summary=dict(kind=\'OFFLINE_SYNTHETIC_NOT_ROBOT_PHYSICS\',tests_run=result.testsRun,\n                 failures=len(result.failures),errors=len(result.errors),skipped=len(result.skipped),passed=result.wasSuccessful())\n    Path(\'sequence_08b_test_summary.json\').write_text(json.dumps(summary,indent=2))\n    print(json.dumps(summary));raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'sequence_08c_worker.py': '"""08C: bounded fork grasp-pose selection using active hand collision geometry.\n\nExpert-demonstration generator, not VLA inference. No contact masks, friction,\ngeometry, force limits, runtime guards or success tests are changed. All trial\ncandidates live only in scratch data. A geometric clearance check is not a grasp.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport sys\nimport traceback\nimport numpy as np\nimport base06b as base\nimport two_object_worker as original\nimport sequence_08b_worker as previous\nimport cup_orientation_worker as cup07c\n\nREVISION = \'08c-active-hand-table-clearance-v1\'\nTABLE_CLEARANCE_M = .0002  # Additional candidate-only margin; table contact is still forbidden.\nOPENING_CANDIDATES_RAD = (.22, .10, .0)\nPINCH_HEIGHT_FRACTIONS = (.20, .50, .75)\nMAX_CANDIDATES = 9\nCANDIDATE_BUDGET_S = 30.\n\n\ndef grasp_candidates(radius, low, high):\n    if not all(math.isfinite(float(v)) for v in (radius, low, high)) or not .003 <= radius <= .005 or low >= high:\n        raise ValueError(\'Reviewed 6–10 mm handle and finite joint bounds required\')\n    result = []\n    for opening in OPENING_CANDIDATES_RAD:\n        if not low + .01 + 1e-9 < opening < high - .01 - 1e-9:\n            continue\n        for fraction in PINCH_HEIGHT_FRACTIONS:\n            result.append(dict(opening_rad=opening, height_offset_m=fraction*radius,\n                               height_fraction_of_radius=fraction))\n    return result[:MAX_CANDIDATES]\n\n\ndef min_plane_coordinate(kind, size, position, rotation, plane_point, plane_normal, vertices=None):\n    """Minimum signed plane distance using actual compiled mesh vertices or primitives.\n\n    For a linear plane support query, the mesh and its convex hull share extrema.\n    No contact masks or shapes are modified. This infinite-plane test is conservative\n    around the finite table\'s edges; ordinary whole-scene contacts are also checked.\n    """\n    p, R = base.checked_pose(position, rotation)\n    t = np.asarray(plane_point, dtype=float)\n    s = np.asarray(size, dtype=float)\n    n = base.normalize(plane_normal)\n    if t.shape != (3,) or not np.isfinite(t).all() or s.shape != (3,) or not np.isfinite(s).all() or np.any(s < 0):\n        raise ValueError(\'Invalid geometry or plane\')\n    local = R.T @ n\n    center = float((p-t) @ n)\n    if kind == \'mesh\':\n        v = np.asarray(vertices, dtype=float)\n        if v.ndim != 2 or v.shape[1] != 3 or not len(v) or not np.isfinite(v).all():\n            raise ValueError(\'Finite compiled mesh vertices required; geom_size is not a substitute\')\n        return center + float(np.min(v @ local))\n    if kind == \'sphere\':\n        support = s[0]\n    elif kind == \'capsule\':\n        support = s[0] + s[1]*abs(local[2])\n    elif kind == \'box\':\n        support = float(np.abs(local) @ s)\n    elif kind == \'cylinder\':\n        support = s[0]*float(np.linalg.norm(local[:2])) + s[1]*abs(local[2])\n    elif kind == \'ellipsoid\':\n        support = float(np.linalg.norm(s*local))\n    else:\n        raise ValueError(\'Unsupported hand geometry: \' + str(kind))\n    return center - float(support)\n\n\ndef mask_compatible(a_type, a_affinity, b_type, b_affinity):\n    return bool((int(a_type) & int(b_affinity)) or (int(b_type) & int(a_affinity)))\n\n\nclass CandidateRejected(RuntimeError):\n    pass\n\n\nclass ForkStage08C(previous.ForkStage08B):\n    def __init__(self, owner):\n        super().__init__(owner)\n        self.moving_tip = owner.needed(self.mj.mjtObj.mjOBJ_GEOM, \'left_moving_jaw_sph_tip1\')\n        self.hand_geoms = []\n        self.mesh_vertices = {}\n        for gid in range(self.m.ngeom):\n            body = int(self.m.geom_bodyid[gid])\n            while body and body != self.gripper:\n                body = int(self.m.body_parentid[body])\n            if body != self.gripper:\n                continue\n            pair_forced = any({int(self.m.pair_geom1[k]), int(self.m.pair_geom2[k])} == {gid, self.o.table}\n                              for k in range(int(self.m.npair)))\n            if not pair_forced and not mask_compatible(self.m.geom_contype[gid], self.m.geom_conaffinity[gid],\n                                                      self.m.geom_contype[self.o.table], self.m.geom_conaffinity[self.o.table]):\n                continue\n            self.hand_geoms.append(gid)\n            if int(self.m.geom_type[gid]) == int(self.mj.mjtGeom.mjGEOM_MESH):\n                mid = int(self.m.geom_dataid[gid]); adr = int(self.m.mesh_vertadr[mid]); count = int(self.m.mesh_vertnum[mid])\n                self.mesh_vertices[gid] = np.array(self.m.mesh_vert[adr:adr+count], dtype=float, copy=True)\n        if not self.hand_geoms:\n            raise ValueError(\'No active left-hand collision geometry found\')\n        self.blocker = [g for g in self.hand_geoms if int(self.m.geom_bodyid[g]) == self.moving\n                        and g in self.mesh_vertices\n                        and owner.name(self.mj.mjtObj.mjOBJ_MESH, self.m.geom_dataid[g]) == \'moving_jaw_so101_gripper_part1_v1\']\n        if len(self.blocker) != 1:\n            raise ValueError(\'Expected the active moving-jaw part1 collision mesh; do not disable or replace it\')\n        owner.report.update(fork_planner_revision=REVISION, fork_clearance_preflight_passed=False,\n            fork_candidate_search=dict(maximum_candidates=MAX_CANDIDATES, table_margin_m=TABLE_CLEARANCE_M,\n                candidate_only_budget_s=CANDIDATE_BUDGET_S, opening_candidates_rad=list(OPENING_CANDIDATES_RAD),\n                height_fractions_of_handle_radius=list(PINCH_HEIGHT_FRACTIONS),\n                collision_masks_unchanged=True, fork_geometry_unchanged=True,\n                scope=\'scratch-only approach + descent + open-to-first-contact jaw sweep; actual grasp still required\',\n                candidates=[]))\n        self.selecting = False\n\n    def _check_budget(self):\n        self.o.budget()\n        if self.selecting:\n            import time\n            if time.monotonic() - self.search_started > CANDIDATE_BUDGET_S:\n                raise RuntimeError(\'08C candidate-search budget exhausted; preserve fork_clearance_selection.json\')\n\n    def _description(self, gid):\n        name = self.o.name(self.mj.mjtObj.mjOBJ_GEOM, gid)\n        bid = int(self.m.geom_bodyid[gid])\n        mesh = None\n        if gid in self.mesh_vertices:\n            mesh = self.o.name(self.mj.mjtObj.mjOBJ_MESH, self.m.geom_dataid[gid])\n        return dict(geom_id=gid, geom_name=name, body=self.o.name(self.mj.mjtObj.mjOBJ_BODY, bid),\n                    mesh=mesh, group=int(self.m.geom_group[gid]),\n                    contype=int(self.m.geom_contype[gid]), conaffinity=int(self.m.geom_conaffinity[gid]))\n\n    def clearance(self, data):\n        self._check_budget()\n        R = data.geom_xmat[self.o.table].reshape(3,3)\n        if int(self.m.geom_type[self.o.table]) != int(self.mj.mjtGeom.mjGEOM_BOX) or not np.allclose(R[:,2],[0,0,1],atol=1e-6):\n            raise ValueError(\'Expected the unchanged horizontal box table\')\n        n = R[:,2]; top = data.geom_xpos[self.o.table] + n*self.m.geom_size[self.o.table,2]\n        distances = []\n        for gid in self.hand_geoms:\n            kind = self.mj.mjtGeom(int(self.m.geom_type[gid])).name.replace(\'mjGEOM_\', \'\').lower()\n            gap = min_plane_coordinate(kind, self.m.geom_size[gid], data.geom_xpos[gid],\n                data.geom_xmat[gid].reshape(3,3), top, n, self.mesh_vertices.get(gid))\n            distances.append((gap,gid))\n        gap, gid = min(distances)\n        return dict(minimum_distance_m=float(gap), **self._description(gid))\n\n    def _open_evaluator(self, opening):\n        evaluate_base = super()._evaluator(False)\n        def evaluate(q):\n            self._check_budget()\n            self.scratch.qpos[self.g[\'qadr\']] = opening\n            return evaluate_base(q)\n        return evaluate\n\n    def _sample_path(self, path, evaluate, allow):\n        worst = None\n        for q in path:\n            evaluate(q)\n            info = self.clearance(self.scratch)\n            if worst is None or info[\'minimum_distance_m\'] < worst[\'minimum_distance_m\']:\n                worst = info\n            if info[\'minimum_distance_m\'] < TABLE_CLEARANCE_M:\n                raise CandidateRejected(\'Hand-table clearance: \' + json.dumps(info))\n            issues = self.contact_state(self.scratch,allow,planning=True)[-1]\n            if issues:\n                raise CandidateRejected(\'Whole-scene contact: \'+repr(issues[:3]))\n        return worst\n\n    def _jaw_sweep(self, qpin, opening):\n        """Check table clearance only until first intended contact; never force a static grasp."""\n        s, mj, m = self.scratch,self.mj,self.m\n        s.qpos[:] = self.d.qpos\n        s.qpos[self.qa] = qpin\n        worst, touched = None, False\n        n = max(1,int(math.ceil(abs(opening-self.close_floor)/.005)))\n        for j,qjaw in enumerate(np.linspace(opening,self.close_floor,n+1)):\n            self._check_budget();s.qpos[self.g[\'qadr\']]=qjaw;mj.mj_forward(m,s)\n            info=self.clearance(s)\n            if worst is None or info[\'minimum_distance_m\']<worst[\'minimum_distance_m\']:worst=info\n            if info[\'minimum_distance_m\']<TABLE_CLEARANCE_M:\n                raise CandidateRejected(\'Closing-jaw table clearance: \'+json.dumps(info))\n            issues=self.contact_state(s,True,planning=True)[-1]\n            if issues:raise CandidateRejected(\'Closing-jaw candidate contact: \'+repr(issues[:3]))\n            for i in range(s.ncon):\n                c=s.contact[i]; a,b=int(c.geom1),int(c.geom2)\n                other=b if a in self.geoms else a\n                if (a in self.geoms or b in self.geoms) and other in self.jaws and float(c.dist)<=.0005:\n                    touched=True\n            if touched:break\n        if not touched:\n            raise CandidateRejected(\'No geometric finger-fork contact in bounded closing interval\')\n        return dict(minimum_clearance=worst,first_geometric_contact_rad=float(qjaw),samples=j+1,\n                    scope=\'opening through first intended geometric contact; not a two-jaw force or grasp proof\')\n\n    def choose_grasp(self):\n        """Bounded selection of actual jaw clearance before physical left-arm motion."""\n        import time\n        baseline = {name: np.array(getattr(self.d,name),copy=True) for name in (\'qpos\',\'qvel\',\'ctrl\',\'act\')}\n        original_time=float(self.d.time)\n        self.search_started=time.monotonic();self.selecting=True\n        Rtarget=original.upright_frame(self.initial_R[:,1])\n        center=self.initial+self.initial_R@np.array([0.,-.020,0.])\n        # Confirm this center lies on the actual straight portion of the existing capsule.\n        hR=self.d.geom_xmat[self.handle].reshape(3,3)\n        hlocal=hR.T@(center-self.d.geom_xpos[self.handle])\n        if np.linalg.norm(hlocal[:2])>.0001 or abs(hlocal[2])>float(self.m.geom_size[self.handle,1])-.002:\n            raise ValueError(\'Intended pinch center is outside the existing straight handle segment\')\n        start=np.array([self.d.ctrl[r[\'cid\']] for r in self.arm])\n        accepted=None\n        try:\n            for index,candidate in enumerate(grasp_candidates(self.radius,self.g[\'low\'],self.g[\'high\'])):\n                self._check_budget()\n                item=dict(candidate_index=index,**candidate,accepted=False)\n                self.o.report[\'fork_candidate_search\'][\'candidates\'].append(item)\n                self.o.event(\'fork_clearance_candidate\',index=index,opening_rad=candidate[\'opening_rad\'],\n                             pinch_above_center_mm=1000*candidate[\'height_offset_m\'])\n                point=center+np.array([0.,0.,candidate[\'height_offset_m\']])\n                opening=candidate[\'opening_rad\']; ev=self._open_evaluator(opening)\n                try:\n                    qpin,diag=original.solve_frame(ev,point,Rtarget,start,self.lo,self.hi,self._check_budget)\n                    item[\'pinch_ik\']=diag\n                    if not diag.get(\'converged\'):raise CandidateRejected(\'Pinch endpoint did not converge\')\n                    ev(qpin)\n                    axis=self.scratch.xmat[self.gripper].reshape(3,3)[:,0]\n                    span=abs(float((self.scratch.geom_xpos[self.moving_tip]-self.scratch.geom_xpos[self.fixed_tip])@axis))\n                    gap=span-float(self.m.geom_size[self.moving_tip,0]+self.m.geom_size[self.fixed_tip,0])\n                    item[\'projected_open_tip_gap_m\']=gap\n                    if gap < 2*self.radius+.001:raise CandidateRejected(\'Opening is too narrow for the handle\')\n                    self._sample_path([qpin],ev,True)\n                    item[\'closure_preview\']=self._jaw_sweep(qpin,opening)\n                    # A candidate must support the complete approach + descent, not only pregrasp.\n                    found=None\n                    item[\'approach_attempts\']=[]\n                    for height in previous.candidate_heights(max(.14,point[2]+.08),point[2]):\n                        self._check_budget();pre=point.copy();pre[2]=height\n                        qa,da=original.solve_frame(ev,pre,Rtarget,qpin,self.lo,self.hi,self._check_budget)\n                        attempt=dict(height_m=height,ik=da,accepted=False);item[\'approach_attempts\'].append(attempt)\n                        if not da.get(\'converged\'):continue\n                        try:\n                            apath=previous.sample_connection(start,qa)\n                            agap=self._sample_path(apath,ev,False)\n                            # Full-frame descent is short and has a common reachable attitude.\n                            seed=qa.copy(); p0,R0,_,_=ev(seed);rotvec=original.rotation_vector(Rtarget@R0.T)\n                            path=[seed.copy()]; diagnostics=[];lgap=None\n                            for k in range(1,41):\n                                a=k/40\n                                q,dg=original.solve_frame(ev,p0+a*(point-p0),original.rotation_exp(a*rotvec)@R0,\n                                                          seed,self.lo,self.hi,self._check_budget)\n                                if not dg.get(\'converged\'):raise CandidateRejected(\'Descent waypoint IK rejected\')\n                                gaprec=self._sample_path(previous.sample_connection(seed,q),ev,True)\n                                if lgap is None or gaprec[\'minimum_distance_m\']<lgap[\'minimum_distance_m\']:lgap=gaprec\n                                path.append(q.copy());diagnostics.append(dg);seed=q\n                            # Recheck the closure sweep at the actual planned descent endpoint.\n                            close=self._jaw_sweep(seed,opening)\n                            approach=base.parameterize(apath,self.o.dt);lower=base.parameterize(path,self.o.dt)\n                            approach.update(target_m=pre.tolist(),target_rotation=Rtarget.tolist(),planning_mode=\'clearance_checked_endpoint_connection\')\n                            lower.update(target_m=point.tolist(),target_rotation=Rtarget.tolist(),diagnostics=diagnostics,planning_mode=\'clearance_checked_descent\')\n                            attempt.update(accepted=True,approach_minimum_clearance=agap,descent_minimum_clearance=lgap)\n                            found=(pre,approach,lower,close);break\n                        except CandidateRejected as exc:\n                            attempt[\'rejection\']=str(exc)\n                    if found is None:raise CandidateRejected(\'No approach + full descent passed\')\n                    pre,approach,lower,close=found\n                    item.update(accepted=True,chosen_pinch_target_m=point.tolist(),chosen_pregrasp_target_m=pre.tolist(),\n                                closure_preview=close,reference_clearance_m=float(pre[2]-point[2]))\n                    accepted=dict(index=index,candidate=item,point=point,R=Rtarget,pre=pre,approach=approach,lower=lower)\n                    break\n                except CandidateRejected as exc:\n                    item[\'rejection\']=str(exc)\n                finally:\n                    base.write_json(self.o.out/\'fork_clearance_selection.json\',self.o.report[\'fork_candidate_search\'])\n            if accepted is None:\n                raise RuntimeError(\'No 08C candidate passed active-jaw table clearance; no left-arm grasp executed. See fork_clearance_selection.json\')\n        finally:\n            self.selecting=False\n            unchanged=(float(self.d.time)==original_time and all(np.array_equal(getattr(self.d,n),a) for n,a in baseline.items()))\n            self.o.report[\'fork_candidate_search\'][\'execution_state_unchanged\']=unchanged\n            base.write_json(self.o.out/\'fork_clearance_selection.json\',self.o.report[\'fork_candidate_search\'])\n            if not unchanged:raise RuntimeError(\'Candidate selection modified execution state\')\n        self.open_q=accepted[\'candidate\'][\'opening_rad\']\n        self.approach_reference=accepted[\'point\'].copy()\n        self.o.report[\'fork_clearance_preflight_passed\']=True\n        self.o.report[\'fork_selected_clearance_candidate\']=accepted[\'candidate\']\n        self.o.report[\'fork_grasp_settings\'].update(opening_rad=self.open_q,\n            grasp_height_offset_m=accepted[\'candidate\'][\'height_offset_m\'],\n            explanation=\'gripper target above existing handle center; fork pose and geometry NOT changed\')\n        self.o.report[\'fork_pregrasp_selection\']=dict(selected_target_m=accepted[\'pre\'].tolist(),\n            reference_clearance_m=accepted[\'candidate\'][\'reference_clearance_m\'])\n        base.write_json(self.o.out/\'plan_fork_pregrasp.json\',accepted[\'approach\'])\n        base.write_json(self.o.out/\'plan_fork_lower_prechecked.json\',accepted[\'lower\'])\n        self.o.event(\'fork_clearance_selected\',candidate_index=accepted[\'index\'],opening_rad=self.open_q,\n            height_offset_mm=1000*accepted[\'candidate\'][\'height_offset_m\'])\n        return accepted\n\n    def plan(self,target,Rtarget,label,allow=False,carry=False):\n        plan=super().plan(target,Rtarget,label,allow,carry)\n        if not carry and label==\'fork_lower\':\n            ev=self._open_evaluator(float(self.d.qpos[self.g[\'qadr\']]))\n            path=np.asarray(plan[\'path\'])\n            for a,b in zip(path[:-1],path[1:]):self._sample_path(previous.sample_connection(a,b),ev,True)\n            self._jaw_sweep(path[-1],float(self.d.qpos[self.g[\'qadr\']]))\n        return plan\n\n    def execute(self):\n        o=self.o;o.event(\'left_fork_stage_start\',same_data=True,start_simulation_time_s=float(self.d.time))\n        o.snapshot(\'fork_start\')\n        selected=self.choose_grasp()\n        R,point=selected[\'R\'],selected[\'point\']\n        self.jaw(self.open_q,\'fork_open\')\n        # Fresh checks at execution entry. The precomputed plan is not proof of dynamics.\n        ev=self._open_evaluator(float(self.d.qpos[self.g[\'qadr\']]))\n        self._sample_path(np.asarray(selected[\'approach\'][\'path\']),ev,False)\n        self.move(selected[\'approach\'],\'fork_pregrasp\');self.hold(.25,\'fork_pregrasp_hold\')\n        o.snapshot(\'fork_pregrasp\')\n        p=self.plan(point,R,\'fork_lower\',True);self.move(p,\'fork_lower\',True);self.hold(.25,\'fork_approach_hold\',True)\n        if np.linalg.norm((self.d.xpos[self.fork]-self.initial)[:2])>.010:raise RuntimeError(\'Fork moved too far before closure\')\n        self.grasp()\n        lifted=self.d.xpos[self.fork].copy()+[0,0,.040]\n        p=self.plan(lifted,self.carry_R,\'fork_lift\',True,True);self.move(p,\'fork_lift\',True);self.hold(.75,\'fork_lift_hold\',True)\n        lift=self.carry_metrics(\'fork_lift_hold\');o.report[\'fork_lift_measurements\']=lift\n        if lift[\'minimum_rise_m\']<.025 or lift[\'two_jaw_contact_fraction\']<.8 or lift[\'maximum_table_support_n\']>=base.CONTACT_MIN_N:\n            raise RuntimeError(\'Fork lift not verified\')\n        o.report[\'fork_lift_verified\']=True;o.snapshot(\'fork_lifted\')\n        goal=self.d.xpos[self.fork].copy();goal[:2]=self.target[:2]\n        p=self.plan(goal,self.carry_R,\'fork_transfer\',True,True);self.move(p,\'fork_transfer\',True);self.hold(.5,\'fork_transfer_hold\',True)\n        trans=self.carry_metrics(\'fork_transfer_hold\');o.report[\'fork_transfer_measurements\']=trans\n        if trans[\'minimum_rise_m\']<.025 or trans[\'two_jaw_contact_fraction\']<.8 or trans[\'maximum_table_support_n\']>=base.CONTACT_MIN_N or trans[\'maximum_horizontal_destination_error_m\']>original.FORK_TOL:\n            raise RuntimeError(\'Fork destination carry failed; no set-down executed\')\n        o.report[\'fork_transfer_verified\']=True;o.snapshot(\'fork_transferred\')\n        clearance=self.bottom_clearance()\n        goal=base.measured_lowering_target(self.d.xpos[self.fork],[0,0,1],clearance)\n        goal[:2]=self.target[:2]\n        p=self.plan(goal,self.carry_R,\'fork_set_down\',True,True);self.move(p,\'fork_set_down\',True);self.hold(.5,\'fork_support_hold\',True)\n        support=float(np.mean([r[\'fork_table_normal_n\']>=base.CONTACT_MIN_N for r in self.stable(\'fork_support_hold\')]))\n        o.report[\'fork_support_fraction\']=support\n        if support<.8:raise RuntimeError(\'Fork table support missing before release\')\n        self.jaw(self.open_q,\'fork_release\',True);o.snapshot(\'fork_released\')\n        retreat=self.point(self.d).copy();retreat[2]=max(.14,retreat[2]+.05)\n        p=self.plan(retreat,R,\'fork_retreat\',True);self.move(p,\'fork_retreat\',True);self.hold(.75,\'fork_final_settle\')\n        rows=self.stable(\'fork_final_settle\',.5)\n        metrics=original.fork_placement_metrics(rows,self.initial,self.target,self.initial_R)\n        o.report[\'fork_placement_measurements\']=metrics\n        o.report[\'preservation_during_fork\']=dict(\n            maximum_cup_displacement_m=max(r[\'cup_displacement_during_fork_m\'] for r in o.rows[self.start_row:]),\n            maximum_plate_displacement_m=max(r[\'plate_displacement_during_fork_m\'] for r in o.rows[self.start_row:]))\n        if not metrics[\'placement_verified\']:raise RuntimeError(\'Fork destination release/orientation check failed\')\n        o.report[\'fork_placement_verified\']=True\n\n\nclass CombinedTrial08C(original.CombinedTrial):\n    def __init__(self,config):\n        super().__init__(config)\n        self.report.update(task_revision=REVISION,cup_preservation_evaluated=False,\n                           cup_preserved_after_fork=None,fork_placement_evaluated=False)\n        self.report[\'notes\'].append(\'08C checks active hand geometry before grasp descent; expert demonstrations, not final VLA control.\')\n\n    def execute(self):\n        # Same cup implementation and same physical state; no reset between objects.\n        cup07c.CupOrientationTrial.execute(self)\n        self.report[\'cup_stage_passed\'] = bool(self.report.get(\'transfer_test_passed\'))\n        self.report[\'cup_stage_end_time_s\'] = float(self.data.time)\n        self.report[\'cup_stage_placement\'] = dict(self.report.get(\'placement_measurements\',{}))\n        if not self.report[\'cup_stage_passed\']:\n            raise RuntimeError(\'Cup must pass before left-arm motion\')\n        self.snapshot(\'cup_placed\')\n        self.report[\'completed\']=False;self.report[\'grasp_test_passed\']=False\n        self.fork_stage=ForkStage08C(self)\n        self.fork_stage.execute()\n        self.report[\'fork_placement_evaluated\']=True\n        rows=self.fork_stage.stable(\'fork_final_settle\',.5)\n        cup=original.cup_metrics.placement_metrics(rows,self.cup_initial,self.target)\n        self.report[\'cup_final_placement\']=cup\n        self.report[\'cup_preservation_evaluated\']=True\n        self.report[\'cup_preserved_after_fork\']=bool(cup[\'placement_verified\'])\n        if not self.report[\'cup_preserved_after_fork\']:\n            raise RuntimeError(\'Cup destination failed after fork placement\')\n        self.report[\'sequence_test_passed\']=True;self.report[\'completed\']=True\n        self.report[\'grasp_test_passed\']=True;self.snapshot(\'sequence_final\')\n\n    def finish(self):\n        if self.fork_stage is not None:\n            rows=self.rows[self.fork_stage.start_row:]\n            self.report[\'fork_execution_rows\']=len(rows)\n            self.report[\'fork_placement_evaluated\']=bool(self.report.get(\'fork_placement_measurements\'))\n            self.report[\'preservation_observed_so_far\']={\n                key:max((float(row[key]) for row in rows if key in row),default=None)\n                for key in (\'cup_displacement_during_fork_m\',\'plate_displacement_during_fork_m\')}\n        super().finish()\n\n\ndef main(config):\n    trial=CombinedTrial08C(config)\n    try:trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.report[\'completed\']=False;trial.report[\'sequence_test_passed\']=False\n        trial.event(\'stopped\',reason=trial.report[\'stops\'][-1])\n        (trial.out/\'failure_traceback.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n        if trial.renderer is not None:trial.snapshot(\'stopped\')\n    finally:trial.finish()\n    print(\'SEQUENCE RESULT \'+json.dumps({k:trial.report.get(k) for k in (\n        \'sequence_test_passed\',\'cup_stage_passed\',\'fork_grasp_verified\',\'fork_lift_verified\',\n        \'fork_placement_verified\',\'cup_preservation_evaluated\',\'cup_preserved_after_fork\',\n        \'stops\',\'report_file\')},allow_nan=False),flush=True)\n    return 0 if trial.report[\'sequence_test_passed\'] else 2\n\n\nif __name__==\'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_sequence_08c.py': '"""Synthetic 08C geometry and source-preservation tests, not robot contact physics."""\nimport ast\nimport json\nimport math\nimport sys\nimport tempfile\nfrom pathlib import Path\nimport unittest\nfrom unittest.mock import patch\nfrom types import SimpleNamespace as NS\nimport numpy as np\nimport sequence_08c_worker as w\n\nclass GeometryTests(unittest.TestCase):\n    def distance(self,kind,size=(1,1,1),p=(0,0,3),R=None,v=None):\n        return w.min_plane_coordinate(kind,size,p,np.eye(3) if R is None else R,[0,0,0],[0,0,1],v)\n    def test_sphere(self):self.assertAlmostEqual(self.distance(\'sphere\'),2.)\n    def test_box(self):self.assertAlmostEqual(self.distance(\'box\'),2.)\n    def test_capsule(self):self.assertAlmostEqual(self.distance(\'capsule\'),1.)\n    def test_cylinder(self):self.assertAlmostEqual(self.distance(\'cylinder\'),2.)\n    def test_ellipsoid(self):self.assertAlmostEqual(self.distance(\'ellipsoid\',size=(1,2,1)),2.)\n    def test_mesh_vertex_support(self):\n        self.assertAlmostEqual(self.distance(\'mesh\',v=[[0,0,-.01],[1,0,.3],[0,1,.5]]),2.99)\n    def test_mesh_does_not_use_geom_size(self):\n        v=[[0,0,-.01],[1,0,.3]]\n        self.assertEqual(self.distance(\'mesh\',size=(999,999,999),v=v),self.distance(\'mesh\',size=(0,0,0),v=v))\n    def test_rotated_mesh(self):\n        R=w.original.rotation_exp([0,math.pi/2,0]);v=np.array([[.005,0,0],[.01,.02,.03]])\n        want=np.min((v@R.T+np.array([0,0,3]))[:,2])\n        self.assertAlmostEqual(self.distance(\'mesh\',R=R,v=v),want)\n    def test_tilted_capsule(self):\n        R=w.original.rotation_exp([math.pi/2,0,0])\n        self.assertAlmostEqual(self.distance(\'capsule\',R=R),2.)\n    def test_mesh_empty_rejected(self):\n        with self.assertRaises(ValueError):self.distance(\'mesh\',v=[])\n    def test_mesh_nan_rejected(self):\n        with self.assertRaises(ValueError):self.distance(\'mesh\',v=[[0,0,float(\'nan\')]])\n    def test_unknown_rejected(self):\n        with self.assertRaises(ValueError):self.distance(\'guess\')\n    def test_negative_penetration_not_clipped(self):\n        self.assertLess(self.distance(\'mesh\',p=[0,0,.0048],v=[[0,0,-.006]]),0.)\n    def test_synthetic_raise_reference_not_object(self):\n        v=np.array([[0,0,-.006],[0,0,0.]])\n        a=self.distance(\'mesh\',p=[0,0,.0048],v=v)\n        b=self.distance(\'mesh\',p=[0,0,.007],v=v)\n        self.assertLess(a,0.);self.assertGreater(b,w.TABLE_CLEARANCE_M)\n    def test_read_only_vertices(self):\n        v=np.array([[0.,0.,-.1]]);s=np.ones(3);before=v.copy()\n        self.distance(\'mesh\',s,v=v);np.testing.assert_array_equal(v,before)\n    def test_invalid_rotation_rejected(self):\n        with self.assertRaises(ValueError):self.distance(\'mesh\',R=np.zeros((3,3)),v=[[0,0,0]])\n\nclass CandidateTests(unittest.TestCase):\n    def test_nine_bounded_candidates(self):\n        c=w.grasp_candidates(.004,-.17453,1.74533)\n        self.assertEqual(len(c),9);self.assertTrue(all(0<x[\'height_offset_m\']<.004 for x in c))\n    def test_original_height_included(self):\n        self.assertAlmostEqual(w.grasp_candidates(.004,-.17453,1.74533)[0][\'height_offset_m\'],.0008)\n    def test_only_reduced_openings(self):\n        self.assertEqual({x[\'opening_rad\'] for x in w.grasp_candidates(.004,-.17453,1.74533)},{.22,.1,0.})\n    def test_large_handle_rejected(self):\n        with self.assertRaises(ValueError):w.grasp_candidates(.02,-.2,1.)\n    def test_nan_handle_rejected(self):\n        with self.assertRaises(ValueError):w.grasp_candidates(float(\'nan\'),-.2,1.)\n    def test_bounds_respected(self):\n        self.assertTrue(all(.1<x[\'opening_rad\']<.3 for x in w.grasp_candidates(.004,.09,.31)))\n    def test_mask_active(self):self.assertTrue(w.mask_compatible(1,1,1,1))\n    def test_visual_only_mask_rejected(self):self.assertFalse(w.mask_compatible(0,0,1,1))\n    def test_one_direction_sufficient(self):self.assertTrue(w.mask_compatible(2,0,0,2))\n    def test_unrelated_masks(self):self.assertFalse(w.mask_compatible(2,2,1,1))\n\nclass RuntimePreservationTests(unittest.TestCase):\n    def test_runtime_guards_and_grasp_inherited(self):\n        for name in (\'tick\',\'contact_state\',\'jaw\',\'hold\',\'grasp\',\'move\',\'stable\',\'carry_metrics\',\'bottom_clearance\'):\n            with self.subTest(name=name):\n                self.assertIs(getattr(w.ForkStage08C,name),getattr(w.original.ForkStage,name))\n    def test_cup_and_recorder_inherited(self):\n        for name in (\'load\',\'tick\',\'camera_file\',\'setup_render\',\'close_on_cup\'):\n            with self.subTest(name=name):\n                self.assertIs(getattr(w.CombinedTrial08C,name),getattr(w.original.CombinedTrial,name))\n    def test_success_thresholds_unchanged(self):\n        self.assertEqual(w.base.GRIPPER_TORQUE_CAP,.25);self.assertEqual(w.original.FORK_TOL,.008)\n        self.assertEqual(w.original.CUP_PRESERVE_TOL,.008);self.assertEqual(w.base.PENETRATION_LIMIT,.003)\n    def test_all_hand_groups_included_no_group_filter(self):\n        import inspect\n        src=inspect.getsource(w.ForkStage08C.__init__)\n        self.assertNotIn(\'geom_group\',src)\n        self.assertIn(\'mesh_vert\',src);self.assertIn(\'moving_jaw_so101_gripper_part1_v1\',src)\n    def test_no_direct_execution_or_model_assignment(self):\n        tree=ast.parse(Path(w.__file__).read_text())\n        for n in ast.walk(tree):\n            if isinstance(n,(ast.Assign,ast.AnnAssign,ast.AugAssign)):\n                targets=n.targets if isinstance(n,ast.Assign) else [n.target]\n                for t in targets:\n                    text=ast.unparse(t)\n                    for blocked in (\'self.d.qpos\',\'self.d.qvel\',\'self.d.ctrl\',\'self.m.geom\',\'self.m.mesh\',\'self.m.body\'):\n                        self.assertFalse(text.startswith(blocked),text)\n    def test_old_position_cannot_pass_new_destination(self):\n        row=dict(fork_x=-.11,fork_y=.1,fork_z=.034,fork_table_normal_n=.1,\n                 fork_fixed_normal_n=0,fork_moving_normal_n=0,fork_orientation_error_deg=0)\n        m=w.original.fork_placement_metrics([row],[-.11,.1,.034],[-.135,.1,.034],np.eye(3))\n        self.assertFalse(m[\'placement_verified\'])\n    def test_bad_path_rejected_before_execution(self):\n        t=w.ForkStage08C.__new__(w.ForkStage08C);t.scratch=NS();t.clearance=lambda _:dict(minimum_distance_m=-.001)\n        with self.assertRaises(w.CandidateRejected):t._sample_path([np.zeros(5)],lambda _:None,True)\n    def test_clearance_margin_not_just_negative_contacts(self):\n        t=w.ForkStage08C.__new__(w.ForkStage08C);t.scratch=NS();t.clearance=lambda _:dict(minimum_distance_m=.0001)\n        with self.assertRaises(w.CandidateRejected):t._sample_path([np.zeros(5)],lambda _:None,True)\n    def test_whole_scene_contact_still_rejected(self):\n        t=w.ForkStage08C.__new__(w.ForkStage08C);t.scratch=NS();t.clearance=lambda _:dict(minimum_distance_m=.002)\n        t.contact_state=lambda *a,**k:(None,None,None,None,[\'unexpected contact\'])\n        with self.assertRaises(w.CandidateRejected):t._sample_path([np.zeros(5)],lambda _:None,True)\n    def test_clear_path_accepts_only_evidence(self):\n        t=w.ForkStage08C.__new__(w.ForkStage08C);t.scratch=NS();t.clearance=lambda _:dict(minimum_distance_m=.002)\n        t.contact_state=lambda *a,**k:(None,None,None,None,[])\n        self.assertAlmostEqual(t._sample_path([np.zeros(5)],lambda _:None,True)[\'minimum_distance_m\'],.002)\n\n\nclass SelectionFlowTests(unittest.TestCase):\n    def stage(self, folder, reject=False):\n        t=w.ForkStage08C.__new__(w.ForkStage08C)\n        t.o=NS(out=Path(folder), report={\'fork_candidate_search\':{\'candidates\':[]},\'fork_grasp_settings\':{}},\n               dt=.005,event=lambda *a,**k:None,budget=lambda:None)\n        t.d=NS(qpos=np.zeros(12),qvel=np.zeros(12),ctrl=np.zeros(12),act=np.zeros(0),time=34.36,\n               geom_xmat=np.tile(np.eye(3).flatten(),(3,1)),geom_xpos=np.array([[0,.08,.034],[0,0,0],[0,0,0.]]))\n        # Real handle\'s local z follows world y.\n        t.d.geom_xmat[0]=w.original.rotation_exp([-math.pi/2,0,0]).flatten()\n        t.m=NS(geom_size=np.array([[.004,.0325,0],[.00075,0,0],[.00075,0,0]]))\n        t.initial=np.array([0.,.1,.034]);t.initial_R=np.eye(3)\n        t.handle=0;t.fixed_tip=1;t.moving_tip=2;t.gripper=0\n        t.radius=.004;t.g={\'low\':-.17453,\'high\':1.74533};t.arm=[{\'cid\':i} for i in range(5)]\n        t.lo=-np.ones(5);t.hi=np.ones(5)\n        t.scratch=NS(xmat=np.eye(3).reshape(1,9),geom_xpos=np.array([[0,0,0],[-.015,0,0],[.015,0,0]]))\n        t._open_evaluator=lambda op:lambda q:(q[:3],np.eye(3),np.c_[np.eye(3),np.zeros((3,2))],np.zeros((3,5)))\n        def sample(*args,**kwargs):\n            if reject:raise w.CandidateRejected(\'synthetic table collision\')\n            return dict(minimum_distance_m=.001)\n        t._sample_path=sample;t._jaw_sweep=lambda *a:dict(samples=3,first_geometric_contact_rad=-.1)\n        return t\n    def solver(self,ev,target,R,seed,lo,hi,budget):\n        q=np.r_[target,0,0];return q,dict(converged=bool(target[2]<=.071),position_error_m=0.,rotation_error_deg=0.)\n    def test_accepted_search_preserves_execution(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=self.stage(folder);before=t.d.qpos.copy()\n            with patch.object(w.original,\'solve_frame\',self.solver):out=t.choose_grasp()\n            self.assertTrue(t.o.report[\'fork_clearance_preflight_passed\'])\n            self.assertTrue(t.o.report[\'fork_candidate_search\'][\'execution_state_unchanged\'])\n            self.assertGreaterEqual(out[\'candidate\'][\'reference_clearance_m\'],.03)\n            np.testing.assert_array_equal(t.d.qpos,before)\n    def test_all_candidates_blocked_stops_before_execution(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=self.stage(folder,reject=True)\n            with patch.object(w.original,\'solve_frame\',self.solver):\n                with self.assertRaisesRegex(RuntimeError,\'No 08C candidate\'):t.choose_grasp()\n            self.assertEqual(len(t.o.report[\'fork_candidate_search\'][\'candidates\']),9)\n            self.assertTrue((Path(folder)/\'fork_clearance_selection.json\').is_file())\n            self.assertEqual(t.d.time,34.36)\n    def test_execution_mutation_detected(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=self.stage(folder)\n            def bad(*a,**k):t.d.ctrl[0]=1.;return self.solver(*a,**k)\n            with patch.object(w.original,\'solve_frame\',bad):\n                with self.assertRaisesRegex(RuntimeError,\'modified execution state\'):t.choose_grasp()\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])\n    result=unittest.TextTestRunner(verbosity=1).run(suite)\n    out=dict(kind=\'SYNTHETIC_GEOMETRY_AND_CODE_TESTS_NOT_MUJOCO_PHYSICS\',tests_run=result.testsRun,\n             failures=len(result.failures),errors=len(result.errors),skipped=len(result.skipped),passed=result.wasSuccessful())\n    Path(\'sequence_08c_test_summary.json\').write_text(json.dumps(out,indent=2))\n    print(json.dumps(out));raise SystemExit(0 if result.wasSuccessful() else 1)\n', 'sequence_08d_worker.py': '"""08D: bounded carry-IK recovery for the expert demonstration generator.\n\nThe unchanged 08C grasp and runtime controller still execute actual contacts.\nNumerical recovery applies only to scratch carrying plans. Acceptance tolerances\nare unchanged. Lift targets are explicitly selected from 40, 35, then 30 mm;\nonly a complete, checked path is returned for execution. Measured rise must\nstill exceed the original 25 mm criterion. This is not learned VLA control.\n"""\nfrom pathlib import Path\nimport json\nimport math\nimport sys\nimport time\nimport traceback\nimport numpy as np\nimport base06b as base\nimport two_object_worker as original\nimport cup_orientation_worker as cup07c\nimport sequence_08b_worker as approach\nimport sequence_08c_worker as previous\n\nREVISION = \'08d-tolerance-scaled-carry-ik-bounded-lift-v1\'\nLIFT_HEIGHTS_M = (.040, .035, .030)\nMAX_RECOVERY_ITERATIONS = 60\nMAX_RECOVERY_STEP_RAD = .025\nCARRY_PLANNING_BUDGET_S = 30.\nDAMPINGS = (1e-4, 1e-2, 1., 100.)\n\n\nclass CarryIKFailure(RuntimeError):\n    def __init__(self, details):\n        self.details = details\n        super().__init__(\'Carrying waypoint did not meet unchanged tolerances: \' + json.dumps(details))\n\n\ndef lift_candidates(current_position, requested_target):\n    p = np.asarray(current_position, dtype=float)\n    target = np.asarray(requested_target, dtype=float)\n    if p.shape != (3,) or target.shape != (3,) or not np.isfinite([p, target]).all():\n        raise ValueError(\'Finite fork positions required\')\n    if not np.allclose(target-p, [0., 0., .040], atol=1e-9, rtol=0):\n        raise ValueError(\'This bounded selection expects the original +40 mm lift request\')\n    return [p + np.array([0., 0., h]) for h in LIFT_HEIGHTS_M]\n\n\ndef tolerance_scaled_solve(evaluate, target, axis, seed, low, high, budget):\n    """Try a tolerance-normalized objective; exact pass thresholds never change.\n\n    The angular residual uses the unit-vector chord equivalent to two degrees.\n    Independent position and angle checks, not the merit value, declare success.\n    This routine updates joint candidates only; evaluate owns scratch data.\n    """\n    target = np.array(target, dtype=float, copy=True)\n    axis = base.normalize(axis)\n    q, lo, hi = [np.array(x, dtype=float, copy=True) for x in (seed, low, high)]\n    if (q.shape != (5,) or lo.shape != q.shape or hi.shape != q.shape\n            or target.shape != (3,) or not np.isfinite([q, lo, hi]).all()\n            or not np.isfinite(target).all() or np.any(lo >= hi)\n            or np.any(q < lo) or np.any(q > hi)):\n        raise ValueError(\'Invalid five-joint recovery inputs\')\n    angular_scale = 2. * math.sin(math.radians(base.AXIS_TOL_DEG) / 2.)\n    if base.POSITION_TOL <= 0 or angular_scale <= 0:\n        raise ValueError(\'Invalid existing tolerances\')\n    evaluations = 0\n\n    def measure(x):\n        nonlocal evaluations\n        budget()\n        p, a, jp, ja = [np.asarray(v, dtype=float) for v in evaluate(x)]\n        evaluations += 1\n        if (p.shape != (3,) or a.shape != (3,) or jp.shape != (3, 5)\n                or ja.shape != (3, 5) or not all(np.isfinite(v).all() for v in (p, a, jp, ja))\n                or not math.isclose(float(np.linalg.norm(a)), 1., abs_tol=1e-6)):\n            raise ValueError(\'Invalid pose, unit axis, or Jacobian\')\n        pe = float(np.linalg.norm(target-p))\n        ae = math.degrees(math.acos(float(np.clip(a @ axis, -1., 1.))))\n        residual = np.r_[(target-p)/base.POSITION_TOL, (axis-a)/angular_scale]\n        J = np.vstack([jp/base.POSITION_TOL, ja/angular_scale])\n        merit = float(residual @ residual)\n        return pe, ae, residual, J, merit\n\n    def result(state, iterations, reason):\n        pe, ae, _, J, _ = state\n        sv = np.linalg.svd(J, compute_uv=False)\n        rank = int(np.linalg.matrix_rank(J))\n        return q.copy(), dict(\n            converged=bool(pe <= base.POSITION_TOL and ae <= base.AXIS_TOL_DEG),\n            reason=reason, iterations=iterations, evaluations=evaluations,\n            position_error_m=pe, axis_error_deg=ae,\n            position_tolerance_m=base.POSITION_TOL, axis_tolerance_deg=base.AXIS_TOL_DEG,\n            scaled_jacobian_rank=rank, scaled_singular_values=sv.tolist(),\n            minimum_joint_limit_margin_rad=float(np.minimum(q-lo, hi-q).min()),\n            method=\'tolerance_normalized_damped_least_squares\')\n\n    state = measure(q)\n    for iteration in range(MAX_RECOVERY_ITERATIONS + 1):\n        if state[0] <= base.POSITION_TOL and state[1] <= base.AXIS_TOL_DEG:\n            return result(state, iteration, \'both_original_gates_satisfied\')\n        if iteration == MAX_RECOVERY_ITERATIONS:\n            return result(state, iteration, \'bounded_iteration_limit\')\n        residual, J, merit = state[2], state[3], state[4]\n        H, g = J.T @ J, J.T @ residual\n        diagonal = np.diag(np.maximum(np.diag(H), 1.))\n        best = None\n        for damping in DAMPINGS:\n            budget()\n            try:\n                dq = np.linalg.solve(H + damping * diagonal, g)\n            except np.linalg.LinAlgError:\n                continue\n            if not np.isfinite(dq).all():\n                continue\n            dq *= min(1., MAX_RECOVERY_STEP_RAD/max(float(np.max(np.abs(dq))), 1e-12))\n            for scale in (1., .5, .25, .125, .0625):\n                candidate = np.clip(q + scale * dq, lo, hi)\n                if np.max(np.abs(candidate-q)) < 1e-12:\n                    continue\n                cs = measure(candidate)\n                if cs[0] <= base.POSITION_TOL and cs[1] <= base.AXIS_TOL_DEG:\n                    q = candidate\n                    return result(cs, iteration+1, \'both_original_gates_satisfied\')\n                if cs[4] < merit - max(1e-12, 1e-10 * merit):\n                    if best is None or cs[4] < best[1][4]:\n                        best = (candidate, cs)\n            if best is not None:\n                break\n        if best is None:\n            # Restore scratch transforms to the returned candidate, not the last rejected sample.\n            state = measure(q)\n            return result(state, iteration+1, \'no_normalized_local_improvement\')\n        q, state = best\n    raise AssertionError(\'Unreachable solver exit\')\n\n\nclass ForkStage08D(previous.ForkStage08C):\n    def __init__(self, owner):\n        super().__init__(owner)\n        owner.report.update(fork_planner_revision=REVISION,\n            fork_solver_recovery=dict(position_tolerance_m=base.POSITION_TOL,\n                axis_tolerance_deg=base.AXIS_TOL_DEG, maximum_iterations=MAX_RECOVERY_ITERATIONS,\n                maximum_joint_candidate_step_rad=MAX_RECOVERY_STEP_RAD,\n                maximum_planning_wall_s=CARRY_PLANNING_BUDGET_S,\n                scope=\'carried-fork scratch solver only; no runtime or success-gate relaxation\'),\n            fork_lift_selection=None, fork_carry_plan_history=[])\n        self.carry_deadline = None\n\n    def carry_budget(self):\n        self.o.budget()\n        if self.carry_deadline is not None and time.monotonic() > self.carry_deadline:\n            raise RuntimeError(\'08D carried-plan budget exceeded; inspect fork_carry_plan_history\')\n\n    def _carry_plan(self, target, Rtarget, label, allow):\n        if not allow:\n            raise ValueError(\'Carried-fork planning requires the existing intended-contact mode\')\n        evaluate = self._evaluator(True)  # Verifies real grasp/contact; modifies scratch only.\n        seed = np.array([self.d.ctrl[r[\'cid\']] for r in self.arm])\n        p0, R0, _, _ = evaluate(seed)\n        a0, desired = R0[:, 2].copy(), Rtarget[:, 2].copy()\n        if cup07c.tilt_deg(a0) > approach.PLANNING_TILT_DEG:\n            raise RuntimeError(\'Initial carried fork exceeds unchanged planning tilt bound\')\n\n        def axis_eval(q):\n            self.carry_budget()\n            p, R, jp, jr = evaluate(q)\n            a = R[:, 2].copy()\n            return p, a, jp, -base.skew(a) @ jr\n\n        path, diagnostics = [seed.copy()], []\n        for k in range(1, 41):\n            self.carry_budget()\n            fraction = k/40\n            goal = p0 + fraction*(target-p0)\n            axis = base.normalize((1-fraction)*a0 + fraction*desired)\n            q, legacy = base.solve_pose(axis_eval, goal, axis, seed, self.lo, self.hi, self.carry_budget)\n            diag = dict(legacy, method=\'unchanged_original_solver\', recovery_used=False)\n            if not legacy.get(\'converged\'):\n                q, recovery = tolerance_scaled_solve(axis_eval, goal, axis, q,\n                                                      self.lo, self.hi, self.carry_budget)\n                diag = dict(recovery, recovery_used=True, original_attempt=legacy)\n                self.o.event(\'fork_solver_recovery\', phase=label, waypoint=k,\n                    original_error_mm=1000*legacy[\'position_error_m\'],\n                    recovered=bool(recovery[\'converged\']), position_error_mm=1000*recovery[\'position_error_m\'],\n                    axis_error_deg=recovery[\'axis_error_deg\'])\n            # Verify both gates using a fresh evaluation even when a solver reports success.\n            p, a, _, _ = axis_eval(q)\n            pe = float(np.linalg.norm(p-goal))\n            ae = math.degrees(math.acos(float(np.clip(a @ axis, -1., 1.))))\n            diag.update(position_error_m=pe, axis_error_deg=ae,\n                        converged=bool(pe <= base.POSITION_TOL and ae <= base.AXIS_TOL_DEG))\n            diagnostics.append(diag)\n            if not diag[\'converged\']:\n                raise CarryIKFailure(dict(phase=label,waypoint=k,target_m=target.tolist(),\n                    waypoint_target_m=goal.tolist(),diagnostics=diagnostics,\n                    candidate_joint_targets_rad=q.tolist(),reason=\'unchanged_position_or_axis_gate_failed\'))\n            # Keep every existing whole-scene contact, 25-degree tilt, and 15-degree orientation check.\n            for candidate in approach.sample_connection(seed, q):\n                evaluate(candidate)\n                issues = self._issues(allow, True)\n                if issues:\n                    fail = dict(phase=label,waypoint=k,issues=issues[:6],target_m=target.tolist())\n                    self.o.report[\'fork_plan_failure\'] = fail\n                    base.write_json(self.o.out/f\'plan_{label}_failure.json\', fail)\n                    raise RuntimeError(f\'{label}: sampled path rejected: {issues[:3]}\')\n            path.append(q.copy())\n            seed = q\n        plan = base.parameterize(path, self.o.dt)\n        plan.update(target_m=target.tolist(),target_rotation=Rtarget.tolist(),carry_prediction_only=True,\n                    diagnostics=diagnostics,planning_mode=\'fork_origin_and_up_axis_with_bounded_solver_recovery\',\n                    planner_revision=REVISION,\n                    recovered_waypoints=sum(bool(x[\'recovery_used\']) for x in diagnostics))\n        return plan\n\n    def plan(self, target, Rtarget, label, allow=False, carry=False):\n        if not carry:\n            return super().plan(target, Rtarget, label, allow, carry)\n        target, Rtarget = base.checked_pose(target, Rtarget)\n        if label not in (\'fork_lift\', \'fork_transfer\', \'fork_set_down\'):\n            raise ValueError(\'Unsupported carried-fork phase\')\n        self.o.event(\'fork_plan_start\',phase=label,revision=REVISION)\n        self.carry_deadline = time.monotonic() + CARRY_PLANNING_BUDGET_S\n        state = {n: np.array(getattr(self.d,n),copy=True) for n in (\'qpos\',\'qvel\',\'ctrl\',\'act\')}\n        t0 = float(self.d.time)\n        fork_now = self.d.xpos[self.fork].copy()\n        candidates = lift_candidates(fork_now, target) if label == \'fork_lift\' else [target]\n        attempts = []\n        selection = None\n        if label == \'fork_lift\':\n            selection = dict(requested_lift_m=.040,candidate_lifts_m=list(LIFT_HEIGHTS_M),\n                measured_minimum_rise_requirement_m=.025,selected_lift_m=None,\n                selected_target_m=None,attempts=attempts,execution_state_unchanged=None,\n                meaning=\'changed commanded height only if the taller complete path fails IK; actual lift still measured\')\n            self.o.report[\'fork_lift_selection\'] = selection\n        try:\n            for index, candidate in enumerate(candidates):\n                self.carry_budget()\n                attempt = dict(candidate_index=index,target_m=candidate.tolist(),accepted=False)\n                if label == \'fork_lift\':\n                    attempt[\'commanded_lift_m\'] = float(candidate[2]-fork_now[2])\n                attempts.append(attempt)\n                try:\n                    plan = self._carry_plan(candidate, Rtarget, label, allow)\n                except CarryIKFailure as exc:\n                    attempt[\'failure\'] = exc.details\n                    base.write_json(self.o.out/f\'plan_{label}_candidate_{index:02d}_failure.json\',exc.details)\n                    self.o.event(\'fork_carry_candidate_rejected\',phase=label,candidate=index,\n                                 waypoint=exc.details[\'waypoint\'],reason=exc.details[\'reason\'])\n                    continue\n                # A collision, contact-precondition failure, or timeout is not swallowed/retried.\n                attempt.update(accepted=True,recovered_waypoints=plan[\'recovered_waypoints\'])\n                if selection is not None:\n                    selection.update(selected_lift_m=float(candidate[2]-fork_now[2]),\n                                     selected_target_m=candidate.tolist())\n                base.write_json(self.o.out/f\'plan_{label}.json\',plan)\n                self.o.event(\'fork_plan_ready\',phase=label,duration_s=plan[\'duration_s\'],\n                    selected_lift_mm=(1000*selection[\'selected_lift_m\'] if selection else None),\n                    recovered_waypoints=plan[\'recovered_waypoints\'])\n                return plan\n            fail = dict(phase=label,reason=\'bounded_candidates_exhausted\',attempts=attempts)\n            self.o.report[\'fork_plan_failure\'] = fail\n            base.write_json(self.o.out/f\'plan_{label}_failure.json\',fail)\n            raise RuntimeError(f\'{label}: no complete path met unchanged gates; inspect fork_lift_selection / plan failure JSON\')\n        finally:\n            self.carry_deadline = None\n            unchanged = float(self.d.time) == t0 and all(np.array_equal(getattr(self.d,n),v) for n,v in state.items())\n            self.o.report[\'fork_carry_plan_history\'].append(dict(phase=label,attempts=attempts,\n                                                               execution_state_unchanged=unchanged))\n            if selection is not None:\n                selection[\'execution_state_unchanged\'] = unchanged\n                base.write_json(self.o.out/\'fork_lift_selection.json\',selection)\n            if not unchanged:\n                raise RuntimeError(\'Carried-path planning changed execution state\')\n\n\nclass CombinedTrial08D(previous.CombinedTrial08C):\n    def __init__(self, config):\n        super().__init__(config)\n        self.report[\'task_revision\'] = REVISION\n        self.report[\'notes\'].append(\'08D numerical recovery and declared 40/35/30 mm lift selection; expert data only.\')\n\n    def execute(self):\n        cup07c.CupOrientationTrial.execute(self)\n        self.report[\'cup_stage_passed\'] = bool(self.report.get(\'transfer_test_passed\'))\n        self.report[\'cup_stage_end_time_s\'] = float(self.data.time)\n        self.report[\'cup_stage_placement\'] = dict(self.report.get(\'placement_measurements\',{}))\n        if not self.report[\'cup_stage_passed\']:\n            raise RuntimeError(\'Cup must pass before left-arm motion\')\n        self.snapshot(\'cup_placed\')\n        self.report[\'completed\'] = False\n        self.report[\'grasp_test_passed\'] = False\n        self.fork_stage = ForkStage08D(self)\n        self.fork_stage.execute()  # Exact inherited 08C physical sequence and measured gates.\n        self.report[\'fork_placement_evaluated\'] = True\n        rows = self.fork_stage.stable(\'fork_final_settle\', .5)\n        cup = original.cup_metrics.placement_metrics(rows, self.cup_initial, self.target)\n        self.report[\'cup_final_placement\'] = cup\n        self.report[\'cup_preservation_evaluated\'] = True\n        self.report[\'cup_preserved_after_fork\'] = bool(cup[\'placement_verified\'])\n        if not self.report[\'cup_preserved_after_fork\']:\n            raise RuntimeError(\'Cup destination failed after fork placement\')\n        self.report.update(sequence_test_passed=True,completed=True,grasp_test_passed=True)\n        self.snapshot(\'sequence_final\')\n\n\ndef main(config):\n    trial = CombinedTrial08D(config)\n    try:\n        trial.execute()\n    except Exception as exc:\n        trial.report[\'stops\'].append(f\'{type(exc).__name__}: {exc}\')\n        trial.report.update(completed=False,sequence_test_passed=False)\n        trial.event(\'stopped\',reason=trial.report[\'stops\'][-1])\n        (trial.out/\'failure_traceback.txt\').write_text(traceback.format_exc(),encoding=\'utf-8\')\n        if trial.renderer is not None:\n            trial.snapshot(\'stopped\')\n    finally:\n        trial.finish()\n    print(\'SEQUENCE RESULT \'+json.dumps({k:trial.report.get(k) for k in (\n        \'sequence_test_passed\',\'cup_stage_passed\',\'fork_grasp_verified\',\'fork_lift_verified\',\n        \'fork_placement_verified\',\'cup_preservation_evaluated\',\'cup_preserved_after_fork\',\n        \'stops\',\'report_file\')},allow_nan=False),flush=True)\n    return 0 if trial.report[\'sequence_test_passed\'] else 2\n\n\nif __name__ == \'__main__\':\n    raise SystemExit(main(json.loads(Path(sys.argv[1]).read_text(encoding=\'utf-8\'))))\n', 'test_sequence_08d.py': '"""Offline numerical and workflow tests; no MuJoCo robot-physics claim."""\nfrom pathlib import Path\nfrom types import SimpleNamespace as NS\nfrom unittest.mock import patch\nimport ast\nimport json\nimport math\nimport sys\nimport tempfile\nimport unittest\nimport numpy as np\nimport sequence_08d_worker as w\n\n\ndef evaluate_full(q):\n    Rx=w.original.rotation_exp([q[3],0,0]);Ry=w.original.rotation_exp([0,q[4],0])\n    R=Rx@Ry;a=R[:,2];jp=np.c_[np.eye(3),np.zeros((3,2))]\n    jr=np.c_[np.zeros((3,3)),[1,0,0],Rx@np.array([0,1,0])]\n    return q[:3].copy(),a,jp,-w.base.skew(a)@jr\n\n\ndef owner_stage(folder):\n    t=w.ForkStage08D.__new__(w.ForkStage08D)\n    t.o=NS(out=Path(folder),report={\'fork_carry_plan_history\':[]},dt=.005,\n           budget=lambda:None,event=lambda *a,**k:None)\n    t.d=NS(ctrl=np.zeros(5),qpos=np.zeros(7),qvel=np.zeros(6),act=np.zeros(0),\n           xpos=np.zeros((1,3)),time=47.84)\n    t.fork=0;t.arm=[{\'cid\':i} for i in range(5)]\n    t.lo=-np.ones(5);t.hi=np.ones(5);t.carry_deadline=None\n    def ev(q):\n        Rx=w.original.rotation_exp([q[3],0,0]);Ry=w.original.rotation_exp([0,q[4],0]);R=Rx@Ry\n        jr=np.c_[np.zeros((3,3)),[1,0,0],Rx@np.array([0,1,0])]\n        return q[:3].copy(),R,np.c_[np.eye(3),np.zeros((3,2))],jr\n    t._evaluator=lambda carry:ev;t._issues=lambda *a:[]\n    return t\n\n\nclass NumericalTests(unittest.TestCase):\n    def call(self,ev=evaluate_full,target=(.01,.02,.03),axis=(0,0,1),seed=None,budget=lambda:None):\n        return w.tolerance_scaled_solve(ev,target,axis,np.zeros(5) if seed is None else seed,\n                                        -np.ones(5),np.ones(5),budget)\n    def test_five_dimension_solution(self):\n        q,d=self.call(axis=w.original.rotation_exp([.05,-.04,0])@np.array([0,0,1.]))\n        self.assertTrue(d[\'converged\']);self.assertLessEqual(d[\'position_error_m\'],.0005)\n        self.assertLessEqual(d[\'axis_error_deg\'],2.)\n    def test_already_at_target(self):\n        _,d=self.call(target=(0,0,0));self.assertTrue(d[\'converged\']);self.assertEqual(d[\'iterations\'],0)\n    def test_original_weight_stagnation_recovered(self):\n        def ev(q):\n            p=np.array([q[0],q[1],.000581+.03*q[2]])\n            a=np.array([math.sin(q[2]),0,math.cos(q[2])])\n            jp=np.zeros((3,5));jp[0,0]=jp[1,1]=1;jp[2,2]=.03\n            ja=np.zeros((3,5));ja[:,2]=[math.cos(q[2]),0,-math.sin(q[2])]\n            return p,a,jp,ja\n        q,old=w.base.solve_pose(ev,[0,0,0],[0,0,1],np.zeros(5),-np.ones(5),np.ones(5),lambda:None)\n        self.assertFalse(old[\'converged\'])\n        _,new=self.call(ev=ev,target=(0,0,0),seed=q)\n        self.assertTrue(new[\'converged\']);self.assertLessEqual(new[\'axis_error_deg\'],2.)\n    def test_unreachable_not_pass(self):\n        def ev(q):return np.zeros(3),np.array([0,0,1]),np.zeros((3,5)),np.zeros((3,5))\n        _,d=self.call(ev=ev,target=(.001,0,0));self.assertFalse(d[\'converged\'])\n    def test_unreachable_orientation_not_pass(self):\n        def ev(q):return np.zeros(3),np.array([0,0,1]),np.zeros((3,5)),np.zeros((3,5))\n        _,d=self.call(ev=ev,target=(0,0,0),axis=(.5,0,1));self.assertFalse(d[\'converged\'])\n    def test_outside_limits_target_not_pass(self):\n        _,d=self.call(target=(2,0,0));self.assertFalse(d[\'converged\'])\n    def test_invalid_shape(self):\n        with self.assertRaises(ValueError):self.call(seed=np.zeros(6))\n    def test_nonfinite_target(self):\n        with self.assertRaises(ValueError):self.call(target=(float(\'nan\'),0,0))\n    def test_zero_axis(self):\n        with self.assertRaises(ValueError):self.call(axis=(0,0,0))\n    def test_bad_jacobian(self):\n        def ev(q):return np.zeros(3),np.array([0,0,1]),np.zeros((3,6)),np.zeros((3,5))\n        with self.assertRaises(ValueError):self.call(ev=ev)\n    def test_nonunit_measured_axis(self):\n        def ev(q):return np.zeros(3),np.array([0,0,2]),np.zeros((3,5)),np.zeros((3,5))\n        with self.assertRaises(ValueError):self.call(ev=ev)\n    def test_budget_propagates(self):\n        def budget():raise TimeoutError(\'test budget\')\n        with self.assertRaises(TimeoutError):self.call(budget=budget)\n    def test_inputs_preserved(self):\n        q=np.zeros(5);target=np.array([.01,.02,.03]);axis=np.array([0.,0.,1.])\n        self.call(seed=q,target=target,axis=axis)\n        np.testing.assert_array_equal(q,0);np.testing.assert_array_equal(target,[.01,.02,.03]);np.testing.assert_array_equal(axis,[0,0,1])\n\n\nclass PlanningTests(unittest.TestCase):\n    def test_bounded_lift_heights(self):\n        out=w.lift_candidates([.1,.1,.03],[.1,.1,.07])\n        np.testing.assert_allclose([p[2]-.03 for p in out],[.04,.035,.03])\n        self.assertEqual(len(out),3)\n    def test_lift_request_not_40_rejected(self):\n        with self.assertRaises(ValueError):w.lift_candidates([0,0,0],[0,0,.2])\n    def test_lift_lateral_offset_rejected(self):\n        with self.assertRaises(ValueError):w.lift_candidates([0,0,0],[.01,0,.04])\n    def test_complete_plan_keeps_40_first(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder);p=t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True,True)\n            self.assertAlmostEqual(t.o.report[\'fork_lift_selection\'][\'selected_lift_m\'],.04)\n            self.assertEqual(len(p[\'path\']),41);self.assertEqual(p[\'recovered_waypoints\'],0)\n            self.assertTrue(t.o.report[\'fork_lift_selection\'][\'execution_state_unchanged\'])\n    def test_numerical_failure_selects_35_without_moving(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder);calls=[]\n            def core(target,*args):\n                calls.append(target[2])\n                if target[2]>.0351:raise w.CarryIKFailure(dict(waypoint=38,reason=\'test local stagnation\'))\n                return dict(path=[[0]*5,[0,0,float(target[2]),0,0]],duration_s=1.,recovered_waypoints=0)\n            t._carry_plan=core;t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True,True)\n            self.assertEqual(calls,[.04,.035]);self.assertEqual(t.d.time,47.84)\n            np.testing.assert_array_equal(t.d.qpos,0);np.testing.assert_array_equal(t.d.ctrl,0)\n            self.assertAlmostEqual(t.o.report[\'fork_lift_selection\'][\'selected_lift_m\'],.035)\n    def test_all_candidates_fail_no_silent_success(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder)\n            def core(*a):raise w.CarryIKFailure(dict(waypoint=38,reason=\'test\'))\n            t._carry_plan=core\n            with self.assertRaises(RuntimeError):t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True,True)\n            self.assertEqual(len(t.o.report[\'fork_lift_selection\'][\'attempts\']),3)\n            self.assertIsNone(t.o.report[\'fork_lift_selection\'][\'selected_lift_m\'])\n    def test_collision_never_triggers_height_retry(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder);count=[0]\n            def core(*a):count[0]+=1;raise RuntimeError(\'collision test\')\n            t._carry_plan=core\n            with self.assertRaisesRegex(RuntimeError,\'collision test\'):t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True,True)\n            self.assertEqual(count[0],1)\n    def test_transfer_gets_no_height_fallback(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder);count=[0]\n            def core(*a):count[0]+=1;raise w.CarryIKFailure(dict(waypoint=38,reason=\'test\'))\n            t._carry_plan=core\n            with self.assertRaises(RuntimeError):t.plan(np.array([.025,0,.04]),np.eye(3),\'fork_transfer\',True,True)\n            self.assertEqual(count[0],1)\n    def test_mutation_detected(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder)\n            def core(*a):\n                t.d.ctrl[0]=1\n                return dict(duration_s=1.,recovered_waypoints=0)\n            t._carry_plan=core\n            with self.assertRaisesRegex(RuntimeError,\'changed execution state\'):t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True,True)\n    def test_collision_samples_still_checked(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder);t._issues=lambda *a:[\'table contact test\']\n            with self.assertRaisesRegex(RuntimeError,\'sampled path rejected\'):t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True,True)\n    def test_false_solver_claim_rejected(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder)\n            def bad(ev,goal,axis,q,*a):return q,dict(converged=True)\n            with patch.object(w.base,\'solve_pose\',bad):\n                with self.assertRaises(w.CarryIKFailure):t._carry_plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',True)\n    def test_no_intended_contact_mode_rejected(self):\n        with tempfile.TemporaryDirectory() as folder:\n            t=owner_stage(folder)\n            with self.assertRaises(ValueError):t.plan(np.array([0,0,.04]),np.eye(3),\'fork_lift\',False,True)\n\n\nclass PreservationTests(unittest.TestCase):\n    def test_physics_methods_exactly_inherited(self):\n        for name in (\'execute\',\'tick\',\'move\',\'jaw\',\'hold\',\'grasp\',\'contact_state\',\'carry_metrics\',\'choose_grasp\',\'bottom_clearance\'):\n            with self.subTest(name=name):\n                self.assertIs(getattr(w.ForkStage08D,name),getattr(w.previous.ForkStage08C,name))\n    def test_cup_methods_exactly_inherited(self):\n        for name in (\'load\',\'tick\',\'plan\',\'plan_object\',\'close_on_cup\',\'move_jaw\',\'finish\'):\n            if hasattr(w.previous.CombinedTrial08C,name):\n                self.assertIs(getattr(w.CombinedTrial08D,name),getattr(w.previous.CombinedTrial08C,name))\n    def test_no_guard_constants_changed(self):\n        self.assertEqual(w.base.POSITION_TOL,.0005);self.assertEqual(w.base.AXIS_TOL_DEG,2.)\n        self.assertEqual(w.original.FORK_TOL,.008);self.assertEqual(w.original.CUP_PRESERVE_TOL,.008)\n        self.assertEqual(w.original.OTHER_TOL,.010);self.assertEqual(w.base.GRIPPER_TORQUE_CAP,.25)\n    def test_no_execution_or_model_assignment_in_new_code(self):\n        tree=ast.parse(Path(w.__file__).read_text())\n        for n in ast.walk(tree):\n            if isinstance(n,(ast.Assign,ast.AnnAssign,ast.AugAssign)):\n                targets=n.targets if isinstance(n,ast.Assign) else [n.target]\n                for target in targets:\n                    name=ast.unparse(target)\n                    for bad in (\'d.qpos\',\'d.qvel\',\'d.ctrl\',\'model.\',\'geom_contype\',\'geom_conaffinity\',\'xfrc_applied\',\'qfrc_applied\'):\n                        self.assertNotIn(bad,name)\n    def test_failed_finish_keeps_outcome_false(self):\n        with tempfile.TemporaryDirectory() as folder:\n            p=Path(folder)/\'source\';p.write_text(\'source\')\n            t=w.CombinedTrial08D(dict(scene=str(p),output=folder,scene_sha256=w.base.digest(p),record_data=False))\n            t.report[\'stops\']=[\'synthetic failed run\'];t.finish()\n            r=json.loads((Path(folder)/\'sequence_report.json\').read_text())\n            self.assertFalse(r[\'sequence_test_passed\']);self.assertIsNone(r[\'cup_preserved_after_fork\'])\n\nif __name__==\'__main__\':\n    suite=unittest.defaultTestLoader.loadTestsFromModule(sys.modules[__name__])\n    result=unittest.TextTestRunner(verbosity=1).run(suite)\n    out=dict(kind=\'OFFLINE_SYNTHETIC_SOLVER_AND_PRESERVATION_TESTS_NOT_ROBOT_PHYSICS\',tests_run=result.testsRun,\n             failures=len(result.failures),errors=len(result.errors),skipped=len(result.skipped),passed=result.wasSuccessful())\n    Path(\'sequence_08d_test_summary.json\').write_text(json.dumps(out,indent=2))\n    print(json.dumps(out));raise SystemExit(0 if result.wasSuccessful() else 1)\n'}


SOURCE_HASHES = {'base06b.py': '027f2a91e66be817776dc36d4f2c434e1f2f1e16fdfee6ed01ad58f86134463e', 'transfer_worker.py': '92e0bcebfd45f7501940ad2f16ec97491349c645c04914d86e9019c1d539f849', 'test_base.py': '8404485229c422d88b6fa857ef8a4f59a921e0b030d935b9a386286e419fa81d', 'test_transfer.py': '6f44de8fb4aa76815e4b81283f14402600d650ac5d914018f7c83323223b0a51', 'object_target_worker.py': '3266ddca071ae4e1f0ceb2f86924221f87750eb18beab392ae1abeff1e190405', 'test_object_target.py': 'a0b82bf9f8a3f34480fe4b1be89e7f311583315026cb3b049c733b1a6cdfde25', 'cup_orientation_worker.py': 'b7ae7a8786df46cf86e9fe5b71d030bb344d2c8555c91ed4d3ba2a3ab42edc8e', 'test_cup_orientation.py': '2ec5c14efd3bb5816c1d0cc4f75c3e76c553b4c3ac7c48babc1eedf4ff5bc19b', 'two_object_worker.py': 'bf9f9c67e6c12980b08d87cd8d0995e9354387ed0ef64a3560686e4b594bd318', 'test_sequence.py': '4df2262ecb2dd8bfed7f3c00ba5bc982b376da4f52b95f8cecab2b101dab0401', 'sequence_08b_worker.py': '8d32e2d8cb4c6c29222fc5c493ad9e222bb4dcb7db53e790457e51c62d83cb65', 'test_sequence_08b.py': '00637aca13b964a1b2f36b7858afd4d51dfdcb1f361ee5cec377f0a59f8c17ab', 'sequence_08c_worker.py': '7b2a89e11057b77b3d5958d517c89b27d8eb9f1f93e5b201529b79bb3521d915', 'test_sequence_08c.py': '23bc77f11e1af6eef61c659999d9f0f59adaf84399ff730602787df8f23ddea6', 'sequence_08d_worker.py': '938e5df6a781a9a29806b609c3771c239121c5e95f5e422af23c2473d639992b', 'test_sequence_08d.py': '865aa5c6dbd934a51fccfed78a4a406987be6a5cdbce83474601e6c784012cc2'}

from pathlib import Path

from datetime import datetime, timezone

import hashlib

import json

import os

import subprocess

import sys

import time

import uuid

from IPython.display import display, Image, Markdown

def read_json(path):
    path=Path(path)
    if not path.is_file():raise FileNotFoundError(path)
    if path.stat().st_size>64*1024*1024:raise ValueError('Unexpectedly large JSON file')
    return json.loads(path.read_text(encoding='utf-8-sig'))

def write_json(path,value):
    path=Path(path);temp=path.with_suffix(path.suffix+'.tmp')
    temp.write_text(json.dumps(value,indent=2,allow_nan=False),encoding='utf-8');temp.replace(path)

def within(root,value):
    root=Path(root).resolve();path=Path(value)
    if not path.is_absolute():path=root/path
    path=path.resolve()
    if not path.is_relative_to(root):raise ValueError('Artifact reference outside selected folder: '+str(path))
    return path

def view_08d(run,show_images=True):
    run=Path(run).resolve();path=run/'sequence_report.json'
    print('\nSEQUENCE SUMMARY — 08D',flush=True)
    print('Run:',run)
    if not path.is_file():
        print('No final report in this run. No new trial launched by the viewer.')
        for name in ('regression.log','cup_orientation_tests.log','sequence_tests.log','sequence.log'):
            p=run/name
            if p.is_file():
                from collections import deque
                with p.open(encoding='utf-8',errors='replace') as f:tail=''.join(deque(f,maxlen=10))
                print('\n'+name+' (last lines):\n'+tail)
        return {'report_available':False,'run':str(run)}
    report=read_json(path)
    print('Stop reasons:',report.get('stops'),flush=True)
    selected_lift=report.get('fork_lift_selection')
    if selected_lift:
        chosen=selected_lift.get('selected_lift_m')
        print('LIFT PLANNING — unchanged 0.5 mm / 2-degree acceptance gates')
        print('Requested lift (mm):',1000*selected_lift['requested_lift_m'])
        print('Selected commanded lift (mm):',None if chosen is None else round(1000*chosen,3))
        print('Minimum measured rise still required (mm):',1000*selected_lift['measured_minimum_rise_requirement_m'])
        print('Planning left execution state unchanged:',selected_lift.get('execution_state_unchanged'))
        for attempt in selected_lift.get('attempts',[]):
            failure=attempt.get('failure',{});diags=failure.get('diagnostics',[])
            residual=diags[-1].get('position_error_m') if diags else None
            print('Candidate lift:',round(1000*attempt['commanded_lift_m'],3),'mm | accepted:',attempt['accepted'],
                  '| recovered waypoints:',attempt.get('recovered_waypoints'),
                  '| failed waypoint:',failure.get('waypoint'),
                  '| residual mm:',None if residual is None else round(1000*residual,4))

    for key in ('sequence_test_passed','cup_stage_passed','fork_grasp_verified','fork_lift_verified',
                'fork_transfer_verified','fork_placement_verified','cup_preserved_after_fork',
                'physics_steps','simulation_time_s','recorded_observation_action_pairs','recording_complete','graphics_error','cup_preservation_evaluated','fork_placement_evaluated'):
        print(f'{key}: {report.get(key)}')
    for key in ('fork_lift_measurements','fork_transfer_measurements','fork_placement_measurements',
                'cup_final_placement','preservation_during_fork','fork_stop_diagnostics'):
        value=report.get(key)
        if value is not None:print('\n'+key+':\n'+json.dumps(value,indent=2))
    launch=run/'launcher_status.json'
    if launch.is_file():print('Full worker process time (s):',read_json(launch).get('wall_time_s'))
    if report.get('observation_manifest'):
        try:
            path=within(run,report['observation_manifest'])
            if path.stat().st_size>64*1024*1024:raise ValueError('Manifest too large')
            with path.open(encoding='utf-8') as f:records=[json.loads(line) for line in f if line.strip()]
            refs=[within(run,p) for r in records for p in r['images'].values()]
            missing=sum(not p.is_file() for p in refs)
            stages=sorted({r.get('stage') for r in records})
            print('\nRECORDING:',len(records),'records |',len(refs),'image references |',missing,'missing')
            print('Stages:',stages,'| Count matches:',len(records)==report.get('recorded_observation_action_pairs'))
            print('Every record contains all three cameras:',all(set(r['images'])=={'top','left_wrist_cam','right_wrist_cam'} for r in records))
        except (OSError,ValueError,KeyError,TypeError) as exc:print('Recording inspection issue:',exc)
    print('Active-hand clearance precheck:', report.get('fork_clearance_preflight_passed'))
    selection=report.get('fork_selected_clearance_candidate')
    if selection:
        print('Selected jaw target (rad):',selection.get('opening_rad'))
        print('Selected gripper reference above handle center (mm):',1000*selection.get('height_offset_m',0))
        preview=selection.get('closure_preview',{})
        print('Closing-to-first-contact minimum predicted hand/table gap (mm):',
              1000*preview.get('minimum_clearance',{}).get('minimum_distance_m',float('nan')))
        print('These are planning measurements, NOT proof of a successful grasp.')
    search=report.get('fork_candidate_search',{})
    print('Candidate checks:',len(search.get('candidates',[])), '| execution state unchanged:',search.get('execution_state_unchanged'))
    if not selection:
        for c in search.get('candidates',[]):
            print('Candidate',c.get('candidate_index'),'rejected:',str(c.get('rejection','not accepted'))[:180])
        print('Detailed candidate evidence: fork_clearance_selection.json')
    print('Episode outcome:', 'PASSED engineering sequence' if report.get('sequence_test_passed') else 'FAILED / PARTIAL engineering sequence')
    for label in ('fork_pregrasp', 'fork_retreat'):
        chosen=report.get(label+'_selection')
        if chosen:
            print(label, '| chosen target:',chosen.get('selected_target_m'),
                  '| reference clearance (mm):',1000*chosen.get('reference_clearance_m',0))
    if report.get('fork_plan_failure'):
        failed=report['fork_plan_failure']
        print('Planning stopped in:',failed.get('phase'),'| details saved in plan failure JSON')
    if report.get('cup_preservation_evaluated') is False:
        print('Cup final preservation: NOT YET EVALUATED (not an automatic cup failure).')
    print('This is not a trained-policy, camera-reasoning or official table-setting score.')
    if show_images:
        for label in ('cup_placed','fork_pregrasp','fork_grasped','fork_transferred','sequence_final','stopped'):
            value=report.get('snapshots',{}).get(label)
            if value:
                try:
                    p=within(run,value)
                    if p.is_file():
                        display(Markdown('### '+label.replace('_',' ').title()+' — actual recorded state'))
                        display(Image(filename=str(p),width=640))
                except (OSError,ValueError) as exc:print('Image unavailable:',exc)
    return report

def validate_passed_07c(root,run_name):
    if not isinstance(run_name,str) or Path(run_name).name!=run_name or '/' in run_name or '\\' in run_name:
        raise ValueError('Source run must be one folder name')
    source=root/'artifacts'/'cup_transfer_07c'/run_name
    r=read_json(source/'transfer_report.json');cfg=read_json(source/'transfer_config.json')
    keys=('completed','transfer_test_passed','grasp_verified','lift_verified','transfer_verified','placement_verified','release_verified')
    if not all(r.get(k) is True for k in keys) or r.get('stops') or r.get('source_scene_modified'):
        raise RuntimeError('Selected 07C checkpoint is not a complete, unchanged pass')
    scene=Path(cfg['scene'])
    if not scene.is_file():raise FileNotFoundError('Source scene missing: '+str(scene))
    if hashlib.sha256(scene.read_bytes()).hexdigest()!=cfg.get('scene_sha256'):
        raise RuntimeError('Scene changed after the passed checkpoint')
    # A fresh episode begins from the source initial state, then physically repeats
    # the cup action once before the new left action. It never loads a final screenshot.
    return source,cfg

def launch_worker(cmd,root,run):
    started=time.monotonic();proc=None;status=None;state={'interrupted':False,'timed_out':False,'command':cmd}
    log=run/'sequence.log';env=os.environ.copy();env.update(PYTHONUTF8='1',PYTHONIOENCODING='utf-8',PYTHONUNBUFFERED='1')
    try:
        status=display(Markdown('**08D worker starting.** Full progress: `sequence.log`'),display_id=True)
        with log.open('w',encoding='utf-8') as writer:
            proc=subprocess.Popen(cmd,cwd=run,env=env,stdout=writer,stderr=subprocess.STDOUT)
            state['pid']=proc.pid
            with log.open('r',encoding='utf-8',errors='replace') as reader:
                latest='initializing'
                while True:
                    for line in reader.readlines():
                        if line.startswith('PROGRESS '):
                            try:
                                e=json.loads(line[len('PROGRESS '):]);latest=e.get('stage','')+' | '+e.get('phase',e.get('reason',''))
                            except ValueError:latest=line[:180]
                        elif 'Traceback' in line or 'Error:' in line:latest=line[:180]
                    elapsed=time.monotonic()-started
                    if status is not None:status.update(Markdown(f'**08D worker: {elapsed:.1f}s.** {latest[:220]} — full log: `sequence.log`'))
                    if proc.poll() is not None:break
                    if elapsed>300:
                        state['timed_out']=True;raise TimeoutError('Worker exceeded 300 seconds; preserve sequence.log')
                    time.sleep(.25)
    except KeyboardInterrupt:
        state['interrupted']=True;raise
    finally:
        if proc is not None and proc.poll() is None:
            proc.terminate()
            try:proc.wait(timeout=5)
            except subprocess.TimeoutExpired:proc.kill();proc.wait(timeout=5)
        state['returncode']=proc.returncode if proc is not None else None;state['wall_time_s']=time.monotonic()-started
        write_json(run/'launcher_status.json',state)
    return state['returncode']

def run_08d(root,source_run,mode='run_once_or_view',record_data=True,start_new_trial=False):
    if mode not in ('run_once_or_view','view_saved'):raise ValueError('Unknown mode')
    if not isinstance(record_data,bool) or not isinstance(start_new_trial,bool):raise ValueError('Boolean options required')
    root=Path(root).expanduser().resolve()
    if not root.is_dir():raise FileNotFoundError(root)
    parent=root/'artifacts'/'two_object_08d';pointer=parent/'selected_run.json';lock=parent/'active_run.lock'
    if lock.exists():raise RuntimeError('An 08D run is active or was interrupted without cleanup. Inspect it; do not start a duplicate.')
    if pointer.is_file() and not start_new_trial:
        selected=read_json(pointer)
        if selected.get('source_run')!=source_run:raise RuntimeError('Selected run uses a different checkpoint')
        print('Displaying the existing 08D attempt; no simulation is running.')
        return view_08d(within(parent,selected['run']))
    if mode=='view_saved':raise FileNotFoundError('No saved 08D attempt yet')
    if Path(sys.prefix).name.lower()!='tableguard':raise RuntimeError('Select Python (TableGuard) to execute a new sequence')
    source,cfg=validate_passed_07c(root,source_run)
    import mujoco
    parent.mkdir(parents=True,exist_ok=True)
    try:fd=os.open(lock,os.O_CREAT|os.O_EXCL|os.O_WRONLY)
    except FileExistsError:raise RuntimeError('A second 08D launcher was blocked')
    os.close(fd)
    run=parent/(datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')+'_'+uuid.uuid4().hex[:6])
    try:
        run.mkdir(exist_ok=False)
        for name,text in SOURCE_TEXTS.items():
            if hashlib.sha256(text.encode()).hexdigest()!=SOURCE_HASHES[name]:raise RuntimeError('Embedded source changed: '+name)
            compile(text,name,'exec');(run/name).write_text(text,encoding='utf-8',newline='\n')
        cfg=dict(cfg,output=str(run),record_data=record_data,offset_m=[.025,0,0],
                 source_07c_run=str(source),purpose='CONTINUOUS_TWO_OBJECT_ENGINEERING_NOT_LEARNED_POLICY',task_revision='08d-tolerance-scaled-carry-ik-bounded-lift-v1')
        path=run/'sequence_config.json';write_json(path,cfg)
        write_json(pointer,dict(run=run.name,source_run=source_run))
        print('NEW 08D RUN:',run,flush=True)
        print('Right cup +25 mm X, then left fork -25 mm X, in the same simulation.',flush=True)
        print('Recording:',record_data,'| No modifications to existing checkpoints.',flush=True)
        env=os.environ.copy();env.update(PYTHONUTF8='1',PYTHONIOENCODING='utf-8')
        tests=[([sys.executable,str(run/'test_base.py'),str(run/'base06b.py')],'regression.log','regression_summary.json'),
               ([sys.executable,str(run/'test_cup_orientation.py')],'cup_orientation_tests.log','cup_orientation_test_summary.json'),
               ([sys.executable,str(run/'test_sequence.py')],'sequence_tests.log','sequence_test_summary.json'),
               ([sys.executable,str(run/'test_sequence_08b.py')],'sequence_08b_tests.log','sequence_08b_test_summary.json'),
               ([sys.executable,str(run/'test_sequence_08c.py')],'sequence_08c_tests.log','sequence_08c_test_summary.json'),
               ([sys.executable,str(run/'test_sequence_08d.py')],'sequence_08d_tests.log','sequence_08d_test_summary.json')]
        print('Checking unchanged cup base and new sequence logic (offline, not physical trials).',flush=True)
        for command,log,summary in tests:
            result=subprocess.run(command,cwd=run,env=env,capture_output=True,text=True,encoding='utf-8',errors='replace',timeout=60)
            (run/log).write_text(result.stdout+'\n'+result.stderr,encoding='utf-8')
            if result.returncode!=0:
                print(result.stdout,result.stderr);raise RuntimeError('Software checks failed; no physics worker launched: '+log)
            r=read_json(run/summary)
            if not r.get('passed'):raise RuntimeError('Test report did not pass')
            print('Offline checks passed:',r['tests_run'],flush=True)
        code=launch_worker([sys.executable,'-u',str(run/'sequence_08d_worker.py'),str(path)],root,run)
        result=view_08d(run)
        if code not in (0,2):raise RuntimeError(f'Unexpected exit {code}; read sequence.log')
        return result
    finally:lock.unlink(missing_ok=True)

print('08D tools loaded. Next cell runs one sequence, or displays the saved attempt.')


## Run once, then view saved results
Leave these settings unchanged for the first attempt. Do not run another worker concurrently. The earlier source checkpoint is used to begin a fresh continuous episode, not to replay a final screenshot.

In [ ]:
ROOT = Path(
    r"C:\Users\Fiza\Downloads"
    r"\TableGuard_Phase1_Python_Starter\TableGuard_Starter"
)
SOURCE_RUN = "20260913T025459Z_3185ba"
MODE = "run_once_or_view"
RECORD_DATA = True
START_NEW_TRIAL = False

RESULT = run_08d(ROOT, SOURCE_RUN, mode=MODE, record_data=RECORD_DATA,
                 start_new_trial=START_NEW_TRIAL)


## Validation boundary
Prepared under Python 3.13.5. The source is syntax-checked for Python 3.11 grammar. Offline tests use synthetic numerical and file fixtures; they do not execute the user's robot model. No MuJoCo physical fork lift/transfer or rendering result is claimed here. The preserved tests run in the user's Python (TableGuard) kernel before a fresh physics attempt.

The authoritative pass/fail evidence is `sequence_report.json` plus the recorded trace and images from the actual run. A partial recording is not a successful full-task demonstration. Retain `fork_lift_selection.json` and any `plan_fork_*_candidate_*_failure.json` records, including when a shorter commanded lift was selected.
